Note:
- This notebook is part of the Gao et al. (2026, in preparation) analysis workspace:
  Gao et al., Minute-scale coupling of chromatin marks and transcriptional bursts (2026, in preparation)
- For data layout, execution order, and key parameters, see the README.md in this folder:
  - analyses/01_Snapshot_analysis/README.md
- State labels:
  - Active = ON
  - Inactive = OFF
  Some intermediate tables may use ON/OFF; we treat them as equivalent to Active/Inactive.


# Snapshot image analysis (public release)

This notebook processes snapshot microscopy data, performs segmentation and quantification, and generates figures used in the manuscript. Each section below states the objective, inputs, and outputs in a GitHub‑ready format.

## Step 0: Prepare ND2 scenes for segmentation

This step converts ND2 scenes into (i) blue‑channel PNG projections for Cellpose and (ii) median‑filtered TIFF stacks for downstream quantification.

- **Inputs:** ND2 files per well/scene.
- **Processing:** Gaussian blur, Z‑projection, intensity stretching, median filtering.
- **Outputs:** PNGs in `outputs/processed_for_cellpose/` and TIFF stacks in `outputs/processed_1/`.

In [ ]:
# ----------------------- ① Set environment variables "beforehand" -----------------------
import os, multiprocessing as mp
os.environ["OMP_NUM_THREADS"]       = "1"   # Prevent runaway multithreading in OpenBLAS, etc.
os.environ["OPENBLAS_NUM_THREADS"]  = "1"

# ----------------------- ② Standard imports -------------------------------
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
from packaging import version
import tifffile as tiff
from aicsimageio import AICSImage
from scipy.ndimage import gaussian_filter, median_filter
from skimage import exposure
import os

# ----------------------- ③ User settings ----------------------------------
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
root = Path(parent_dir)

BASE        = Path(root)        # ★Experiment folder
RAW         = Path(os.path.dirname(os.path.dirname((parent_dir))))   / "data" / "01_Snapshot_analysis" / "nd2"
OUT1        = BASE / "outputs" / "processed_for_cellpose"
OUT2        = BASE / "outputs" / "processed_1"
SIGMA_BLUR  = 5
MEDIAN_RAD  = 1
SATURATION  = 0.0035
WELLS       = range(1, 2)          # For only well004, use range(4,5)
MAX_WORKERS = max(os.cpu_count() - 1, 1)

OUT1.mkdir(exist_ok=True, parents=True)
OUT2.mkdir(exist_ok=True, parents=True)

# ----------------------- ④ Safe wrapper for overwriting tifffile --------------------
_OVER_OK = version.parse(tiff.__version__) >= version.parse("2023.8.30")
def imwrite(path: Path, data, **kwargs):
    if path.exists(): path.unlink()
    tiff.imwrite(path, data, **kwargs)

# ----------------------- ⑤ Image processing utilities -----------------------
def z_max(arr):            return arr.max(axis=0)

def stretch(img, sat):
    lo, hi = np.percentile(img, [sat*100, 100-sat*100])
    return exposure.rescale_intensity(img, in_range=(lo, hi))

def save_png_blue(img2d, path):
    rgb = np.zeros((*img2d.shape, 3), np.uint16)
    rgb[..., 2] = np.clip(img2d, 0, 65535).astype(np.uint16)
    imwrite(path, rgb, photometric="rgb")

# ----------------------- ⑥ Worker function (per scene) --------------------
def process_scene(well, nd2_path, scene_no, scene_name):
    
    img = AICSImage(nd2_path)
    img.set_scene(scene_name)
    stack = np.squeeze(img.get_image_data("TZCYX"), axis=0)     # (Z,C,Y,X)

    # ----- PNG for Cellpose (blue channel only) -----
    ch2, ch3     = stack[:, 1], stack[:, 2]
    dup_blur     = gaussian_filter(np.stack([ch2, ch3], 1),
                                   sigma=(0,0,SIGMA_BLUR,SIGMA_BLUR))
    proj_cs      = stretch(z_max(dup_blur[:,0] + dup_blur[:,1]), SATURATION)
    png_out      = OUT1 / f"well00{well}_series-{scene_no:03d}.png"
    save_png_blue(proj_cs, png_out)

    # ----- Z-stack TIFF with median filter -----
    med = median_filter(stack,
                        size=(1,1,MEDIAN_RAD*2+1, MEDIAN_RAD*2+1))[np.newaxis,...]
    tif_out = OUT2 / f"well00{well}_series-{scene_no:03d}.tif"
    imwrite(tif_out,
            np.clip(med, 0, 65535).astype(np.uint16),
            imagej=True, metadata={'axes':'TZCYX'})

    return f"[well{well}] scene {scene_no:03d} done"

# ----------------------- ⑦ Create job list -----------------------------
jobs = []
for well in WELLS:
    nd2 = RAW / f"well00{well}.nd2"
    for n, sc in enumerate(AICSImage(nd2).scenes, 1):
        jobs.append( (well, nd2, n, sc) )

# ----------------------- ⑧ Parallel execution (Notebook OK) ----------------------
def run_parallel():
    # --------- Only change here ---------
    ctx = mp.get_context("fork")          # ← spawn → fork
    # ---------------------------------
    with ProcessPoolExecutor(max_workers=MAX_WORKERS,
                             mp_context=ctx) as pool:
        futs = [pool.submit(process_scene, *j) for j in jobs]
        for f in as_completed(futs):
            print(f.result())

    print("=== All finished ===")

# -------------------------------------------------------------------------
run_parallel()      # Start here when running the cell


## Step 1: Cellpose segmentation (nuclei masks)

Run Cellpose on the PNG projections to generate segmentation masks and overlay images.

- **Inputs:** PNG projections from Step 0.
- **Outputs:** mask TIFFs in `outputs/segmented/seg_tif/` and overlay PNGs in `outputs/segmented/png/`.
- **Execution mode:** serial (not parallel). GPU is used automatically if available; set `FORCE_CPU = True` to force CPU.

In [ ]:
# ----------------------------- MUST be first ------------------------------
import os

# Prevent thread oversubscription and kernel crashes (set before torch/numpy/scipy imports)
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
# -------------------------------------------------------------------------

import time
from pathlib import Path
from functools import lru_cache

# --- Fix drawing backend to non-GUI (safe for Notebook; files are saved) ---
import matplotlib
#matplotlib.use("Agg")
import matplotlib.pyplot as plt

import numpy as np
import torch
from cellpose import models, io, plot
import tifffile as tiff
from packaging import version

# ------------------------- User settings ----------------------------------
BASE_DIR  = Path.cwd().parent                    # Example .../250429-live1/
IN_DIR    = BASE_DIR / "outputs" / "processed_for_cellpose"
PNG_DIR   = BASE_DIR / "outputs" / "segmented/png"
TIF_DIR   = BASE_DIR / "outputs" / "segmented/seg_tif"
PNG_DIR.mkdir(parents=True, exist_ok=True)
TIF_DIR.mkdir(parents=True, exist_ok=True)

WELLS        = [f"well{n:03d}" for n in range(1, 2)]   # 001
SERIES_RANGE = range(1, 11)                           # 001–100
DIAMETER     = 100
CHANNELS     = [3, 0]                                  # cyto=R, nucleus=B

# Force CPU even if CUDA is available
FORCE_CPU = False

# GPU device index to use when CUDA is available (usually 0)
GPU_ID = 0
# --------------------------------------------------------------------------


# ------------------------- Small utilities --------------------------------
def has_cuda_gpu() -> bool:
    """CUDA GPU availability (Cellpose gpu=True typically implies CUDA)."""
    return torch.cuda.is_available() and (torch.cuda.device_count() > 0)


# ---------------- tifffile overwrite-safe wrapper -------------------------
def safe_imwrite(path: Path, data, **kw):
    """Safely overwrite and save using tifffile, absorbing API differences"""
    if path.exists():
        path.unlink()
    tiff.imwrite(path, data, **kw)


# --------------- "Once per process" Cellpose initialization ----------------
@lru_cache(None)
def get_model(gpu_id: int):
    """
    Initialize only the first time called in a given process.
    gpu_id >= 0 selects CUDA; -1 selects CPU.
    """
    if gpu_id >= 0 and has_cuda_gpu():
        torch.cuda.set_device(gpu_id)
        return models.Cellpose(
            gpu=True,
            model_type="nuclei",
            device=torch.device(f"cuda:{gpu_id}")
        )
    return models.Cellpose(gpu=False, model_type="nuclei")


def normalize99_local(img: np.ndarray, p_low=1.0, p_high=99.0, use_nonzero=True) -> np.ndarray:
    """
    Robust percentile normalization to [0,1].
    - If use_nonzero=True, compute percentiles on non-zero pixels to avoid background-dominated images.
    - Works for uint16/uint8/float.
    """
    x = img.astype(np.float32, copy=False)

    if use_nonzero:
        v = x[x > 0]
        if v.size >= 16:
            lo, hi = np.percentile(v, [p_low, p_high])
        else:
            lo, hi = np.percentile(x, [p_low, p_high])
    else:
        lo, hi = np.percentile(x, [p_low, p_high])

    # Avoid division by zero
    denom = (hi - lo)
    if denom <= 1e-6:
        # Fallback: use min/max
        lo2, hi2 = float(np.min(x)), float(np.max(x))
        denom2 = (hi2 - lo2)
        if denom2 <= 1e-6:
            return np.zeros_like(x, dtype=np.float32)
        y = (x - lo2) / (denom2 + 1e-6)
    else:
        y = (x - lo) / (denom + 1e-6)

    return np.clip(y, 0.0, 1.0)


# ------------------- Worker to process one PNG ----------------------------
def process_image(series: int, well: str, gpu_id: int):
    s_tag   = f"series-{series:03d}"
    img_in  = IN_DIR / f"{well}_{s_tag}.png"

    if not img_in.exists():
        raise FileNotFoundError(f"Input not found: {img_in}")

    img = io.imread(str(img_in))

    if img.ndim == 3:
        img = img[..., 2]   # (H,W) にする

    cp_model = get_model(gpu_id)
    masks, flows, *_ = cp_model.eval(img, diameter=DIAMETER, channels=[0, 0])

    # 表示はCellpose推奨の正規化
    img_disp = normalize99_local(img, p_low=1.0, p_high=99.5, use_nonzero=True)

    fig = plt.figure(figsize=(12, 5))
    plot.show_segmentation(fig, img_disp, masks, flows[0], channels=[0, 0])
    fig.tight_layout()
    fig.savefig(PNG_DIR / f"{well}_{s_tag}.png", dpi=300)
    plt.close(fig)

    # ---------- Mask TIFF (uint16) ----
    safe_imwrite(TIF_DIR / f"{well}_{s_tag}.tiff", masks.astype("uint16"))

    dev = f"cuda:{gpu_id}" if gpu_id >= 0 else "CPU"
    return f"[{well}] {s_tag} — done on {dev}"


# --------------------------- Serial run -----------------------------------
def run_serial(gpu_id: int):
    t0 = time.time()
    for well in WELLS:
        for s in SERIES_RANGE:
            try:
                print(process_image(s, well, gpu_id))
            except Exception as e:
                print("⚠️ error:", well, f"series-{s:03d}", e)
                raise
    print(f"\n=== All finished (serial) in {time.time()-t0:.1f} s ===")


# ------------------------- Entry point ------------------------------------
def run():
    """
    Serial execution with optional GPU.
    - If FORCE_CPU=True: run serially on CPU.
    - If CUDA GPU is available: run serially on the selected GPU.
    - Otherwise: run serially on CPU.
    """
    if FORCE_CPU:
        print("FORCE_CPU=True -> run serial on CPU")
        return run_serial(-1)

    if has_cuda_gpu():
        gpu_count = torch.cuda.device_count()
        use_id = GPU_ID if GPU_ID < gpu_count else 0
        if GPU_ID >= gpu_count:
            print(f"GPU_ID={GPU_ID} out of range; using cuda:0")
        print(f"CUDA GPU detected -> run serial on cuda:{use_id}")
        return run_serial(use_id)

    print("CUDA GPU not detected -> run serial on CPU")
    return run_serial(-1)


# Execute this cell to run
if __name__ == "__main__":
    run()

## Step 2: Per‑cell quantification from masks

Extract per‑cell mean intensities and cell area from a fixed Z‑slice using the Cellpose masks.

- **Inputs:** median‑filtered stacks (`outputs/processed_1/`) and masks (`outputs/segmented/seg_tif/`).
- **Outputs:** a per‑cell table with `image_id`, `cell_id`, channel means, and `cell_area_px`.

In [ ]:

# %% ---------------- Notebook-friendly Cellpose batch --------------------------------
# Dependencies:  pip install "cellpose<=2.2" tifffile torch matplotlib packaging

import os, multiprocessing as mp, time, traceback, logging, sys
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from itertools import chain

import numpy as np
import pandas as pd
import tifffile as tiff

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(processName)s %(levelname)s: %(message)s")

# ---------------------- User settings for paths, etc. ---------------------------------
BASE_DIR   = Path.cwd().parent
PROC_DIR   = BASE_DIR / "outputs" / "processed_1"        # 11-Z x 3-ch median-filtered stack
MASK_DIR   = BASE_DIR / "outputs" / "segmented/seg_tif"  # Cellpose mask (uint16)
WELLS      = ["well001"]#["well001","well002","well003","well004","well005","well006","well007","well008"]
SERIES     = range(1, 11)                   # series-001 … 100
Z_SLICE    = 10                               # 0-based index → 11th slice
N_WORKERS  = max(1, os.cpu_count() - 1)

# -------------------- Analyze one image (=1 scene) -----------------------------
def analyze_one_image(args):
    well, series = args
    tag         = f"series-{series:03d}"
    image_id    = f"{well}_{tag}"
    proc_path   = PROC_DIR / f"{image_id}.tif"
    mask_path   = MASK_DIR / f"{image_id}.tiff"

    try:
        proc = tiff.imread(proc_path, mode="r")   # mem-map read
        mask = tiff.imread(mask_path)
    except Exception as e:
        logging.error("I/O error for %s → %s", image_id, e)
        return []                                 # ← Skip this scene

    cell_ids = np.unique(mask)[1:]                # 0 is background
    if cell_ids.size == 0:
        return []

    slice_img = proc[Z_SLICE]                     # shape: (C, Y, X)

    rows = []
    for cid in cell_ids:
        m = mask == cid                           # bool 2-D mask
        if not m.any():
            continue
        # ----------- Added: Area calculation (pixel²) ----------------
        area_px = int(m.sum())                    # Total number of True
        # -------------------------------------------------------------
        means = [slice_img[c][m].mean() for c in range(3)]
        rows.append(dict(
            image_id               = image_id,
            cell_id                = int(cid),
            SNAPtag_mean_intensity = means[0],
            MCP_mean_intensity     = means[1],
            mTetR_mean_intensity   = means[2],
            cell_area_px           = area_px      # ★ New column
        ))
    return rows

# ------------------- main: Scan all scenes in parallel --------------------------
def main():
    jobs = [(w, s) for w in WELLS for s in SERIES]

    ctx = mp.get_context("fork")                  # Use 'fork' in Notebook
    with ProcessPoolExecutor(max_workers=N_WORKERS,
                             mp_context=ctx) as pool:
        futs = {pool.submit(analyze_one_image, j): j for j in jobs}

        all_rows = []
        for fut in as_completed(futs):
            try:
                all_rows.extend(fut.result())
            except Exception as e:
                job = futs.get(fut, "<unknown job>")
                logging.error("Crash in job %s: %s\n%s",
                            job, e, traceback.format_exc())

    return pd.DataFrame(all_rows)                 # ← DataFrame with cell_area_px column

# --------------------------- Run ---------------------------------------
if __name__ == "__main__":
    df = main()
    


"""
Extracts 'well' and 'series' numbers from the 'image_id' column of a DataFrame.

- Adds a new column 'well_no' by extracting the integer following 'well' in the 'image_id' string.
- Adds a new column 'series_no' by extracting the integer following 'series-' in the 'image_id' string.
- Both extracted values are converted to integers, ignoring any leading zeros.
- Displays the first few rows of the updated DataFrame for verification.
"""

# ❶ Extract well number and series number from string ─────────────────────────
df["well_no"]   = (
    df["image_id"]
      .str.extract(r"well(\d+)")        # Extract only the digits after 'well'
      .astype(int)                      # Convert to integer, ignoring leading zeros
)

df["series_no"] = (
    df["image_id"]
      .str.extract(r"series-(\d+)")     # Extract only the digits after 'series-'
      .astype(int)
)

# ❷ Check -----------------------------------------------------------------
df.head()


## QC: Pairwise intensity relationships

Inspect pairwise scatter/KDE plots among SNAPtag, MCP, and mTetR mean intensities to check overall signal distributions.

- **Inputs:** per‑cell intensity table from Step 2.
- **Outputs:** interactive diagnostic plots (no files written).

In [ ]:
import seaborn as sns
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt

df_results = df

# Select necessary columns from DataFrame
df_filtered = df_results[['SNAPtag_mean_intensity', 'MCP_mean_intensity', 'mTetR_mean_intensity']]

# Set figure size
plt.figure(figsize=(7, 7))

# 1. Scatter plot + KDE for SNAPtag and MCP
sns.jointplot(
    x='SNAPtag_mean_intensity', 
    y='MCP_mean_intensity', 
    data=df_filtered, 
    kind='scatter', 
    marginal_kws=dict(bins=30, fill=True), 
    space=0.2
).plot_joint(sns.kdeplot, color="r", zorder=0, levels=6)
plt.suptitle('SNAPtag vs MCP with KDE', y=1.02)  # Title

# Show plot
plt.show()

# 2. Scatter plot + KDE for MCP and mTetR
sns.jointplot(
    x='MCP_mean_intensity', 
    y='mTetR_mean_intensity', 
    data=df_filtered, 
    kind='scatter', 
    marginal_kws=dict(bins=30, fill=True), 
    space=0.2
).plot_joint(sns.kdeplot, color="g", zorder=0, levels=6)
plt.suptitle('MCP vs mTetR with KDE', y=1.02)  # Title

# Show plot
plt.show()

# 3. Scatter plot + KDE for SNAPtag and mTetR
sns.jointplot(
    x='SNAPtag_mean_intensity', 
    y='mTetR_mean_intensity', 
    data=df_filtered, 
    kind='scatter', 
    marginal_kws=dict(bins=30, fill=True), 
    space=0.2
).plot_joint(sns.kdeplot, color="b", zorder=0, levels=6)
plt.suptitle('SNAPtag vs mTetR with KDE', y=1.02)  # Title

# Show plot
plt.show()

## QC: Intensity‑based filtering for candidate cells

Define intensity thresholds for MCP and mTetR (minimum and +3 SD) and visualize filtered versus excluded cells.

- **Inputs:** per‑cell intensity table.
- **Outputs:** filtered DataFrame (`df_within_3std`) and diagnostic scatter/KDE plot.

In [ ]:

# Calculate the mean and standard deviation for each channel

mean_snaptaf = df_results['SNAPtag_mean_intensity'].mean()
std_snaptaf = df_results['SNAPtag_mean_intensity'].std()

mean_mcp = df_results['MCP_mean_intensity'].mean()
std_mcp = df_results['MCP_mean_intensity'].std()

mean_mtetR = df_results['mTetR_mean_intensity'].mean()
std_mtetR = df_results['mTetR_mean_intensity'].std()

# Create a flag indicating whether values are within ±3 standard deviations
df_results['within_3std'] = (
    # (np.abs(df_results['SNAPtag_mean_intensity'] - mean_snaptaf) <= 2 * std_snaptaf) &
    (df_results['MCP_mean_intensity'] >= 500) &
    (df_results['MCP_mean_intensity'] <= mean_mcp + 3 * std_mcp) &
    (df_results['mTetR_mean_intensity'] >= 520) &
    (df_results['mTetR_mean_intensity'] <= mean_mtetR + 3 * std_mtetR)
)

# Split into two datasets to color data inside and outside the range
df_within_3std = df_results[df_results['within_3std']]
df_outside_3std = df_results[~df_results['within_3std']]

# Set figure size
plt.figure(figsize=(10, 10))

# 2. Scatter plot + KDE for MCP and mTetR, with color coding
sns.jointplot(
    x='MCP_mean_intensity', 
    y='mTetR_mean_intensity', 
    data=df_within_3std, 
    kind='scatter', 
    color='blue',  # Show within ±3 std in blue
    marginal_kws=dict(bins=30, fill=True), 
    space=0.2
).plot_joint(sns.kdeplot, color="blue", zorder=0, levels=6)

# Show cells outside the range in red
sns.scatterplot(
    x='MCP_mean_intensity', 
    y='mTetR_mean_intensity', 
    data=df_outside_3std, 
    color='red',  # Show outside ±3 std in red
    label='Outside 3 std'
)

plt.suptitle('MCP vs mTetR with KDE (±3 std colored)', y=1.02)  # Title

# Show plot
plt.show()

## Export filtered cell list

Save the filtered cell table for downstream spot detection.

- **Output:** `outputs/segmented/csv/df_within_3std.csv`.

In [ ]:
import os
analysis_root = os.path.dirname(os.getcwd())
# Create the folder if it does not exist
file_path = analysis_root + '/outputs/segmented/csv'
if not os.path.exists(file_path):
    os.makedirs(file_path)

# Save the DataFrame as a CSV to the specified path

file_path = analysis_root + '/outputs/segmented/csv/df_within_3std.csv'

df_within_3std.to_csv(file_path, index=False)

## Spot detection with Big‑FISH (setup)

Initialize Big‑FISH dependencies and load the filtered per‑cell table for spot detection.

- **Input:** `df_within_3std.csv` from the previous step.
- **Output:** in‑memory DataFrame used for parallel spot calling.

In [ ]:
import os
import numpy as np
import bigfish
import bigfish.stack as stack
import bigfish.detection as detection
import bigfish.multistack as multistack
import bigfish.plot as plot
import pandas as pd

import numpy as np
import pandas as pd
import tifffile as tiff
from skimage.measure import regionprops
from skimage.io import imread
from skimage.measure import regionprops


"""
This script performs the following tasks:
1. Retrieves the current working directory.
2. Determines the parent directory two levels above the current directory.
3. Constructs the file path to a CSV file named 'df_within_3std.csv' located in the 'segmented/csv' subdirectory of the parent directory.
4. Loads the CSV file into a pandas DataFrame.
5. Prints the first few rows of the loaded DataFrame to verify successful loading.
"""


# Get current directory path
current_directory = os.getcwd()
# Get the path two levels above the current directory
analysis_root = os.path.dirname(current_directory)

# Load the saved CSV file as a DataFrame
file_path = analysis_root + '/outputs/segmented/csv/df_within_3std.csv'
df_loaded = pd.read_csv(file_path)

# Check if the DataFrame was loaded correctly
print(df_loaded.head())

## Detect per‑cell MCP spots (parallel)

Identify the brightest MCP spot inside each cell mask and store its coordinates.

- **Inputs:** `df_within_3std.csv`, processed stacks, and Cellpose masks.
- **Outputs:** `df_loaded_updated.csv` with `spot_z`, `spot_y`, `spot_x`.

In [ ]:
# %% --------------------------------  Imports & paths  --------------------------------
import os, sys, logging, multiprocessing as mp
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import tifffile as tiff
from skimage.measure import regionprops

import bigfish.detection as detection     # Heavy imports are also used by workers, so it's OK to import at the top

# ------------ Logging settings (optional) ------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(processName)s %(levelname)s: %(message)s"
 )

# ------------ Directory settings ------------
BASE_DIR = Path.cwd().parent              # Example: …/250429-live1/
PROC_DIR = BASE_DIR / "outputs" /'processed_1'
MASK_DIR = BASE_DIR / "outputs" /'segmented' / 'seg_tif'
CSV_DIR  = BASE_DIR / "outputs" /'segmented' / 'csv'
CSV_DIR.mkdir(exist_ok=True, parents=True)

# %% -------------------------------  I/O helpers  --------------------------------------
def save_df_loaded(df: pd.DataFrame, file_path: Path):
    df.to_csv(file_path, index=False)

def load_df_loaded(file_path: Path) -> pd.DataFrame:
    return pd.read_csv(file_path)

# %% ------------------------------  per-row worker  ------------------------------------
def process_row(args):
    """
    args : tuple(row_index:int, row:pd.Series)
    Returns : (row_index, (z,y,x)) or (row_index, None)
    """
    row_index, row = args
    image_id   = row['image_id']
    cell_id    = int(row['cell_id'])
    threshold  = row['mTetR_mean_intensity'] / 20

    try:
        # Generate paths
        mask_path = MASK_DIR / f"{image_id}.tiff"
        img_path  = PROC_DIR / f"{image_id}.tif"

        mask_img  = tiff.imread(mask_path)
        proc_img  = tiff.imread(img_path)                # shape (Z,C,Y,X)

        # cell mask and bbox
        cell_mask = mask_img == cell_id
        props = regionprops(cell_mask.astype(np.uint8))
        if not props:
            return row_index, None
        min_r, min_c, max_r, max_c = props[0].bbox

        # 3rd channel, crop
        crop = proc_img[:, 2, min_r:max_r, min_c:max_c]  # Z,Y,X
        crop_mask = cell_mask[min_r:max_r, min_c:max_c]

        # spot detection (bigfish)
        spots = detection.detect_spots(
            images=crop,
            threshold=threshold,
            return_threshold=False,
            voxel_size=(500, 130, 130),
            spot_radius=(500, 300, 300)
        )

        # Take the maximum value limited to spots inside the cell
        best = None
        best_val = -np.inf
        for z, y, x in spots:
            y_i, x_i = int(y), int(x)
            if 0 <= y_i < crop_mask.shape[0] and 0 <= x_i < crop_mask.shape[1]:
                if crop_mask[y_i, x_i]:
                    val = crop[int(z), y_i, x_i]
                    if val > best_val:
                        best_val, best = val, (int(z), y_i+min_r, x_i+min_c)

        return row_index, best
    except Exception as e:
        logging.error("row %s failed: %s", row_index, e)
        return row_index, None

# %% ------------------------------  main parallel  -------------------------------------
def run_parallel(df_loaded: pd.DataFrame, workers: int = 8) -> pd.DataFrame:
    # ctx = mp.get_context("fork")  # Optional: enable if you need an explicit fork context
    tasks = [(i, row) for i, row in df_loaded.iterrows()]

    with ProcessPoolExecutor(max_workers=workers) as pool:  # mp_context removed for notebook stability
        futures = [pool.submit(process_row, t) for t in tasks]
        for fut in as_completed(futures):
            idx, spot = fut.result()
            if spot is not None:
                z, y, x = spot
                df_loaded.loc[idx, ['spot_z', 'spot_y', 'spot_x']] = (z, y, x)
    return df_loaded

# %% -----------------------------  entry-point cell  -----------------------------------
if __name__ == "__main__":
    # In Notebook, just run this cell once
    mp.set_start_method("fork", force=True)

    df_loaded = load_df_loaded(CSV_DIR / "df_within_3std.csv")
    df_loaded = run_parallel(df_loaded, workers=os.cpu_count()-1)
    save_df_loaded(df_loaded, CSV_DIR / "df_loaded_updated.csv")
    
    display(df_loaded.head())             # Check on Notebook
    df_loaded_updated = df_loaded

## Crop spot‑centered patches

Extract 19x19 (3‑channel) crops centered on the detected MCP spots for downstream averaging.

- **Inputs:** `df_loaded_updated` and processed stacks.
- **Outputs:** per‑cell TIFF crops in `outputs/segmented/spots_img/`.

In [ ]:

# %% -------------------- Imports & Common Settings --------------------
import os, logging, multiprocessing as mp
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import tifffile as tiff
import pandas as pd

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(processName)s %(levelname)s: %(message)s")

# Assumes df_loaded_updated already exists from the previous cell, etc.
assert 'df_loaded_updated' in globals(), "Please prepare df_loaded_updated first"

BASE_DIR = Path.cwd().parent                 # Example: …/250429-live1/
PROC_DIR = BASE_DIR / "outputs" / 'processed_1'
SAVE_DIR = BASE_DIR / "outputs" / 'segmented' / 'spots_img'
SAVE_DIR.mkdir(exist_ok=True, parents=True)

N_WORKERS = max(1, os.cpu_count() - 1)       # CPU cores minus 1

# %% -------------------- Function to process one row --------------------
def crop_one_row(args):
    """args = (row_index, row_series)"""
    idx, row = args
    try:
        image_id  = row['image_id']
        cell_id   = int(row['cell_id'])
        z, y, x   = map(int, (row['spot_z'], row['spot_y'], row['spot_x']))

        img_path  = PROC_DIR / f"{image_id}.tif"
        img       = tiff.imread(img_path)                     # (Z,C,Y,X)

        # Skip cells near the edge
        if y < 9 or x < 9 or y > img.shape[2]-10 or x > img.shape[3]-10:
            return f"skip edge cell {cell_id}"

        crop = img[z, :, y-9:y+10, x-9:x+10]                  # shape (3,19,19)
        if crop.shape != (3, 19, 19):
            return f"bad shape {cell_id}"

        save_path = SAVE_DIR / f"{image_id}_{cell_id}.tif"
        tiff.imwrite(save_path, crop)
        return f"saved {save_path.name}"
    except Exception as e:
        return f"error row {idx}: {e}"

# %% -------------------- Parallel execution ------------------------------
def run_parallel(df: pd.DataFrame, workers: int = N_WORKERS):
    ctx = mp.get_context("fork")               # "fork" is safe in Notebook
    tasks = [(i, row) for i, row in df.iterrows()]

    with ProcessPoolExecutor(max_workers=workers) as pool:
        for fut in as_completed(pool.submit(crop_one_row, t) for t in tasks):
            msg = fut.result()
            logging.info(msg)

# %% -------------------- Entry point ----------------------
if __name__ == "__main__":
    mp.set_start_method("fork", force=True)    # Only needs to be called oncec
    run_parallel(df_loaded_updated)

## Random control: sample points within masks

Generate random (z, y, x) coordinates inside each cell mask to build control crops.

- **Inputs:** `df_loaded_updated.csv` and mask TIFFs.
- **Outputs:** `df_loaded_random.csv` with random coordinates.

In [ ]:

# Choose random locations.

# %% --------------------------- Imports & Common Settings ---------------------------
import os, logging, multiprocessing as mp, random
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import tifffile as tiff
from skimage.measure import regionprops

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(processName)s %(levelname)s: %(message)s")

# ------------ Directories ------------
BASE_DIR   = Path.cwd().parent              # Example: …/250429-live1/
MASK_DIR   = BASE_DIR / "outputs" /'segmented' / 'seg_tif'
CSV_DIR    = BASE_DIR / "outputs" / 'segmented' / 'csv'
CSV_DIR.mkdir(exist_ok=True, parents=True)

# ------------ Load global DataFrame ------------
df_loaded = pd.read_csv(CSV_DIR / 'df_loaded_updated.csv')
N_WORKERS = max(1, os.cpu_count() - 1)      # CPU cores - 1

Z_TOTAL = 21                 
Z_LO, Z_HI = 7, 14         

# %% -------------------------- Function to process one row -----------------------------
def process_row_random(row_index: int):
    """
    Get (image_id, cell_id) from the row at row_index,
    and randomly select 5 points (z, y, x) within the corresponding mask.
    Returns: list[dict] or [] (if cell not found/error)
    """
    try:
        row = df_loaded.iloc[row_index]               # Directly reference DF since using fork
        image_id, cell_id = row['image_id'], int(row['cell_id'])

        mask_path = MASK_DIR / f"{image_id}.tiff" 
        mask_img  = tiff.imread(mask_path)
        cell_mask = (mask_img == cell_id)

        ys, xs = np.where(cell_mask)
        if ys.size == 0:
            return []

        out = []
        for _ in range(5):
            idx = np.random.randint(0, ys.size)
            y   = int(ys[idx])
            x   = int(xs[idx])

            z = random.randint(Z_LO, Z_HI)

            out.append(dict(
                image_id=image_id,
                cell_id=cell_id,
                random_z=z, random_y=y, random_x=x
            ))
        return out

    except Exception as e:
        logging.error("row %s failed: %s", row_index, e)
        return []

# %% --------------------------- Parallel execution --------------------------------------
def run_parallel_random(workers: int = N_WORKERS) -> pd.DataFrame:
    ctx = mp.get_context("fork")                      # "fork" is safe in Notebook
    with ProcessPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(process_row_random, i) for i in range(len(df_loaded))]

        all_rows = []
        for fut in as_completed(futures):
            all_rows.extend(fut.result())

    return pd.DataFrame(all_rows)

# %% --------------------------- Entry point ------------------------------
if __name__ == "__main__":
    mp.set_start_method("fork", force=True)           # Call only once
    df_loaded_random = run_parallel_random()
    display(df_loaded_random.head())

    out_path = CSV_DIR / 'df_loaded_random.csv'
    df_loaded_random.to_csv(out_path, index=False)
    print(f"saved → {out_path}")

## Random control: downsample to one point per cell

Reduce random points to a single coordinate per cell to avoid over‑representation.

- **Input:** `df_loaded_random.csv` (five points per cell).
- **Output:** `df_loaded_random2.csv` (one point per cell).

In [ ]:

# Load the saved CSV file as a DataFrame
file_path = analysis_root + '/outputs/segmented/csv/df_loaded_random.csv'
df_loaded_random = pd.read_csv(file_path)

# Five data points were obtained for each, but that's too many, so keep only one for each.
df_loaded_random = df_loaded_random.iloc[::5]

df_loaded_random = df_loaded_random.reset_index(drop=True)

# Save the processed df_loaded_random
df_loaded_random.to_csv(analysis_root + "/outputs/segmented/csv/df_loaded_random2.csv", index=False)

# Check if the DataFrame was loaded correctly
df_loaded_random

## Random control: extract random crops

Create 19x19 (3‑channel) crops centered at random coordinates to serve as control patches.

- **Input:** `df_loaded_random2.csv`.
- **Output:** random crop TIFFs in `outputs/segmented/spots_img_random/`.

In [ ]:

# %% ------------------------- Imports & Global Settings -------------------------
import os, logging, multiprocessing as mp
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import tifffile as tiff
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(processName)s %(levelname)s: %(message)s")

# Assumes df_loaded_random already exists in the Notebook
assert 'df_loaded_random' in globals(), "Please load `df_loaded_random` first"

# --------- Paths ---------
BASE_DIR  = Path.cwd().parent                      # …/250429-live1/
PROC_DIR  = BASE_DIR / "outputs" / 'processed_1'
SAVE_DIR  = BASE_DIR / "outputs" / 'segmented' / 'spots_img_random'
SAVE_DIR.mkdir(exist_ok=True, parents=True)

N_WORKERS = max(1, os.cpu_count() - 1)             # CPU cores minus 1

# %% -------------------- Function to process one row (=1 spot) ------------------------
def save_one_spot(idx: int):
    """
    For the (image_id, cell_id, random_z, y, x) in row idx,
    extract a 19x19 (3ch) crop and save as TIFF.
    Returns: result message string
    """
    try:
        row = df_loaded_random.iloc[idx]           # Share DF if using fork
        image_id   = row['image_id']
        cell_id    = int(row['cell_id'])
        z, y, x    = map(int, (row['random_z'], row['random_y'], row['random_x']))

        img_path   = PROC_DIR / f"{image_id}.tif"
        img        = tiff.imread(img_path)         # (Z,C,Y,X)

        # Skip near the edge
        if y < 9 or x < 9 or y > img.shape[2]-10 or x > img.shape[3]-10:
            return f"skip edge {cell_id}"

        crop = img[z, :, y-9:y+10, x-9:x+10]       # (3,19,19)
        if crop.shape != (3, 19, 19):
            return f"bad shape {cell_id}"

        out_path = SAVE_DIR / f"{image_id}_{cell_id}.tif"
        tiff.imwrite(out_path, crop)
        return f"saved {out_path.name}"
    except Exception as e:
        return f"error {idx}: {e}"

# %% ----------------------- Parallel execution function ------------------------------------
def run_parallel_save(workers=N_WORKERS):
    ctx = mp.get_context('fork')                   # "fork" is safe in Notebook
    with ProcessPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(save_one_spot, i) for i in range(len(df_loaded_random))]
        for fut in as_completed(futures):
            logging.info(fut.result())

# %% ----------------------- Entry point ---------------------------------
if __name__ == "__main__":
    mp.set_start_method("fork", force=True)        # Ignored if already fork
    run_parallel_save()

## 19x19 crop analysis utilities (imports)

Load analysis dependencies and define core utility functions used for ON/OFF calling, random controls, and plotting.

- **Expected image shape:** 19x19x3 crops (SNAPtag, MCP, mTetR).
- **Outputs:** in‑memory helper functions and constants.

In [ ]:
# %% ------------------------- imports & common functions -------------------------
import os, glob, math, warnings, multiprocessing as mp, logging
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import tifffile as tiff
from skimage import io
from scipy.spatial import distance
import trackpy as tp
from matplotlib.ticker import MaxNLocator

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(processName)s %(levelname)s: %(message)s")

# ----------------- minimal replacement of oh_functions -----------------
def min_dist_2d(ax, ay, bx, by):
    """Return nearest index in (bx,by) to point (ax,ay), and its distance (pixels)."""
    ax = float(np.asarray(ax).ravel()[0])
    ay = float(np.asarray(ay).ravel()[0])

    bx = np.asarray(bx, dtype=float)
    by = np.asarray(by, dtype=float)

    dists = np.hypot(bx - ax, by - ay)   # sqrt(dx^2 + dy^2)
    idx = int(np.argmin(dists))
    return idx, float(dists[idx])

def intensity_calc(yx, img, pad=3, return_aux=False):
    """
    yx : ndarray shape (1,2)  (y, x)
    img: ndarray
    """
    img_p = np.pad(img, pad_width=pad, mode="constant")
    yx_p = np.asarray(yx, dtype=float) + pad

    df = tp.refine_com(img_p, img_p, radius=pad, coords=yx_p)

    mass_col = "raw_mass" if "raw_mass" in df.columns else "mass"
    area = math.pi * (pad ** 2)

    signal_int = df[mass_col] / area
    img_mean = float(np.mean(img))
    r_mass = signal_int / img_mean

    df["r_mass"] = r_mass
    if return_aux:
        df["signal_int"] = signal_int
        df["img_mean"] = img_mean
        return df[["r_mass", "signal_int", "img_mean"]]
    return df[["r_mass"]]


# %% -------------------- worker to analyze a single TIF --------------------
def analyze_one_tif(path, threshold_mTetR, threshold_MCP):
    """
    Parameters
    ----------
    path : str
        TIF file path (19�19�3, uint16)

    Returns
    -------
    result : dict | None
        dict = {
            'df_row' : one-row DataFrame,
            'state'  : 'ON' or 'OFF',
            'imgs'   : [img0, img1, img2]  (each 19�19 float32)
        }
    """
    try:
        img = io.imread(path)
        if img.shape != (19, 19, 3):
            return None

        img0, img1, img2 = [img[:, :, ch].astype(np.float32) for ch in range(3)]

        # ---------- spot detection ----------
        df0 = tp.locate(img0, 5)
        if df0.empty:
            return None
        df0 = tp.refine_leastsq(df0, img0, 5, fit_function='gauss').reset_index(drop=True)

        df1 = tp.locate(img1, 5, topn=1)
        df1 = (tp.refine_leastsq(df1, img1, 5, fit_function='gauss')
               if not df1.empty else pd.DataFrame())
        df1_int        = intensity_calc(df1[['y','x']].values, img1) if not df1.empty else None
        df1['r_mcp']   = df1_int['r_mass'].to_numpy()[0] if df1_int is not None else np.nan   # a

        df2 = tp.locate(img2, 5, topn=1)
        if df2.empty:
            df2 = pd.DataFrame([[np.nan]*7], columns=['x','y','mass','size','ecc','signal','raw_mass'])
        else:
            df2 = tp.refine_leastsq(df2, img2, 5, fit_function='gauss')
        df2[['x','y']] = 9.0
        # refine & intensity
        df2_int = intensity_calc(df2[['y','x']].values, img2, return_aux=True)

        df2['r_mTetR']         = df2_int['r_mass'].iat[0]
        df2['mTetR_signal_int'] = df2_int['signal_int'].iat[0]
        df2['mTetR_mean']       = df2_int['img_mean'].iat[0]

        # threshold
        if (df2['r_mTetR'] < threshold_mTetR).all():
            return None

        # distances & state
        df2 = df2.reset_index(drop=True)
        idx, dist_s = min_dist_2d(df2['x'], df2['y'], df0['x'], df0['y'])
        df2['dist_s_t'] = dist_s * 130
        df2['r_snap']   = df0.iloc[idx, -1]

        if not df1.empty and not df1['r_mcp'].isnull().all():
            _, dist_m = min_dist_2d(df2['x'], df2['y'], df1['x'], df1['y'])
            on = (df1['r_mcp'].values[0] >= threshold_MCP) and (dist_m <= 3)
            df2['state'] = 'ON' if on else 'OFF'
            df2['dist_m_t'] = dist_m * 130
            df2['r_mcp'] = df1['r_mcp'].values[0]
        else:
            df2['state'] = 'OFF'
            df2['r_mcp'] = np.nan

        df2['image_path'] = str(path)


        return dict(
            df_row = df2,               # 新しい列 mTetR_mean, mTetR_signal_int を含む
            state  = df2['state'].iloc[0],
            imgs   = [img0, img1, img2]
        )

    except Exception as e:
        logging.error("failed %s : %s", path, e)
        return None

# %% -------------------- store images into on/off list --------------------
def append_imgs(bucket, imgs):
    for ch in range(3):
        bucket[ch].append(imgs[ch])

# %% -------------------- main parallel processing function ----------------------
def main_process_without_shift_parallel(input_dir,
                                        target_name,
                                        threshold_mTetR=1.0,
                                        threshold_MCP=1.0,
                                        workers=None):
    """
    Parallel version of main_process_without_shift
    ----------------------------------
    Parameters
    ----------
    input_dir      : directory that contains images  
    target_name    : common file prefix (e.g. 'well001_series')  
    threshold_mTetR, threshold_MCP : thresholds  
    workers        : number of parallel workers (None = CPU cores 1)

    Returns
    -------
    df_all, on_images(dict), off_images(dict), averaged(dict)
    """
    if workers is None:
        workers = max(1, os.cpu_count()-1)

    paths = sorted(Path(input_dir).glob(f"{target_name}*.tif"))
    if not paths:
        raise FileNotFoundError(f"No TIFs matched '{target_name}' in {input_dir}")

    on_imgs  = {0:[],1:[],2:[]}
    off_imgs = {0:[],1:[],2:[]}
    rows = []

    ctx = mp.get_context("fork")
    with ProcessPoolExecutor(max_workers=workers, mp_context=ctx) as pool:
        futs = [pool.submit(analyze_one_tif, p, threshold_mTetR, threshold_MCP) for p in paths]
        for fut in as_completed(futs):
            res = fut.result()
            if res is None:                           # skipped / failed
                continue
            rows.append(res['df_row'])
            append_imgs(on_imgs if res['state']=='ON' else off_imgs, res['imgs'])

    df_all = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

    # --- averaging (no shift) ---
    averaged = {}
    for ch in range(3):
        if on_imgs[ch]:
            averaged[f'channel_{ch}_ON']  = np.mean(on_imgs[ch],  axis=0)
        if off_imgs[ch]:
            averaged[f'channel_{ch}_OFF'] = np.mean(off_imgs[ch], axis=0)

    return df_all, on_imgs, off_imgs, averaged

# %% ----------------------- usage example ----------------------
"""
input_dir   = '/path/to/tif_folder'
target_name = 'well001_series'          # example
df_all, on_dict, off_dict, ave = main_process_without_shift_parallel(
    input_dir, target_name,
    threshold_mTetR=0.5, threshold_MCP=0.5, workers=8
)
display(df_all.head())
"""




# %% --------------------------- imports ---------------------------------
import os, glob, time, multiprocessing as mp
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
from skimage import io            # tifffile.imread is also acceptable

# %% -------------------------- worker -----------------------------------
def _read_one(path):
    """
    Parameters
    ----------
    path : pathlib.Path | str

    Returns
    -------
    np.ndarray | None    # image (19,19,3), uint16 / uint8
    """
    try:
        img = io.imread(str(path))
        return img if img.shape == (19, 19, 3) else None
    except Exception:
        return None

# %% -------------------- Parallel average / median ----------------------
def average_or_median_images_random_parallel(input_dir,
                                             target_name,
                                             workers=None):
    """
    Read all TIFs whose prefix matches the given one  
    and return the mean / median image per channel (no shift).

    Notes
    -----
    * Assumes fixed image size of 19�19�3  
    * On Ubuntu notebooks, mp_context='fork' works safely
    """
    t0 = time.time()
    paths = sorted(Path(input_dir).glob(f"*{target_name}*.tif"))
    if not paths:
        print("No files found")
        return None, None, None, 0

    if workers is None:
        workers = max(1, os.cpu_count() - 1)

    ctx = mp.get_context('fork')       # safe even in notebooks
    imgs = []
    with ProcessPoolExecutor(max_workers=workers, mp_context=ctx) as pool:
        futs = [pool.submit(_read_one, p) for p in paths]
        for i, fut in enumerate(as_completed(futs), 1):
            img = fut.result()
            if img is not None:
                imgs.append(img)
            if i % 50 == 0:
                print(f"{i}/{len(paths)} files processed")

    if not imgs:
        print("All images were invalid shape")
        return None, None, None, 0

    on_images = np.stack(imgs, axis=0).astype(np.float32)   # (N,19,19,3)

    # mean / median (per channel)
    ave = np.mean(on_images, axis=0)      # (19,19,3)
    med = np.median(on_images, axis=0)    # (19,19,3)

    # reshape to (3,19,19)
    ave = np.transpose(ave, (2,0,1))
    med = np.transpose(med, (2,0,1))

    print(f"Done in {time.time()-t0:.1f}s   images = {len(imgs)}")
    return on_images, ave, med, len(imgs)

# %% ---------------------- example usage --------------------------------
"""
input_dir   = '/path/to/tif_folder'
target_name = 'well001_series'

imgs, ave, med, n = average_or_median_images_random_parallel(
    input_dir, target_name, workers=8)

# ave[0] : channel-0 mean image (19�19)
# med[1] : channel-1 median image &
"""



# display and return median image
def process_images_median(images, state, channel):
    if images:
        image_averaged = np.median(images, axis=0)
        # display(image_averaged, f'{state} State Channel {channel} - Averaged')
        return image_averaged


# Radial intensity calculation (same as before)
def calculate_radial_intensity(image, center, max_radius):
    cy, cx = center
    y, x = np.indices((image.shape))
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2).astype(np.int64)
    mask = r <= max_radius
    tbin = np.bincount(r[mask].ravel(), image[mask].ravel())
    nr = np.bincount(r[mask].ravel())
    return tbin / nr


# wrapper function
def show_all_channels_without_random(averaged_channel_0_on, averaged_channel_0_off,
                      averaged_channel_1_on, averaged_channel_1_off,
                      averaged_channel_2_on, averaged_channel_2_off,
                      snap_title="SNAP"):
    center = (9, 9)  # center of 19�19
    max_radius = 9

    # Channel 0 (Ser2ph)
    show_channel_plot_without_random(averaged_channel_0_on, averaged_channel_0_off,
                      f'{snap_title} ON', f'{snap_title} OFF',
                      snap_title, center, max_radius)

    # Channel 1 (MCP)
    show_channel_plot_without_random(averaged_channel_1_on, averaged_channel_1_off,
                      'MCP ON', 'MCP OFF', 'MCP', 
                      center, max_radius)

    # Channel 2 (mTetR)
    show_channel_plot_without_random(averaged_channel_2_on, averaged_channel_2_off,
                      'mTetR ON', 'mTetR OFF', 'mTetR', 
                      center, max_radius)




# function to display ON, OFF, and random-average images side by side
def show_channel_plot(on_image, off_image, random_image, active_title, inactive_title, random_title, channel_title, center, max_radius):
    """Display ON/OFF/Random-average images and radial plot for each channel side by side."""

    # create plots
    fig, (ax_on, ax_off, ax_random, ax_radial) = plt.subplots(1, 4, figsize=(16, 4))

    # calculate vmin/vmax to align colorbars
    vmin = min(np.nanmin(on_image), np.nanmin(off_image), np.nanmin(random_image))
    vmax = max(np.nanmax(on_image), np.nanmax(off_image), np.nanmax(random_image))

    # 2D intensity plot for ON state
    im_on = ax_on.imshow(on_image, cmap='viridis', origin='lower', vmin=vmin, vmax=vmax)
    ax_on.set_title(active_title, fontsize=14)
    ax_on.set_xticks([]), ax_on.set_yticks([])

    # 2D intensity plot for OFF state
    im_off = ax_off.imshow(off_image, cmap='viridis', origin='lower', vmin=vmin, vmax=vmax)
    ax_off.set_title(inactive_title, fontsize=14)
    ax_off.set_xticks([]), ax_off.set_yticks([])

    # plot random-average image
    im_random = ax_random.imshow(random_image, cmap='viridis', origin='lower', vmin=vmin, vmax=vmax)
    ax_random.set_title(random_title, fontsize=14)
    ax_random.set_xticks([]), ax_random.set_yticks([])

    # place colorbar below ON/OFF/Random plots (adjust width)
    cbar_x0 = (ax_on.get_position().x0 + ax_random.get_position().x1) / 2 - 0.05  # adjust position to center
    cbar_ax = fig.add_axes([cbar_x0, ax_random.get_position().y0 - 0.05, 0.25, 0.02])  # width = 0.25
    plt.colorbar(im_on, cax=cbar_ax, orientation='horizontal')

    # Radial Intensity Distribution Plot
    active_intensity = calculate_radial_intensity(on_image, center, max_radius)
    inactive_intensity = calculate_radial_intensity(off_image, center, max_radius)
    random_intensity = calculate_radial_intensity(random_image, center, max_radius)

    ax_radial.plot(active_intensity, color='red', label='ON', linewidth=2)
    ax_radial.plot(inactive_intensity, color='blue', label='OFF', linewidth=2)
    ax_radial.plot(random_intensity, color='green', label='Random avg', linewidth=2)

    ax_radial.set_xlabel('Radius (pixels)', fontsize=14)
    ax_radial.set_ylabel('Intensity', fontsize=14)

    # show legend on the right of the plot
    ax_radial.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=14)

    ax_radial.set_xticks(np.arange(0, max_radius + 1, 2))
    ax_radial.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax_radial.grid(True)
    ax_radial.spines['top'].set_visible(False)
    ax_radial.spines['right'].set_visible(False)
    ax_radial.tick_params(axis='x', labelsize=14)
    ax_radial.tick_params(axis='y', labelsize=14)
    ax_radial.set_title(f'{channel_title} Radial Intensity')

    # show figure
    plt.tight_layout(pad=2.0)
    plt.show()

# wrapper function
def show_all_channels(averaged_channel_0_on, averaged_channel_0_off, processed_image_ave_random_0,
                      averaged_channel_1_on, averaged_channel_1_off, processed_image_ave_random_1,
                      averaged_channel_2_on, averaged_channel_2_off, processed_image_ave_random_2,
                      snap_title="SNAP"):
    center = (9, 9)  # center of 19�19
    max_radius = 9

    # Channel 0 (Ser2ph)
    show_channel_plot(averaged_channel_0_on, averaged_channel_0_off, processed_image_ave_random_0,
                      f'{snap_title} ON', f'{snap_title} OFF', f'{snap_title} Random Avg',
                      snap_title, center, max_radius)

    # Channel 1 (MCP)
    show_channel_plot(averaged_channel_1_on, averaged_channel_1_off, processed_image_ave_random_1,
                      'MCP ON', 'MCP OFF', 'MCP Random Avg', 
                      'MCP', center, max_radius)

    # Channel 2 (mTetR)
    show_channel_plot(averaged_channel_2_on, averaged_channel_2_off, processed_image_ave_random_2,
                      'mTetR ON', 'mTetR OFF', 'mTetR Random Avg', 
                      'mTetR', center, max_radius)


## Configure input/output directories

Define the input directory containing spot‑centered crops and create the output directory for selected images.

- **Input:** `outputs/segmented/spots_img/`.
- **Output:** `outputs/segmented/selected_spots_img/`.

In [ ]:
# Get current directory path
current_directory = os.getcwd()
# Get the path two levels above the current directory
analysis_root = os.path.dirname(current_directory)

input_dir = os.path.join(analysis_root + '/outputs/segmented/spots_img')
output_dir = os.path.join(analysis_root + "/outputs/segmented/selected_spots_img")
# Create the directory if it does not exist
os.makedirs(output_dir, exist_ok=True)

## ON/OFF calling and per‑well aggregation

Run the ON/OFF classification for each well using intensity thresholds, and collect per‑cell tables plus ON/OFF image stacks.

- **Inputs:** selected spot‑centered crops.
- **Outputs:** `df_3_all_*`, `on_images_*`, `off_images_*`, and averaged images per well.

In [ ]:

# well001
target_name = "well001"
df_3_all_well001, on_images_well001, off_images_well001, averaged_images_well001 = main_process_without_shift_parallel(input_dir, target_name, threshold_mTetR = 1.30, threshold_MCP = 1.15, workers=None)

#  # well002
# target_name = "well002"
# df_3_all_well002, on_images_well002, off_images_well002, averaged_images_well002 = main_process_without_shift_parallel(input_dir, target_name, threshold_mTetR = 1.30, threshold_MCP = 1.15, workers=None)


In [ ]:
df_3_all_well001

## Random controls: per‑well averages/medians

Compute average and median random crops per channel to serve as baseline references.

- **Inputs:** random crops in `outputs/segmented/spots_img_random/`.
- **Outputs:** per‑well random stacks and summary images.

In [ ]:
# random
input_dir = os.path.join(analysis_root + '/outputs/segmented/spots_img_random')

# well001
on_images_random_well001, processed_image_ave_random_well001, processed_image_med_random_well001, image_num_random_well001 = average_or_median_images_random_parallel(input_dir, "well001", workers=None)

# # well002
# on_images_random_well002, processed_image_ave_random_well002, processed_image_med_random_well002, image_num_random_well002 = average_or_median_images_random_parallel(input_dir, "well002", workers=None)

## Harmonize identifiers for downstream export

Parse `image_id` and `cell_id` from file names to standardize keys across tables.

- **Input:** `df_3_all_well001` (and other wells if enabled).
- **Output:** new `image_id` and `cell_id` columns.

In [ ]:
import pandas as pd
import re

# Function to extract image_id and cell_id
def extract_ids(image_path):
    # Extract filename
    file_name = image_path
    # Use regex to extract image_id and cell_id
    match = re.match(r'.*/(.*_series-\d+)_(\d+)\.tif', file_name)
    if match:
        image_id = match.group(1)
        cell_id = match.group(2)
        return image_id, cell_id
    return None, None

# Add new columns
df_3_all_well001['image_id'], df_3_all_well001['cell_id'] = zip(*df_3_all_well001['image_path'].map(extract_ids))
# df_3_all_well002['image_id'], df_3_all_well002['cell_id'] = zip(*df_3_all_well002['image_path'].map(extract_ids))

## Persist ON/OFF and random stacks

Save per‑well ON/OFF image stacks and random controls to pickle files for reuse.

- **Output:** `outputs/pkl/saved_onoff_data_wellXXX.pkl`.

In [ ]:
import os
import pickle

# Save ON/OFF/random stacks into per-well pickle files.
out_dir = os.path.join(analysis_root, "outputs", "pkl")
os.makedirs(out_dir, exist_ok=True)

wells = [f"well{idx:03d}" for idx in range(1, 9)]  # adjust if needed
keys_to_collect = ["on_images", "off_images", "on_images_random"]

n_written = 0
skipped = []

for well in wells:
    payload = {}
    missing = []
    for key in keys_to_collect:
        var_name = f"{key}_{well}"
        obj = globals().get(var_name, None)
        if obj is None:
            missing.append(var_name)
        else:
            payload[key] = obj

    if not payload:
        skipped.append(well)
        continue

    if missing:
        print(f"Warning ({well}): missing -> {', '.join(missing)}")

    out_pkl = os.path.join(out_dir, f"saved_onoff_data_{well}.pkl")
    with open(out_pkl, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
    n_written += 1
    print(f"Saved {list(payload.keys())} to {out_pkl}")

if skipped:
    print("Skipped wells (no variables found):", ", ".join(skipped))

print(f"Done. Wrote {n_written} per-well pickle file(s) to {out_dir}")

In [ ]:
from pathlib import Path
import pandas as pd

# Merge segmented per-cell CSV with ON/OFF calls (df_3_all_well*) and save per-well CSVs.
thr_mtetr = 1.30
thr_mcp = 1.15

def _normalize_image_id_series(s: pd.Series) -> pd.Series:
    s = s.astype(str)
    parts = s.str.extract(r"^(well\d{3})_series-(\d+)$")
    mask = parts[0].notna() & parts[1].notna()
    out = s.copy()
    out.loc[mask] = parts.loc[mask, 0] + "_series-" + parts.loc[mask, 1].str.zfill(3)
    return out

def _derive_ids_from_image_path(s: pd.Series) -> tuple[pd.Series, pd.Series]:
    s = s.astype(str)
    well = s.str.extract(r"(well\d{3})", expand=False)
    series = s.str.extract(r"series[-_]?0*(\d+)", expand=False)
    image_id = pd.Series([pd.NA] * len(s), index=s.index, dtype="object")
    ok = well.notna() & series.notna()
    image_id.loc[ok] = well.loc[ok] + "_series-" + series.loc[ok].astype(str).str.zfill(3)

    cell = s.str.extract(r"cell[-_]?0*(\d+)", expand=False)
    if cell.isna().all():
        # common pattern: .../well001_series-001_22.tif
        cell = s.str.extract(r"_(\d+)\.[A-Za-z0-9]+$", expand=False)
    if cell.isna().all():
        cell = s.str.extract(r"_(\d+)$", expand=False)
    if cell.isna().all():
        cell = s.str.extract(r"label[-_]?0*(\d+)", expand=False)

    return image_id, cell

segmented_csv = Path(analysis_root) / "outputs" / "segmented" / "csv" / "df_loaded_updated.csv"
if not segmented_csv.exists():
    raise FileNotFoundError(f"Not found: {segmented_csv}")

df_loaded_updated = pd.read_csv(segmented_csv)

# Normalize keys
df_loaded_updated["image_id"] = _normalize_image_id_series(df_loaded_updated["image_id"])
df_loaded_updated["cell_id"] = pd.to_numeric(df_loaded_updated["cell_id"], errors="coerce").astype("Int64")
df_loaded_updated = df_loaded_updated.dropna(subset=["cell_id"])
df_loaded_updated["cell_id"] = df_loaded_updated["cell_id"].astype(int)

# Add chamber_well if missing
if "chamber_well" not in df_loaded_updated.columns:
    df_loaded_updated["chamber_well"] = df_loaded_updated["image_id"].str.extract(r"^(well\d{3})", expand=False)

# Collect df_3_all_* tables present in the notebook (per well)
df_state_tables: dict[str, pd.DataFrame] = {}
for var_name, obj in list(globals().items()):
    if not var_name.startswith("df_3_all_well"):
        continue
    if not isinstance(obj, pd.DataFrame):
        continue
    well = var_name.replace("df_3_all_", "")  # e.g. df_3_all_well001 -> well001
    df_state_tables[well] = obj

if not df_state_tables:
    raise RuntimeError("No df_3_all_well* DataFrame found. Run the ON/OFF calling cells first.")

out_dir = Path(analysis_root) / "outputs" / "csv"
out_dir.mkdir(parents=True, exist_ok=True)

# Decide output naming: if only one well, match the legacy single-file name.
multiple_wells = len(df_state_tables) > 1

for well, df_3_all in sorted(df_state_tables.items()):
    df_state_source = df_3_all.copy()

    # Some df_3_all_* tables store identifiers only in image_path; derive image_id/cell_id if needed.
    if ("image_id" not in df_state_source.columns) or ("cell_id" not in df_state_source.columns):
        if "image_path" not in df_state_source.columns:
            raise KeyError(
                f"{well}: df_3_all missing image_id/cell_id and no image_path to derive them; available={list(df_state_source.columns)}"
            )
        s = df_state_source["image_path"].astype(str)
        if "image_id" not in df_state_source.columns:
            # Try strict capture first, then derive from well+series tokens
            img = s.str.extract(r"(well\d{3}_series-\d+)", expand=False)
            if img.isna().all():
                img, _ = _derive_ids_from_image_path(s)
            df_state_source["image_id"] = img
        if "cell_id" not in df_state_source.columns:
            # Try several patterns
            cell = s.str.extract(r"cell[_-]?(\d+)", expand=False)
            if cell.isna().all():
                _, cell = _derive_ids_from_image_path(s)
            df_state_source["cell_id"] = cell

    required_cols = {"image_id", "cell_id", "state"}
    missing_cols = required_cols.difference(df_state_source.columns)
    if missing_cols:
        raise KeyError(f"{well}: df_3_all missing columns {sorted(missing_cols)}; available={list(df_state_source.columns)}")

    df_state = df_state_source[["image_id", "cell_id", "state"]].copy()
    df_state["image_id"] = _normalize_image_id_series(df_state["image_id"])
    df_state["cell_id"] = pd.to_numeric(df_state["cell_id"], errors="coerce").astype("Int64")
    df_state = df_state.dropna(subset=["image_id", "cell_id"])
    df_state["cell_id"] = df_state["cell_id"].astype(int)

    if len(df_state) == 0:
        ex = None
        if "image_path" in df_state_source.columns and len(df_state_source) > 0:
            ex = str(df_state_source["image_path"].iloc[0])
        raise RuntimeError(
            f"{well}: df_state became empty after deriving ids. Example image_path={ex!r}. "
            "Please confirm the filename pattern in df_3_all_* and update the regex."
        )

    # Guard against duplicates (merge explosion)
    dup = df_state.duplicated(subset=["image_id", "cell_id"], keep=False)
    if dup.any():
        n_dup = int(dup.sum())
        print(f"Warning: {well}: {n_dup} duplicate (image_id, cell_id) rows in df_state; keeping first.")
        df_state = df_state.drop_duplicates(subset=["image_id", "cell_id"], keep="first")

    left_unique = not df_loaded_updated.duplicated(subset=["image_id", "cell_id"]).any()
    right_unique = not df_state.duplicated(subset=["image_id", "cell_id"]).any()
    validate = "one_to_one" if (left_unique and right_unique) else None

    df_out = df_loaded_updated.merge(
        df_state,
        on=["image_id", "cell_id"],
        how="inner",
        validate=validate,
    )

    # Column order (keep extras at the end)
    preferred = [
        "image_id",
        "cell_id",
        "SNAPtag_mean_intensity",
        "MCP_mean_intensity",
        "mTetR_mean_intensity",
        "cell_area_px",
        "well_no",
        "series_no",
        "within_3std",
        "chamber_well",
        "spot_z",
        "spot_y",
        "spot_x",
        "state",
    ]
    cols = [c for c in preferred if c in df_out.columns] + [c for c in df_out.columns if c not in preferred]
    df_out = df_out[cols]


    out_csv = out_dir / f"df_loaded_updated_with_state_{well}.csv"

    df_out.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}  (n={len(df_out)})")

## Load experiment metadata (pkl locations)

Import the Excel sheet that maps experimental conditions to pkl file locations.

- **Output:** `df_first_sheet` for downstream loading and renaming.

In [ ]:
import pandas as pd

excel_path = '../../../data/01_Snapshot_analysis/pkl/pkl_file_position_original.xlsx'
df_first_sheet = pd.read_excel(excel_path, sheet_name=0)
df_first_sheet

## Load and rename ON/OFF/random stacks from pkls

Read the per‑well pkl files listed in the metadata sheet and register standardized variable names for downstream plotting.

- **Inputs:** `df_first_sheet` and pkl files.
- **Outputs:** in‑memory variables like `{target}_{factor}_rep{n}_on_images`.

In [ ]:
import os
import re
import pickle

class NumpyCompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # Handle internal path changes between NumPy 2.x and 1.x
        if module.startswith("numpy._core"):
            module = "numpy.core" + module[len("numpy._core"):]
        return super().find_class(module, name)

def pickle_load_compat(file_obj):
    return NumpyCompatUnpickler(file_obj).load()

# Load and rename ON/OFF/random data from pkl_file paths listed in df_first_sheet


def _clean_text(s):
    # Normalize spaces and trim
    return str(s).replace('\u00A0', ' ').replace('\u3000', ' ').strip()

def _extract_int(s, default=None):
    # Get first integer found in the string
    m = re.search(r'\d+', _clean_text(s))
    return int(m.group()) if m else default

def _sanitize_ident(s):
    # Keep ASCII letters, numbers, and underscores for variable names
    s = _clean_text(s)
    s = re.sub(r'\W+', '_', s)
    s = s.strip('_')
    if not s:
        s = "x"
    if s[0].isdigit():
        s = "v_" + s
    return s

def _normalize_path(p):
    p = _clean_text(p)
    p = os.path.expanduser(p)
    if os.path.isabs(p):
        return os.path.normpath(p)
    return os.path.normpath(os.path.join(os.getcwd(), p))

# --- prepare rows (drop missing pkl_file) ---
rows = df_first_sheet.copy()
rows = rows.dropna(subset=['pkl_file'])

# Cache for loaded pkl files
loaded_pkls = {}
created_vars = []
warnings = []

for i, row in rows.iterrows():
    pkl_path = _normalize_path(row['pkl_file'])
    well_id = row.get('well_id', None)
    well_num = _extract_int(well_id) if well_id is not None else None
    well_key = f"well{well_num:03d}" if well_num is not None else None

    # Load pkl once per file
    if pkl_path not in loaded_pkls:
        try:
            with open(pkl_path, "rb") as f:
                loaded_pkls[pkl_path] = pickle_load_compat(f)
        except Exception as e:
            warnings.append(f"Failed to load {pkl_path}: {e}")
            continue

    blob = loaded_pkls[pkl_path]
    # Expect a dict that contains keys like on_images_well001 (legacy) or on_images (single-well)
    if well_key is None:
        key_map = {
            "on_images": "on_images",
            "off_images": "off_images",
            "on_images_random": "on_images_random",
        }
    else:
        key_map = {
            "on_images":         f"on_images_{well_key}",
            "off_images":        f"off_images_{well_key}",
            "on_images_random":  f"on_images_random_{well_key}",
        }

    # Build new variable name prefix: {STtag_KI_target}_{Factor_name}_rep{replicate}
    target = _sanitize_ident(row.get('STtag_KI_target', ''))
    factor = _sanitize_ident(row.get('Factor_name', ''))
    rep = _extract_int(row.get('replicate', ''), default=_clean_text(row.get('replicate', '')))
    rep_str = str(rep)

    prefix = f"{target}_{factor}_rep{rep_str}"

    # Create globals for each key if present
    for short_name, old_key in key_map.items():
        if isinstance(blob, dict) and old_key in blob:
            new_var = f"{prefix}_{short_name}"
            if new_var in globals():
                warnings.append(f"Overwriting existing variable: {new_var}")
            globals()[new_var] = blob[old_key]
            created_vars.append(new_var)
        else:
            warnings.append(f"Missing key in {os.path.basename(pkl_path)}: {old_key}")

# Summary
unique_pkls = len(loaded_pkls)
print(f"Loaded {unique_pkls} pkl file(s). Created {len(created_vars)} variable(s).")
if warnings:
    print("Warnings:")
    # Print unique warnings to keep output concise
    for msg in sorted(set(warnings)):
        print(" -", msg)

## Plotting utilities (composite panels)

Define shared plotting helpers (radial profiles, normalization, ECDF, and violin plots) used in the figure panels.

- **Inputs:** ON/OFF stacks and random controls.
- **Outputs:** reusable functions (no files written).

In [ ]:
# Additional imports (once near the top)
import matplotlib.transforms as mtransforms
import matplotlib.patheffects as PathEffects

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import MaxNLocator
from scipy.stats import mannwhitneyu

# Use Arial if available; fallback will be used if not installed
plt.rcParams.update({
    "font.family": "Arial",
    "figure.dpi": 120,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
})

# ---- Core constants and utilities (unchanged) ----------------------------
PIXEL_SIZE_UM = 0.13
CENTER = (9, 9)
R_MAX = 9

def _aggregate_image(img_list, method="mean"):
    arr = np.stack(img_list, axis=0).astype(np.float32)
    if method == "mean":
        return np.mean(arr, axis=0)
    elif method == "median":
        return np.median(arr, axis=0)
    else:
        raise ValueError("method must be 'mean' or 'median'")

def radial_profile(image, center=CENTER, r_max=R_MAX):
    yy, xx = np.indices(image.shape)
    r = np.sqrt((xx - center[1])**2 + (yy - center[0])**2).astype(np.int64)
    mask = (r <= r_max)
    counts = np.bincount(r[mask].ravel(), minlength=r_max+1)
    sums   = np.bincount(r[mask].ravel(), weights=image[mask].ravel(), minlength=r_max+1)
    prof = sums / np.maximum(counts, 1)
    r_um = np.arange(r_max + 1) * PIXEL_SIZE_UM
    return r_um, prof

# def radial_profiles_from_stack(img_list, rand_img=None, scale=1.0):
#     profs = []
#     for im in img_list:
#         ip = (im - rand_img) if rand_img is not None else im
#         _, pr = radial_profile(ip * scale)
#         profs.append(pr)
#     return np.stack(profs, axis=0) if len(profs) else np.empty((0, R_MAX+1))

def radial_profiles_from_stack(img_list, rand_img, scale=1.0):
    profs = []
    for img in img_list:
        arr = (img - rand_img) * scale           # order is important
        r, prof = radial_profile(arr)            # same center and R_MAX as 2D images
        profs.append(prof)
    return np.asarray(profs)

def sem(a, axis=0):
    a = np.asarray(a, dtype=float)
    n = np.sum(np.isfinite(a), axis=axis)
    return np.nanstd(a, axis=axis, ddof=1) / np.sqrt(np.maximum(n, 1))

def central_pixel_values(img_list, rand_img, scale, center=CENTER):
    cy, cx = center
    vals = []
    for im in img_list:
        vals.append(float((im - rand_img)[cy, cx] * scale))
    return np.array(vals, dtype=float)

def _format_p(p):
    return "<1e-99" if p < 1e-99 else f"{p:.2e}"

def _ensure_rand_avg_3d(rand_stack_4d, rand_avg_3d=None):
    if rand_avg_3d is None:
        assert rand_stack_4d is not None and rand_stack_4d.ndim == 4 and rand_stack_4d.shape[-1] == 3, \
            "rand_stack_4d is expected to have shape (N, 19, 19, 3)"
        rand_avg_3d = np.mean(rand_stack_4d, axis=0)  # (19,19,3)
    if rand_avg_3d.shape == (19, 19, 3):
        rand_avg_3d = np.transpose(rand_avg_3d, (2, 0, 1))
    return rand_avg_3d.astype(np.float32, copy=False)

def p_to_stars(p):
    return "****" if p < 1e-4 else ("***" if p < 1e-3 else ("**" if p < 1e-2 else ("*" if p < 0.05 else "n.s.")))

def compute_scale(on_list, off_list, rand_avg_img, mode="pixel_max", agg_method="mean"):
    def _agg(lst):
        arr = np.stack(lst, axis=0)
        return np.mean(arr, axis=0) if agg_method=="mean" else np.median(arr, axis=0)
    if mode == "pixel_max":
        on_avg  = _agg(on_list)  - rand_avg_img
        off_avg = _agg(off_list) - rand_avg_img
        maxv = float(np.nanmax([np.nanmax(on_avg), np.nanmax(off_avg)]))
    elif mode == "radial_max":
        on_profs  = radial_profiles_from_stack(on_list,  rand_img=rand_avg_img, scale=1.0)
        off_profs = radial_profiles_from_stack(off_list, rand_img=rand_avg_img, scale=1.0)
        maxv = float(np.nanmax([np.nanmean(on_profs, axis=0).max(),
                                np.nanmean(off_profs, axis=0).max()]))
    elif mode == "on_center":
        cps = central_pixel_values(on_list, rand_avg_img, scale=1.0)
        maxv = float(np.nanmean(cps)) if len(cps) else np.nan
    else:
        raise ValueError("normalize_mode must be 'pixel_max' or 'radial_max' or 'on_center'")
    return 100.0 / maxv if (np.isfinite(maxv) and maxv > 0) else 1.0

# Additional: compute ECDF (empirical CDF)
def _ecdf(vals: np.ndarray):
    """Returns: sorted x, cumulative y (0..1)."""
    if vals.size == 0:
        return np.array([0.0]), np.array([0.0])
    x = np.sort(vals)
    y = np.arange(1, x.size+1) / x.size
    return x, y

# ---- Single-row plotting (6 columns) ------------------------------------
def plot_channel_row(
    ax_row, ch, on_imgs_dict, off_imgs_dict,
    rand_stack_4d, rand_avg_3d, title_for_ch,
    normalize_mode="pixel_max", agg_method="mean",
    image_aspect="auto",
    show_p_value=True,
    bracket_axes_y=0.96,
    violin_headroom=0.15
):
    # Layout: [Active | Inactive | Radial | Legend | Violin | ECDF]
    ax_on, ax_off, ax_rad, ax_leg, ax_vio, ax_ecdf = ax_row

    # --- Inputs ---
    on_list  = on_imgs_dict.get(ch, [])
    off_list = off_imgs_dict.get(ch, [])
    assert len(on_list) > 0 and len(off_list) > 0, f"channel {ch}: Active / Inactive lists are empty"

    rand_avg_img = rand_avg_3d[ch]
    rand_list_ch = [rand_stack_4d[i, :, :, ch] for i in range(rand_stack_4d.shape[0])]

    # --- Scaling factor ---
    s = compute_scale(on_list, off_list, rand_avg_img, mode=normalize_mode, agg_method=agg_method)

    # --- Display images (mean/median minus random mean) ---
    on_disp  = (_aggregate_image(on_list,  method=agg_method) - rand_avg_img) * s
    off_disp = (_aggregate_image(off_list, method=agg_method) - rand_avg_img) * s

    # --- 2D image (diverging colormap) ---
    vabs = max(np.nanmax(np.abs(on_disp)), np.nanmax(np.abs(off_disp))) or 1.0
    norm = TwoSlopeNorm(vmin=-vabs, vcenter=0.0, vmax=vabs)
    im_on  = ax_on.imshow(on_disp,  cmap="coolwarm", norm=norm, origin="lower")
    im_off = ax_off.imshow(off_disp, cmap="coolwarm", norm=norm, origin="lower")
    for a, t in zip((ax_on, ax_off), (f"{title_for_ch} Active", f"{title_for_ch} Inactive")):
        a.set_title(t); a.set_xticks([]); a.set_yticks([]); a.set_aspect(image_aspect)

    # --- Show N in white text only for SNAPtag row ---
    if ch == 0:
        n_on, n_off = len(on_list), len(off_list)
        for a, n in ((ax_on, n_on), (ax_off, n_off)):
            a.text(0.02, 0.02, f"N = {n}",
                   color="white", fontsize=10, ha="left", va="bottom",
                   transform=a.transAxes,
                   path_effects=[PathEffects.withStroke(linewidth=2.0, foreground="black")])

    # --- Colorbar below Active/Inactive ---
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    cax = inset_axes(ax_off, width="220%", height="6%", loc="lower left",
                     bbox_to_anchor=(-1.1, -0.34, 1.1, 1), bbox_transform=ax_off.transAxes, borderpad=0)
    cb = plt.colorbar(im_on, cax=cax, orientation="horizontal")
    cb.set_label("Intensity (baseline-subtracted, max=100)")

    # --- Radial profiles (Active/Inactive/Random ± SEM) ---
    on_profs   = radial_profiles_from_stack(on_list,  rand_img=rand_avg_img, scale=s)
    off_profs  = radial_profiles_from_stack(off_list, rand_img=rand_avg_img, scale=s)
    rand_profs = radial_profiles_from_stack(rand_list_ch, rand_img=rand_avg_img, scale=s)

    r_um, _    = radial_profile(on_disp)
    mean_on,  sem_on  = np.nanmean(on_profs,  axis=0), sem(on_profs,  axis=0)
    mean_off, sem_off = np.nanmean(off_profs, axis=0), sem(off_profs, axis=0)
    mean_rnd, sem_rnd = np.nanmean(rand_profs, axis=0), sem(rand_profs, axis=0)

    line_on,  = ax_rad.plot(r_um, mean_on,  lw=2, label="Active")
    line_off, = ax_rad.plot(r_um, mean_off, lw=2, label="Inactive")
    line_rnd, = ax_rad.plot(r_um, mean_rnd, lw=2, label="Random")
    ax_rad.fill_between(r_um, mean_on-sem_on,   mean_on+sem_on,   alpha=0.25, color=line_on.get_color())
    ax_rad.fill_between(r_um, mean_off-sem_off, mean_off+sem_off, alpha=0.25, color=line_off.get_color())
    ax_rad.fill_between(r_um, mean_rnd-sem_rnd, mean_rnd+sem_rnd, alpha=0.20, color=line_rnd.get_color())
    ax_rad.set_xlabel("Radius (um)")
    ax_rad.set_ylabel("Intensity (max=100)")
    ax_rad.set_xticks(np.arange(0, (R_MAX+1)*PIXEL_SIZE_UM, 2*PIXEL_SIZE_UM))
    ax_rad.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax_rad.grid(True, alpha=0.3)
    ax_rad.set_title(f"{title_for_ch} radial intensity")

    # Legend in dedicated column
    ax_leg.axis("off")
    ax_leg.legend(handles=[line_on, line_off, line_rnd],
                  labels=["Active", "Inactive", "Random"], loc="center", frameon=False)

    # --- Violin (central pixel distribution) ---
    cp_on  = central_pixel_values(on_list,  rand_avg_img, s)
    cp_off = central_pixel_values(off_list, rand_avg_img, s)
    vp = ax_vio.violinplot([cp_on, cp_off], positions=[1, 2], showmeans=True, showmedians=True, widths=0.8)
    for b, col in zip(vp['bodies'], (line_on.get_color(), line_off.get_color())):
        b.set_facecolor(col); b.set_edgecolor("black"); b.set_alpha(0.4)
    for part in ('cbars','cmins','cmaxes','cmeans','cmedians'):
        if part in vp:
            vp[part].set_edgecolor('black'); vp[part].set_linewidth(1.0)
    ax_vio.set_xlim(0.5, 2.5)
    ax_vio.set_xticks([1, 2]); ax_vio.set_xticklabels(["Active", "Inactive"])
    ax_vio.set_ylabel("Central pixel (max=100)")
    ax_vio.grid(axis='y', alpha=0.3)
    ax_vio.set_title(f"{title_for_ch} center pixel")

    # Expand upper margin to fit MWU stars and p-values
    if len(cp_on) and len(cp_off):
        ymin = float(np.nanmin([cp_on.min(), cp_off.min()]))
        ymax = float(np.nanmax([cp_on.max(), cp_off.max()]))
        rng  = max(1.0, ymax - ymin)
        ax_vio.set_ylim(ymin - 0.05*rng, ymax + violin_headroom*rng)

        stat, pval = mannwhitneyu(cp_on, cp_off, alternative="two-sided")
        stars = p_to_stars(pval)
        trans = mtransforms.blended_transform_factory(ax_vio.transData, ax_vio.transAxes)
        ax_vio.plot([1, 2], [bracket_axes_y, bracket_axes_y],
                    transform=trans, color="black", lw=1.2, clip_on=False)
        ax_vio.text(1.5, min(0.99, bracket_axes_y + 0.02), stars,
                    transform=trans, ha="center", va="bottom",
                    fontsize=11, clip_on=False)
        if show_p_value:
            ax_vio.text(1.5, min(0.99, bracket_axes_y + 0.10),
                        f"p={_format_p(pval)}",
                        transform=trans, ha="center", va="bottom",
                        fontsize=9, clip_on=False)

    # --- Additional: ECDF (central pixel) + MWU annotation ---
    x_on,  y_on  = _ecdf(cp_on)
    x_off, y_off = _ecdf(cp_off)
    ec_on,  = ax_ecdf.plot(x_on,  y_on,  lw=2, label="Active",  color=line_on.get_color())
    ec_off, = ax_ecdf.plot(x_off, y_off, lw=2, label="Inactive", color=line_off.get_color())
    ax_ecdf.set_xlabel("Central pixel (max=100)")
    ax_ecdf.set_ylabel("Cumulative prob.")
    ax_ecdf.set_ylim(0, 1.0)
    ax_ecdf.grid(True, alpha=0.3)
    ax_ecdf.set_title(f"{title_for_ch} center ECDF")

    if len(cp_on) and len(cp_off):
        stat, pval = mannwhitneyu(cp_on, cp_off, alternative="two-sided")
        stars = p_to_stars(pval)
        ax_ecdf.text(0.98, 0.98, f"{stars}\nMWU p={_format_p(pval)}",
                     ha="right", va="top", transform=ax_ecdf.transAxes,
                     fontsize=10,
                     bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.6, edgecolor="none"))

# ---- 3x6 composite figure -----------------------------------------------
def show_all_channels_with_violin(
    on_imgs_dict, off_imgs_dict, rand_stack_4d, rand_avg_3d=None,
    snap_title="SNAPtag", figsize=(22, 15),
    normalize_mode="pixel_max", agg_method="mean",
    image_aspect="auto", show_p_value=True,
    bracket_axes_y=0.96, violin_headroom=0.15,
    save_png=None
):
    """
    3 rows x 6 columns: [Active | Inactive | Radial | Legend | Violin | ECDF]
    """
    rand_avg_3d = _ensure_rand_avg_3d(rand_stack_4d, rand_avg_3d)

    fig, axes = plt.subplots(
        nrows=3, ncols=6, figsize=figsize,
        gridspec_kw={'width_ratios': [1.0, 1.0, 1.2, 0.18, 1.2, 1.2]},
        constrained_layout=False
    )
    fig.subplots_adjust(hspace=0.90, wspace=0.60, right=0.985)

    plot_channel_row(axes[0], 0, on_imgs_dict, off_imgs_dict, rand_stack_4d, rand_avg_3d,
                     title_for_ch=snap_title, normalize_mode=normalize_mode, agg_method=agg_method,
                     image_aspect=image_aspect, show_p_value=show_p_value,
                     bracket_axes_y=bracket_axes_y, violin_headroom=violin_headroom)
    plot_channel_row(axes[1], 1, on_imgs_dict, off_imgs_dict, rand_stack_4d, rand_avg_3d,
                     title_for_ch="MCP", normalize_mode=normalize_mode, agg_method=agg_method,
                     image_aspect=image_aspect, show_p_value=show_p_value,
                     bracket_axes_y=bracket_axes_y, violin_headroom=violin_headroom)
    plot_channel_row(axes[2], 2, on_imgs_dict, off_imgs_dict, rand_stack_4d, rand_avg_3d,
                     title_for_ch="mTetR", normalize_mode=normalize_mode, agg_method=agg_method,
                     image_aspect=image_aspect, show_p_value=show_p_value,
                     bracket_axes_y=bracket_axes_y, violin_headroom=violin_headroom)

    if save_png:
        fig.savefig(save_png, dpi=300, bbox_inches="tight")
        print(f"Saved → {save_png}")
    return fig

## Composite figure: Active/Inactive, radial profiles, and violin

Generate the main multi‑panel summary figure with Active/Inactive images, radial profiles, and violin plots.

- **Inputs:** ON/OFF stacks and random controls.
- **Outputs:** in‑memory figure object (save in later cells).

In [ ]:
def _add_pair_colorbar(fig, ax_left, ax_right, im,
                       label="Intensity (baseline-subtracted, max=100)",
                       pad_frac=0.012, height_frac=0.018):
    fig.canvas.draw()  # use the final layout to read BBoxes
    bL = ax_left.get_position()
    bR = ax_right.get_position()
    x0 = bL.x0
    x1 = bR.x1
    y0 = min(bL.y0, bR.y0)

    h  = height_frac
    y  = max(0.0, y0 - pad_frac - h)           # keep inside the figure
    cax = fig.add_axes([x0, y, x1 - x0, h])    # add in figure coordinates

    cb  = fig.colorbar(im, cax=cax, orientation="horizontal")
    cb.set_label(label)
    return cax, cb


def _mask_disc(shape, center, R):
    yy, xx = np.indices(shape)
    r = np.sqrt((xx-center[1])**2 + (yy-center[0])**2)
    return (r <= R)

def _mask_annulus(shape, center, r_in, r_out):
    yy, xx = np.indices(shape)
    r = np.sqrt((xx-center[1])**2 + (yy-center[0])**2)
    return (r >= r_in) & (r <= r_out)

def metric_values(img_list, rand_img, scale,
                  metric="center", R_core_px=1, annulus_px=(4,8), center=CENTER):
    """Return center-based metric values for each image as an array."""
    vals = []
    shp = rand_img.shape
    if metric == "center":
        cy, cx = center
        for im in img_list:
            vals.append(((im - rand_img)[cy, cx]) * scale)
    elif metric == "core_mean":
        m_core = _mask_disc(shp, center, R_core_px)
        for im in img_list:
            a = (im - rand_img) * scale
            vals.append(np.nanmean(a[m_core]))
    elif metric == "core_minus_annulus":
        m_core = _mask_disc(shp, center, R_core_px)
        m_ann  = _mask_annulus(shp, center, annulus_px[0], annulus_px[1])
        for im in img_list:
            a = (im - rand_img) * scale
            vals.append(np.nanmean(a[m_core]) - np.nanmean(a[m_ann]))
    else:
        raise ValueError("metric must be 'center', 'core_mean', or 'core_minus_annulus'")
    return np.asarray(vals, float)

# ==== Added: BH-FDR (p-value vector -> significance flags) ====
def fdr_bh(pvals, alpha=0.05):
    p = np.asarray(pvals, float)
    m = p.size
    order = np.argsort(p)
    ranked = p[order]
    thresh = alpha * (np.arange(1, m + 1) / m)
    passed = ranked <= thresh
    sig = np.zeros(m, dtype=bool)
    if np.any(passed):
        k = np.max(np.where(passed))
        sig[order[:k + 1]] = True
    return sig

def plot_channel_row(
    ax_row, ch, on_imgs_dict, off_imgs_dict,
    rand_stack_4d, rand_avg_3d, title_for_ch,
    normalize_mode="pixel_max", agg_method="mean",
    image_aspect="auto",
    show_p_value=True, bracket_axes_y=0.96, violin_headroom=0.15,
    metric_kind="center", R_core_px=1, annulus_px=(4,8),
    show_radial_diff=True, mark_radial_sig=True, fdr_alpha=0.05
):
    # Layout: [Active image | Inactive image | Radial | Legend | Violin]
    ax_on, ax_off, ax_rad, ax_leg, ax_vio = ax_row

    # --- Input ---
    on_list  = on_imgs_dict.get(ch, [])
    off_list = off_imgs_dict.get(ch, [])
    assert len(on_list)>0 and len(off_list)>0, f"channel {ch}: Active/Inactive lists are empty"

    rand_avg_img = rand_avg_3d[ch]
    rand_list_ch = [rand_stack_4d[i, :, :, ch] for i in range(rand_stack_4d.shape[0])]

    # --- Scaling factor ---
    s = compute_scale(on_list, off_list, rand_avg_img, mode=normalize_mode, agg_method=agg_method)

    # --- Display images (aggregate - random mean) ---
    on_disp  = (_aggregate_image(on_list,  method=agg_method) - rand_avg_img) * s
    off_disp = (_aggregate_image(off_list, method=agg_method) - rand_avg_img) * s

    # --- 2D images (diverging colormap centered at zero) ---
    vabs = max(np.nanmax(np.abs(on_disp)), np.nanmax(np.abs(off_disp))) or 1.0
    norm = TwoSlopeNorm(vmin=-vabs, vcenter=0.0, vmax=vabs)
    im_on  = ax_on.imshow(on_disp,  cmap="coolwarm", norm=norm, origin="lower")
    im_off = ax_off.imshow(off_disp, cmap="coolwarm", norm=norm, origin="lower")

    # Left-aligned titles; show N at the right edge only for SNAPtag row (ch==0)
    title_active   = f"{title_for_ch} Active"
    title_inactive = f"{title_for_ch} Inactive"

    ax_on.set_title(title_active,   loc="left");  ax_on.set_xticks([]);  ax_on.set_yticks([]);  ax_on.set_aspect(image_aspect)
    ax_off.set_title(title_inactive, loc="left"); ax_off.set_xticks([]); ax_off.set_yticks([]); ax_off.set_aspect(image_aspect)

    if ch == 0:
        n_act  = len(on_list)
        n_inac = len(off_list)
        # Right-aligned N on the same title row (fine-tune with y=1.00–1.03 if needed)
        ax_on.text(0.99, 1.02,  f"(N = {n_act:,})",  transform=ax_on.transAxes,
                   ha="right", va="bottom", fontsize=10)
        ax_off.text(0.99, 1.02, f"(N = {n_inac:,})", transform=ax_off.transAxes,
                   ha="right", va="bottom", fontsize=10)

    # --- Colorbar below Active/Inactive (as before) ---
    _, _ = _add_pair_colorbar(ax_on.figure, ax_on, ax_off, im_on,
                            label="Intensity (baseline-subtracted, max=100)")

    # --- Radial (±SEM) ---
    on_profs   = radial_profiles_from_stack(on_list,  rand_img=rand_avg_img, scale=s)
    off_profs  = radial_profiles_from_stack(off_list, rand_img=rand_avg_img, scale=s)
    rand_profs = radial_profiles_from_stack(rand_list_ch, rand_img=rand_avg_img, scale=s)

    r_um, _    = radial_profile(on_disp)
    mean_on,  sem_on  = np.nanmean(on_profs,  axis=0), sem(on_profs,  axis=0)
    mean_off, sem_off = np.nanmean(off_profs, axis=0), sem(off_profs, axis=0)
    mean_rnd, sem_rnd = np.nanmean(rand_profs, axis=0), sem(rand_profs, axis=0)

    line_on,  = ax_rad.plot(r_um, mean_on,  lw=2, label="Active",   color=COLOR_ACTIVE)
    line_off, = ax_rad.plot(r_um, mean_off, lw=2, label="Inactive", color=COLOR_INACTIVE)
    line_rnd, = ax_rad.plot(r_um, mean_rnd, lw=2, label="Random",   color=COLOR_RANDOM)
    ax_rad.fill_between(r_um, mean_on -sem_on,  mean_on +sem_on,  alpha=0.25, color=COLOR_ACTIVE)
    ax_rad.fill_between(r_um, mean_off-sem_off, mean_off+sem_off, alpha=0.25, color=COLOR_INACTIVE)
    ax_rad.fill_between(r_um, mean_rnd-sem_rnd, mean_rnd+sem_rnd, alpha=0.20, color=COLOR_RANDOM)

    # Difference curve (Active - Inactive) with uncertainty band
    if show_radial_diff:
        diff = mean_on - mean_off
        sem_diff = np.sqrt(sem_on**2 + sem_off**2)
        line_diff, = ax_rad.plot(r_um, diff, lw=1.8, ls="--", color=COLOR_DIFF,
                                 label="Active - Inactive")
        ax_rad.fill_between(r_um, diff - sem_diff, diff + sem_diff, color=COLOR_DIFF, alpha=0.15)
        
        if mark_radial_sig and on_profs.size and off_profs.size:
            pvals = [mannwhitneyu(on_profs[:,i], off_profs[:,i], alternative="two-sided").pvalue
            for i in range(on_profs.shape[1])] 
            sig = fdr_bh(pvals, alpha=fdr_alpha)
            y0, y1 = ax_rad.get_ylim()
            for i, ok in enumerate(sig): 
                if ok: 
                    ax_rad.vlines(r_um[i], y0, y0 + 0.05*(y1-y0), color="r", lw=2, alpha=0.6)
                     

    ax_rad.set_xlabel("Radius (um)")
    ax_rad.set_ylabel("Intensity (max=100)")
    ax_rad.set_xticks(np.arange(0, (R_MAX+1)*PIXEL_SIZE_UM, 2*PIXEL_SIZE_UM))
    ax_rad.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax_rad.grid(True, alpha=0.3)
    ax_rad.set_title(f"{title_for_ch} radial intensity")

    # Legend goes to the dedicated column (tight spacing)
    ax_leg.axis("off")
    handles = [line_on, line_off, line_rnd]
    if show_radial_diff:
        handles.append(line_diff)
    ax_leg.legend(handles=handles, labels=[h.get_label() for h in handles],
                  loc="center", frameon=False)

    # --- Violin: center-based metric ---
    cp_on  = metric_values(on_list,  rand_avg_img, s, metric=metric_kind, R_core_px=R_core_px, annulus_px=annulus_px)
    cp_off = metric_values(off_list, rand_avg_img, s, metric=metric_kind, R_core_px=R_core_px, annulus_px=annulus_px)

    label_map = {
        "center": "center pixel",
        "core_mean": f"core ≤{R_core_px}px mean",
        "core_minus_annulus": f"core ≤{R_core_px}px − annulus {annulus_px[0]}–{annulus_px[1]}px"
    }
    ylab = f"{label_map[metric_kind]} (max=100)"

    vp = ax_vio.violinplot([cp_on, cp_off], positions=[1, 2], showmeans=True, showmedians=True, widths=0.8)
    for b, col in zip(vp['bodies'], (COLOR_ACTIVE, COLOR_INACTIVE)):
        b.set_facecolor(col); b.set_edgecolor("black"); b.set_alpha(0.40)
    for part in ('cbars','cmins','cmaxes','cmeans','cmedians'):
        if part in vp: vp[part].set_edgecolor('black'); vp[part].set_linewidth(1.0)
    ax_vio.set_xlim(0.5, 2.5)
    ax_vio.set_xticks([1, 2]); ax_vio.set_xticklabels(["Active", "Inactive"])
    ax_vio.set_ylabel(ylab)
    ax_vio.grid(axis='y', alpha=0.3)
    ax_vio.set_title(f"{title_for_ch} {label_map[metric_kind]}")

    if cp_on.size and cp_off.size:
        ymin = float(np.nanmin([cp_on.min(), cp_off.min()]))
        ymax = float(np.nanmax([cp_on.max(), cp_off.max()]))
        rng  = max(1.0, ymax - ymin)
        ax_vio.set_ylim(ymin - 0.05*rng, ymax + violin_headroom*rng)

        # Mann–Whitney (stars + p-value)
        stat, pval = mannwhitneyu(cp_on, cp_off, alternative="two-sided")
        stars = p_to_stars(pval)
        trans = mtransforms.blended_transform_factory(ax_vio.transData, ax_vio.transAxes)
        ax_vio.plot([1, 2], [bracket_axes_y, bracket_axes_y], transform=trans, color="black", lw=1.2, clip_on=False)
        ax_vio.text(1.5, min(0.99, bracket_axes_y + 0.02), stars, transform=trans,
                    ha="center", va="bottom", fontsize=11, clip_on=False)
        if show_p_value:
            ax_vio.text(1.5, min(0.99, bracket_axes_y + 0.10), f"p={_format_p(pval)}",
                        transform=trans, ha="center", va="bottom", fontsize=9, clip_on=False)


def show_all_channels_with_violin(
    on_imgs_dict, off_imgs_dict, rand_stack_4d, rand_avg_3d=None,
    snap_title="SNAPtag", figsize=(22, 15),
    normalize_mode="pixel_max", agg_method="mean",
    image_aspect="auto", show_p_value=True,
    bracket_axes_y=0.96, violin_headroom=0.15,
    metric_kind="core_minus_annulus", R_core_px=1, annulus_px=(4,8),
    show_radial_diff=True, mark_radial_sig=False, fdr_alpha=0.05,
    save_png=None, save_pdf=None       # add PDF output
):
    """
    3 rows x 5 columns: [Active | Inactive | Radial | Legend | Violin]
    """
    rand_avg_3d = _ensure_rand_avg_3d(rand_stack_4d, rand_avg_3d)

    fig, axes = plt.subplots(
        nrows=3, ncols=5, figsize=figsize,
        # Slim legend column and tighter spacing to Radial
        gridspec_kw={'width_ratios': [1.0, 1.0, 1.0, 0.0, 1.0]},
        constrained_layout=False
    )
    fig.subplots_adjust(hspace=0.90, wspace=0.50, right=0.985)

    # SNAPtag (ch0)
    plot_channel_row(
        axes[0], 0, on_imgs_dict, off_imgs_dict, rand_stack_4d, rand_avg_3d,
        title_for_ch=snap_title, normalize_mode=normalize_mode, agg_method=agg_method,
        image_aspect=image_aspect, show_p_value=show_p_value,
        bracket_axes_y=bracket_axes_y, violin_headroom=violin_headroom,
        metric_kind=metric_kind, R_core_px=R_core_px, annulus_px=annulus_px,
        show_radial_diff=show_radial_diff, mark_radial_sig=mark_radial_sig, fdr_alpha=fdr_alpha
    )
    # MCP (ch1)
    plot_channel_row(
        axes[1], 1, on_imgs_dict, off_imgs_dict, rand_stack_4d, rand_avg_3d,
        title_for_ch="MCP", normalize_mode=normalize_mode, agg_method=agg_method,
        image_aspect=image_aspect, show_p_value=show_p_value,
        bracket_axes_y=bracket_axes_y, violin_headroom=violin_headroom,
        metric_kind=metric_kind, R_core_px=R_core_px, annulus_px=annulus_px,
        show_radial_diff=show_radial_diff, mark_radial_sig=mark_radial_sig, fdr_alpha=fdr_alpha
    )
    # mTetR (ch2)
    plot_channel_row(
        axes[2], 2, on_imgs_dict, off_imgs_dict, rand_stack_4d, rand_avg_3d,
        title_for_ch="mTetR", normalize_mode=normalize_mode, agg_method=agg_method,
        image_aspect=image_aspect, show_p_value=show_p_value,
        bracket_axes_y=bracket_axes_y, violin_headroom=violin_headroom,
        metric_kind=metric_kind, R_core_px=R_core_px, annulus_px=annulus_px,
        show_radial_diff=show_radial_diff, mark_radial_sig=mark_radial_sig, fdr_alpha=fdr_alpha
    )

    # Save
    if save_png:
        fig.savefig(save_png, dpi=300, bbox_inches="tight")
        print(f"Saved → {save_png}")
    if save_pdf:
        fig.savefig(save_pdf, format="pdf", bbox_inches="tight")
        print(f"Saved → {save_pdf}")
    return fig

## Composite figure (variant): optional violin and radial significance

Alternative plotting utilities that support optional violins and radial significance markers.

- **Purpose:** generate panel variants for different figure layouts.
- **Output:** helper functions used by batch export cells.

In [ ]:
# ---- Palette (Active=orange, Inactive=blue, Random=green, Diff=gray) ----
COLOR_ACTIVE   = "tab:orange"
COLOR_INACTIVE = "tab:blue"
COLOR_RANDOM   = "tab:green"
COLOR_DIFF     = "0.35"      # dark gray

# --- Stable in PDF: place a paired colorbar directly under the left/right images ---
def _add_pair_colorbar(fig, ax_left, ax_right, im,
                       label="Intensity (baseline-subtracted, max=100)",
                       pad_frac=0.012, height_frac=0.018):
    fig.canvas.draw()  # use BBoxes after layout finalization
    bL = ax_left.get_position()
    bR = ax_right.get_position()
    x0 = bL.x0
    x1 = bR.x1
    y0 = min(bL.y0, bR.y0)
    h  = height_frac
    y  = max(0.0, y0 - pad_frac - h)           # keep inside the figure
    cax = fig.add_axes([x0, y, x1 - x0, h])
    cb  = fig.colorbar(im, cax=cax, orientation="horizontal")
    cb.set_label(label)
    return cax, cb

# ==== BH-FDR (p-value vector -> significance flags) ====
def fdr_bh(pvals, alpha=0.05):
    p = np.asarray(pvals, float)
    m = p.size
    order = np.argsort(p)
    ranked = p[order]
    thresh = alpha * (np.arange(1, m + 1) / m)
    passed = ranked <= thresh
    sig = np.zeros(m, dtype=bool)
    if np.any(passed):
        k = np.max(np.where(passed))
        sig[order[:k + 1]] = True
    return sig

# ---- Plot one row (= up to 5 columns) ----------------------------------------
def plot_channel_row(
    ax_row, ch, on_imgs_dict, off_imgs_dict,
    rand_stack_4d, rand_avg_3d, title_for_ch,
    normalize_mode="pixel_max", agg_method="mean",
    image_aspect="auto",
    show_p_value=True, bracket_axes_y=0.96, violin_headroom=0.15,
    metric_kind="center", R_core_px=1, annulus_px=(4,8),
    show_radial_diff=True, mark_radial_sig=True, fdr_alpha=0.05,
    show_violin=False,                   # Added: toggle the rightmost violin (default OFF)
    sig_star_y_frac=0.05, sig_star_fs=12 # Added: y-position/size for radial significance stars
):
    # Layout: [Active image | Inactive image | Radial | Legend | (Violin)]
    # Keep 5 axes for minimal changes; hide the violin axis if unused.
    if len(ax_row) == 5:
        ax_on, ax_off, ax_rad, ax_leg, ax_vio = ax_row
    else:  # allow 4-column layout as a fallback
        ax_on, ax_off, ax_rad, ax_leg = ax_row
        ax_vio = None

    # --- Input ---
    on_list  = on_imgs_dict.get(ch, [])
    off_list = off_imgs_dict.get(ch, [])
    assert len(on_list)>0 and len(off_list)>0, f"channel {ch}: Active/Inactive lists are empty"

    rand_avg_img = rand_avg_3d[ch]
    rand_list_ch = [rand_stack_4d[i, :, :, ch] for i in range(rand_stack_4d.shape[0])]

    # --- Scaling factor ---
    s = compute_scale(on_list, off_list, rand_avg_img, mode=normalize_mode, agg_method=agg_method)

    # --- Display images (aggregate - random mean) ---
    on_disp  = (_aggregate_image(on_list,  method=agg_method) - rand_avg_img) * s
    off_disp = (_aggregate_image(off_list, method=agg_method) - rand_avg_img) * s

    # --- 2D images (diverging colormap) ---
    vabs = max(np.nanmax(np.abs(on_disp)), np.nanmax(np.abs(off_disp))) or 1.0
    norm = TwoSlopeNorm(vmin=-vabs, vcenter=0.0, vmax=vabs)
    im_on  = ax_on.imshow(on_disp,  cmap="coolwarm", norm=norm, origin="lower")
    im_off = ax_off.imshow(off_disp, cmap="coolwarm", norm=norm, origin="lower")

    # Left-aligned titles; show N at right for SNAPtag row (ch==0)
    title_active   = f"{title_for_ch} Active"
    title_inactive = f"{title_for_ch} Inactive"
    ax_on.set_title(title_active,   loc="left");  ax_on.set_xticks([]);  ax_on.set_yticks([]);  ax_on.set_aspect(image_aspect)
    ax_off.set_title(title_inactive, loc="left"); ax_off.set_xticks([]); ax_off.set_yticks([]); ax_off.set_aspect(image_aspect)
    if ch == 0:
        n_act, n_inac = len(on_list), len(off_list)
        ax_on.text(0.99, 1.02,  f"(N = {n_act:,})",  transform=ax_on.transAxes,  ha="right", va="bottom", fontsize=10)
        ax_off.text(0.99, 1.02, f"(N = {n_inac:,})", transform=ax_off.transAxes, ha="right", va="bottom", fontsize=10)

    # --- Colorbar (under Active/Inactive) ---
    _add_pair_colorbar(ax_on.figure, ax_on, ax_off, im_on,
                       label="Intensity (baseline-subtracted, max=100)")

    # --- Radial (±SEM) ---
    on_profs   = radial_profiles_from_stack(on_list,  rand_img=rand_avg_img, scale=s)
    off_profs  = radial_profiles_from_stack(off_list, rand_img=rand_avg_img, scale=s)
    rand_profs = radial_profiles_from_stack(rand_list_ch, rand_img=rand_avg_img, scale=s)

    r_um, _    = radial_profile(on_disp)
    mean_on,  sem_on  = np.nanmean(on_profs,  axis=0), sem(on_profs,  axis=0)
    mean_off, sem_off = np.nanmean(off_profs, axis=0), sem(off_profs, axis=0)
    mean_rnd, sem_rnd = np.nanmean(rand_profs, axis=0), sem(rand_profs, axis=0)

    line_on,  = ax_rad.plot(r_um, mean_on,  lw=2, label="Active",   color=COLOR_ACTIVE)
    line_off, = ax_rad.plot(r_um, mean_off, lw=2, label="Inactive", color=COLOR_INACTIVE)
    line_rnd, = ax_rad.plot(r_um, mean_rnd, lw=2, label="Random",   color=COLOR_RANDOM)
    ax_rad.fill_between(r_um, mean_on -sem_on,  mean_on +sem_on,  alpha=0.25, color=COLOR_ACTIVE)
    ax_rad.fill_between(r_um, mean_off-sem_off, mean_off+sem_off, alpha=0.25, color=COLOR_INACTIVE)
    ax_rad.fill_between(r_um, mean_rnd-sem_rnd, mean_rnd+sem_rnd, alpha=0.20, color=COLOR_RANDOM)

    # Difference curve (Active - Inactive) with uncertainty band
    line_diff = None
    if show_radial_diff:
        diff = mean_on - mean_off
        sem_diff = np.sqrt(sem_on**2 + sem_off**2)
        line_diff, = ax_rad.plot(r_um, diff, lw=1.8, ls="--", color=COLOR_DIFF,
                                 label="Active - Inactive")
        ax_rad.fill_between(r_um, diff - sem_diff, diff + sem_diff, color=COLOR_DIFF, alpha=0.15)

    # Significance markers (*) for radial positions
    if mark_radial_sig and on_profs.size and off_profs.size:
        pvals = [mannwhitneyu(on_profs[:, i], off_profs[:, i], alternative="two-sided").pvalue
                 for i in range(on_profs.shape[1])]
        sig = fdr_bh(pvals, alpha=fdr_alpha)
        y0, y1 = ax_rad.get_ylim()
        y_star = y0 + sig_star_y_frac * (y1 - y0)
        for i, ok in enumerate(sig):
            if ok:
                ax_rad.text(r_um[i], y_star, "*", color="r",
                            ha="center", va="bottom", fontsize=sig_star_fs, clip_on=False)

    ax_rad.set_xlabel("Radius (um)")
    ax_rad.set_ylabel("Intensity (max=100)")
    ax_rad.set_xticks(np.arange(0, (R_MAX+1)*PIXEL_SIZE_UM, 2*PIXEL_SIZE_UM))
    ax_rad.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax_rad.grid(True, alpha=0.3)
    ax_rad.set_title(f"{title_for_ch} radial intensity")

    # Legend (tight spacing)
    ax_leg.axis("off")
    handles = [line_on, line_off, line_rnd]
    if line_diff is not None:
        handles.append(line_diff)
    ax_leg.legend(handles=handles, labels=[h.get_label() for h in handles],
                  loc="center", frameon=False)

    # --- Rightmost violin (default hidden) ---
    if show_violin and (ax_vio is not None):
        cp_on  = metric_values(on_list,  rand_avg_img, s, metric=metric_kind, R_core_px=R_core_px, annulus_px=annulus_px)
        cp_off = metric_values(off_list, rand_avg_img, s, metric=metric_kind, R_core_px=R_core_px, annulus_px=annulus_px)

        label_map = {
            "center": "center pixel",
            "core_mean": f"core ≤{R_core_px}px mean",
            "core_minus_annulus": f"core ≤{R_core_px}px − annulus {annulus_px[0]}–{annulus_px[1]}px"
        }
        ylab = f"{label_map[metric_kind]} (max=100)"

        vp = ax_vio.violinplot([cp_on, cp_off], positions=[1, 2], showmeans=True, showmedians=True, widths=0.8)
        for b, col in zip(vp['bodies'], (COLOR_ACTIVE, COLOR_INACTIVE)):
            b.set_facecolor(col); b.set_edgecolor("black"); b.set_alpha(0.40)
        for part in ('cbars','cmins','cmaxes','cmeans','cmedians'):
            if part in vp: vp[part].set_edgecolor('black'); vp[part].set_linewidth(1.0)
        ax_vio.set_xlim(0.5, 2.5)
        ax_vio.set_xticks([1, 2]); ax_vio.set_xticklabels(["Active", "Inactive"])
        ax_vio.set_ylabel(ylab)
        ax_vio.grid(axis='y', alpha=0.3)
        ax_vio.set_title(f"{title_for_ch} {label_map[metric_kind]}")

        if cp_on.size and cp_off.size:
            ymin = float(np.nanmin([cp_on.min(), cp_off.min()]))
            ymax = float(np.nanmax([cp_on.max(), cp_off.max()]))
            rng  = max(1.0, ymax - ymin)
            ax_vio.set_ylim(ymin - 0.05*rng, ymax + violin_headroom*rng)

            stat, pval = mannwhitneyu(cp_on, cp_off, alternative="two-sided")
            stars = p_to_stars(pval)
            trans = mtransforms.blended_transform_factory(ax_vio.transData, ax_vio.transAxes)
            ax_vio.plot([1, 2], [bracket_axes_y, bracket_axes_y], transform=trans, color="black", lw=1.2, clip_on=False)
            if show_p_value:
                ax_vio.text(1.5, min(0.99, bracket_axes_y + 0.10), f"p={_format_p(pval)}",
                            transform=trans, ha="center", va="bottom", fontsize=9, clip_on=False)
            ax_vio.text(1.5, min(0.99, bracket_axes_y + 0.02), stars,
                        transform=trans, ha="center", va="bottom", fontsize=11, clip_on=False)
    else:
        if ax_vio is not None:
            ax_vio.set_visible(False)


# ---- 3x(4–5) composite figure -----------------------------------------------
def show_all_channels_with_violin(
    on_imgs_dict, off_imgs_dict, rand_stack_4d, rand_avg_3d=None,
    snap_title="SNAPtag", figsize=(22, 15),
    normalize_mode="pixel_max", agg_method="mean",
    image_aspect="auto", show_p_value=True,
    bracket_axes_y=0.96, violin_headroom=0.15,
    metric_kind="core_minus_annulus", R_core_px=1, annulus_px=(4,8),
    show_radial_diff=True, mark_radial_sig=False, fdr_alpha=0.05,
    show_violin=False,                # Added: toggle overall violin display (default OFF)
    save_png=None, save_pdf=None
):
    """
    3 rows x (4 or 5 columns): [Active | Inactive | Radial | Legend | (Violin)]
    """
    rand_avg_3d = _ensure_rand_avg_3d(rand_stack_4d, rand_avg_3d)

    # If violin is hidden, set the rightmost column width to zero
    width_ratios = [1.0, 1.0, 1.0, 0.0, (1.0 if show_violin else 0.0)]
    fig, axes = plt.subplots(
        nrows=3, ncols=5, figsize=figsize,
        gridspec_kw={'width_ratios': width_ratios},
        constrained_layout=False
    )
    fig.subplots_adjust(hspace=0.90, wspace=0.50, right=0.985)

    # SNAPtag (ch0)
    plot_channel_row(
        axes[0], 0, on_imgs_dict, off_imgs_dict, rand_stack_4d, rand_avg_3d,
        title_for_ch=snap_title, normalize_mode=normalize_mode, agg_method=agg_method,
        image_aspect=image_aspect, show_p_value=show_p_value,
        bracket_axes_y=bracket_axes_y, violin_headroom=violin_headroom,
        metric_kind=metric_kind, R_core_px=R_core_px, annulus_px=annulus_px,
        show_radial_diff=show_radial_diff, mark_radial_sig=mark_radial_sig, fdr_alpha=fdr_alpha,
        show_violin=show_violin
    )
    # MCP (ch1)
    plot_channel_row(
        axes[1], 1, on_imgs_dict, off_imgs_dict, rand_stack_4d, rand_avg_3d,
        title_for_ch="MCP", normalize_mode=normalize_mode, agg_method=agg_method,
        image_aspect=image_aspect, show_p_value=show_p_value,
        bracket_axes_y=bracket_axes_y, violin_headroom=violin_headroom,
        metric_kind=metric_kind, R_core_px=R_core_px, annulus_px=annulus_px,
        show_radial_diff=show_radial_diff, mark_radial_sig=mark_radial_sig, fdr_alpha=fdr_alpha,
        show_violin=show_violin
    )
    # mTetR (ch2)
    plot_channel_row(
        axes[2], 2, on_imgs_dict, off_imgs_dict, rand_stack_4d, rand_avg_3d,
        title_for_ch="mTetR", normalize_mode=normalize_mode, agg_method=agg_method,
        image_aspect=image_aspect, show_p_value=show_p_value,
        bracket_axes_y=bracket_axes_y, violin_headroom=violin_headroom,
        metric_kind=metric_kind, R_core_px=R_core_px, annulus_px=annulus_px,
        show_radial_diff=show_radial_diff, mark_radial_sig=mark_radial_sig, fdr_alpha=fdr_alpha,
        show_violin=show_violin
    )

    # Save
    if save_png:
        fig.savefig(save_png, dpi=300, bbox_inches="tight")
        print(f"Saved → {save_png}")
    if save_pdf:
        fig.savefig(save_pdf, format="pdf", bbox_inches="tight")
        print(f"Saved → {save_pdf}")
    return fig

## Batch export: full composite panels

Export composite panels for each condition in bulk.

- **Output directory:** `outputs/Figs_thr1.30_1.15/All_mean/`.
- **Filename pattern:** `{prefix}.pdf`.

In [ ]:
# === Batch export: save {prefix}.pdf to Accumulation/Figs_thr1.30_1.15/All ===
import os
import matplotlib.pyplot as plt

# 1) Prepare output directory
out_dir = os.path.join("..", "outputs","Figs_thr1.30_1.15", "All_mean")
os.makedirs(out_dir, exist_ok=True)

# 2) Determine prefixes to process
#    Prefer created_vars from the loader (rows with NaN pkl_file were never created)
prefixes = []
if 'created_vars' in globals() and created_vars:
    prefixes = sorted(set(v[:-len("_on_images")] for v in created_vars
                          if v.endswith("_on_images")))

#    Fallback: reconstruct from df_first_sheet if needed (use only existing entries)
if not prefixes and 'df_first_sheet' in globals():
    # Use helper functions defined in the loader if available
    _sanitize = globals().get('_sanitize_ident', lambda s: str(s))
    _extract  = globals().get('_extract_int',   lambda s, default=None: default or str(s))
    for _, row in df_first_sheet.dropna(subset=['pkl_file']).iterrows():
        target = _sanitize(row.get('STtag_KI_target', ''))
        factor = _sanitize(row.get('Factor_name', ''))
        rep    = _extract(row.get('replicate', ''), default=str(row.get('replicate', '')))
        prefixes.append(f"{target}_{factor}_rep{rep}")
    # Remove duplicates
    prefixes = sorted(set(prefixes))

# 3) Iterate and save PDFs
skipped = []
saved   = []

for prefix in prefixes:
    on_name  = f"{prefix}_on_images"
    off_name = f"{prefix}_off_images"
    rnd_name = f"{prefix}_on_images_random"

    on_imgs  = globals().get(on_name,  None)
    off_imgs = globals().get(off_name, None)
    rnd      = globals().get(rnd_name, None)

    # Skip missing inputs
    missing = [n for n, obj in [(on_name,on_imgs), (off_name,off_imgs), (rnd_name,rnd)] if obj is None]
    if missing:
        skipped.append((prefix, f"missing: {', '.join(missing)}"))
        continue

    out_pdf = os.path.join(out_dir, f"{prefix}.pdf")

    # ---- Create figure and save immediately (no display) ----
    fig = show_all_channels_with_violin(
        on_imgs_dict=on_imgs,
        off_imgs_dict=off_imgs,
        rand_stack_4d=rnd,
        snap_title="SNAPtag",          # Title is always SNAPtag
        figsize=(12, 14),
        normalize_mode="pixel_max", agg_method="mean",
        image_aspect="equal",
        show_p_value=True, bracket_axes_y=0.80, violin_headroom=0.40,
        metric_kind="core_minus_annulus", R_core_px=2, annulus_px=(6,8),
        show_radial_diff=False,         # Do not show the Active - Inactive curve
        mark_radial_sig=True,           # Do not draw radial significance ticks
        save_pdf=out_pdf                # Save as PDF
    )
    plt.close(fig)
    print(f"Saved → {out_pdf}")
    saved.append(out_pdf)

# 4) Summary (optional)
if skipped:
    print("\nSkipped:")
    for p, reason in skipped:
        print(f" - {p}: {reason}")

## SNAP‑only horizontal panels

Define a compact layout and batch‑export horizontal SNAPtag‑only panels.

- **Output directory:** `outputs/Figs_thr1.30_1.15/All_mean_SNAPonly_hori/`.
- **Filename pattern:** `{prefix}.pdf`.

In [ ]:
def plot_channel_row2(
    ax_row,
    ch,
    on_imgs_dict,
    off_imgs_dict,
    rand_stack_4d,
    rand_avg_3d,
    title_for_ch="SNAPtag",
    normalize_mode="pixel_max",
    agg_method="mean",
    image_aspect="equal",
    show_p_value=True,
    metric_kind="core_minus_annulus",
    R_core_px=2,
    annulus_px=(6,8),
    show_radial_diff=False,
    mark_radial_sig=True,
    fdr_alpha=0.05,
    font_size=12,
):
    """
    ax_row = [ax_on, ax_off, ax_rad, ax_leg]
    For SNAPtag (ch0). N is displayed above each image and no text is drawn inside the images.
    """
    ax_on, ax_off, ax_rad, ax_leg = ax_row

    # Inputs
    on_list  = on_imgs_dict.get(ch, [])
    off_list = off_imgs_dict.get(ch, [])
    assert len(on_list)>0 and len(off_list)>0, f"channel {ch}: Active/Inactive lists are empty"

    rand_avg_img = rand_avg_3d[ch]
    rand_list_ch = [rand_stack_4d[i, :, :, ch] for i in range(rand_stack_4d.shape[0])]

    # Scale
    s = compute_scale(on_list, off_list, rand_avg_img,
                      mode=normalize_mode, agg_method=agg_method)

    # Aggregate images
    on_disp  = (_aggregate_image(on_list,  method=agg_method) - rand_avg_img) * s
    off_disp = (_aggregate_image(off_list, method=agg_method) - rand_avg_img) * s

    # 2D images
    vabs = max(np.nanmax(np.abs(on_disp)), np.nanmax(np.abs(off_disp)))
    vabs = vabs if vabs > 0 else 1.0
    norm = TwoSlopeNorm(vmin=-vabs, vcenter=0.0, vmax=vabs)

    ax_on.imshow(on_disp,  cmap="coolwarm", norm=norm, origin="lower")
    ax_off.imshow(off_disp, cmap="coolwarm", norm=norm, origin="lower")

    for ax in (ax_on, ax_off):
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_aspect(image_aspect)

    # N in the title area
    n_on, n_off = len(on_list), len(off_list)
   
    
        # ---- N at lower-left (italic N, regular numbers) ----
    ax_on.text(
        0.02, 0.02,
        rf"$\it{{N}}$ = {n_on:,}",
        transform=ax_on.transAxes,
        ha="left", va="bottom",
        fontsize=font_size,
        color="white",
        path_effects=[PathEffects.withStroke(linewidth=2, foreground="black")]
    )

    ax_off.text(
        0.02, 0.02,
        rf"$\it{{N}}$ = {n_off:,}",
        transform=ax_off.transAxes,
        ha="left", va="bottom",
        fontsize=font_size,
        color="white",
        path_effects=[PathEffects.withStroke(linewidth=2, foreground="black")]
    )


    # ---- Radial intensity ----
    on_profs   = radial_profiles_from_stack(on_list,  rand_img=rand_avg_img, scale=s)
    off_profs  = radial_profiles_from_stack(off_list, rand_img=rand_avg_img, scale=s)
    rand_profs = radial_profiles_from_stack(rand_list_ch, rand_img=rand_avg_img, scale=s)

    r_um, _ = radial_profile(on_disp)

    mean_on,  sem_on  = np.nanmean(on_profs,axis=0), sem(on_profs,axis=0)
    mean_off, sem_off = np.nanmean(off_profs,axis=0), sem(off_profs,axis=0)
    mean_rnd, sem_rnd = np.nanmean(rand_profs,axis=0), sem(rand_profs,axis=0)

    line_on,  = ax_rad.plot(r_um, mean_on,  lw=2, label="Active",   color="tab:orange")
    line_off, = ax_rad.plot(r_um, mean_off, lw=2, label="Inactive", color="tab:blue")
    line_rnd, = ax_rad.plot(r_um, mean_rnd, lw=2, label="Random",   color="tab:green")

    ax_rad.fill_between(r_um, mean_on-sem_on,  mean_on+sem_on,
                        alpha=0.25, color="tab:orange")
    ax_rad.fill_between(r_um, mean_off-sem_off, mean_off+sem_off,
                        alpha=0.25, color="tab:blue")
    ax_rad.fill_between(r_um, mean_rnd-sem_rnd, mean_rnd+sem_rnd,
                        alpha=0.20, color="tab:green")

    # Difference curve is not shown in this layout (show_radial_diff=False recommended)
    if mark_radial_sig:
        from scipy.stats import mannwhitneyu
        pvals = [mannwhitneyu(on_profs[:, i], off_profs[:, i],
                              alternative="two-sided").pvalue
                 for i in range(on_profs.shape[1])]
        sig = fdr_bh(pvals, alpha=fdr_alpha)
        y0, y1 = ax_rad.get_ylim()
        y_star = y0 + 0.05*(y1 - y0)
        for i, ok in enumerate(sig):
            if ok:
                ax_rad.text(r_um[i], y_star, "*", color="red",
                            ha="center", va="bottom",
                            fontsize=font_size, clip_on=False)

    ax_rad.set_xlabel("Radius (um)", fontsize=font_size)
    ax_rad.set_ylabel("Intensity",    fontsize=font_size,
                     labelpad=-4)  # shift slightly right to reduce overlap
    ax_rad.tick_params(labelsize=font_size-1)
    ax_rad.grid(True, alpha=0.3)
    ax_rad.set_title("Radial intensity", fontsize=font_size+2)

    # Keep horizontal space by using automatic aspect
    ax_rad.set_aspect("auto")

    ax_leg.axis("off")
    ax_leg.legend(handles=[line_on, line_off, line_rnd],
                  labels=["Active", "Inactive", "Random"],
                  loc="center", frameon=False,
                  fontsize=font_size)






def show_snap_only2(
    on_imgs_dict,
    off_imgs_dict,
    rand_stack_4d,
    rand_avg_3d=None,
    figsize=(9, 3.2),
    font_size=12,
    normalize_mode="pixel_max",
    agg_method="mean",
    metric_kind="core_minus_annulus",
    R_core_px=2,
    annulus_px=(6,8),
    show_radial_diff=False,
    mark_radial_sig=True,
    save_pdf=None,
    save_png=None
):
    rand_avg_3d = _ensure_rand_avg_3d(rand_stack_4d, rand_avg_3d)

    # 1 row x 5 columns: [Active image | Inactive image | spacer | Radial | Legend]
    fig, axes = plt.subplots(
        nrows=1, ncols=5, figsize=figsize,
        gridspec_kw={'width_ratios': [1.0, 1.0, 0.15, 1.1, 0.65]},
        constrained_layout=False
    )
    # Align top/bottom between images and radial panel
    fig.subplots_adjust(left=0.04, right=0.98,
                        bottom=0.20, top=0.90,
                        wspace=0.15, hspace=0.0)

    # Hide the middle spacer column (axes[2])
    axes[2].axis("off")

    # plot_channel_row2 only needs Active, Inactive, Radial, Legend
    ax_row = [axes[0], axes[1], axes[3], axes[4]]

    plot_channel_row2(
        ax_row,
        0,
        on_imgs_dict,
        off_imgs_dict,
        rand_stack_4d,
        rand_avg_3d,
        title_for_ch="SNAPtag",
        normalize_mode=normalize_mode,
        agg_method=agg_method,
        metric_kind=metric_kind,
        R_core_px=R_core_px,
        annulus_px=annulus_px,
        show_radial_diff=show_radial_diff,
        mark_radial_sig=mark_radial_sig,
        font_size=font_size
    )

    # ===== New: enforce matched panel heights =====
    box_img = axes[0].get_position()   # Active image panel
    box_rad = axes[3].get_position()   # radial intensity panel
    axes[3].set_position([box_rad.x0, box_img.y0, box_rad.width, box_img.height])

    box_leg = axes[4].get_position()
    axes[4].set_position([box_leg.x0, box_img.y0, box_leg.width, box_img.height])
    # =============================================

    if save_pdf:
        fig.savefig(save_pdf, dpi=300, bbox_inches="tight")
        print("Saved →", save_pdf)
    if save_png:
        fig.savefig(save_png, dpi=300, bbox_inches="tight")
        print("Saved →", save_png)

    return fig

## Batch export: SNAP‑only horizontal panels

Generate SNAPtag‑only horizontal PDFs for all prefixes using the compact layout.

- **Output directory:** `outputs/Figs_thr1.30_1.15/All_mean_SNAPonly_hori/`.

In [ ]:
# === Batch export: save {prefix}.pdf to Accumulation/Figs_thr1.30_1.15/All_mean_SNAPonly_hori ===
import os
import matplotlib.pyplot as plt

# 1) Prepare output directory
out_dir = os.path.join("..", "outputs", "Figs_thr1.30_1.15", "All_mean_SNAPonly_hori")
os.makedirs(out_dir, exist_ok=True)

# 2) Determine prefixes to process
#    Prefer created_vars from the loader (rows with NaN pkl_file were never created)
prefixes = []
if 'created_vars' in globals() and created_vars:
    prefixes = sorted(
        set(v[:-len("_on_images")] for v in created_vars if v.endswith("_on_images"))
    )

#    Fallback: reconstruct from df_first_sheet if needed (use only existing entries)
if not prefixes and 'df_first_sheet' in globals():
    _sanitize = globals().get('_sanitize_ident', lambda s: str(s))
    _extract  = globals().get('_extract_int',   lambda s, default=None: default or str(s))
    for _, row in df_first_sheet.dropna(subset=['pkl_file']).iterrows():
        target = _sanitize(row.get('STtag_KI_target', ''))
        factor = _sanitize(row.get('Factor_name', ''))
        rep    = _extract(row.get('replicate', ''), default=str(row.get('replicate', '')))
        prefixes.append(f"{target}_{factor}_rep{rep}")
    prefixes = sorted(set(prefixes))

# 3) Iterate and save PDFs
skipped = []
saved   = []

for prefix in prefixes:
    on_name  = f"{prefix}_on_images"
    off_name = f"{prefix}_off_images"
    rnd_name = f"{prefix}_on_images_random"

    on_imgs  = globals().get(on_name,  None)
    off_imgs = globals().get(off_name, None)
    rnd      = globals().get(rnd_name, None)

    # Skip missing inputs
    missing = [n for n, obj in [(on_name,on_imgs), (off_name,off_imgs), (rnd_name,rnd)]
               if obj is None]
    if missing:
        skipped.append((prefix, f"missing: {', '.join(missing)}"))
        continue

    out_pdf = os.path.join(out_dir, f"{prefix}.pdf")

    # ---- Create a horizontal SNAPtag figure and save immediately (no display) ----
    fig = show_snap_only2(
        on_imgs_dict=on_imgs,
        off_imgs_dict=off_imgs,
        rand_stack_4d=rnd,
        rand_avg_3d=None,
        figsize=(9, 3.2),    # adjust if needed
        font_size=16,        # set font size here
        normalize_mode="pixel_max",
        agg_method="mean",
        metric_kind="core_minus_annulus",
        R_core_px=2,
        annulus_px=(6,8),
        show_radial_diff=False,
        mark_radial_sig=True,
        save_pdf=out_pdf
    )
    plt.close(fig)
    print(f"Saved → {out_pdf}")
    saved.append(out_pdf)

# 4) Summary (optional)
if skipped:
    print("\nSkipped:")
    for p, reason in skipped:
        print(f" - {p}: {reason}")

## Batch generation of horizontal mTetR and MCP panels

Generate separate horizontal panels for mTetR (ch2) and MCP (ch1) using the channel‑selectable layout.

- **Output directory:** `outputs/Figs_thr1.30_1.15/All_mean_SNAPonly_mTetR_MCP/`.
- **Filename pattern:** `{prefix}_mTetR.pdf`, `{prefix}_MCP.pdf`.

In [ ]:
# ===== Wrapper with channel selection =====
def show_channel_horizontal(
    on_imgs_dict,
    off_imgs_dict,
    rand_stack_4d,
    channel=0,  # 0: SNAPtag, 1: MCP, 2: mTetR
    rand_avg_3d=None,
    figsize=(9, 3.2),
    font_size=16,
    normalize_mode="pixel_max",
    agg_method="mean",
    metric_kind="core_minus_annulus",
    R_core_px=2,
    annulus_px=(6,8),
    show_radial_diff=False,
    mark_radial_sig=True,
    save_pdf=None,
    save_png=None
):
    """
    Channel-selectable wrapper for `show_snap_only2()`.
    This passes the channel index to `plot_channel_row2()` to render mTetR (ch=2) or MCP (ch=1).
    """
    rand_avg_3d = _ensure_rand_avg_3d(rand_stack_4d, rand_avg_3d)

    # 1 row x 5 columns: [Active image | Inactive image | spacer | Radial | Legend]
    fig, axes = plt.subplots(
        nrows=1, ncols=5, figsize=figsize,
        gridspec_kw={'width_ratios': [1.0, 1.0, 0.15, 1.1, 0.65]},
        constrained_layout=False
    )
    fig.subplots_adjust(left=0.04, right=0.98,
                        bottom=0.20, top=0.90,
                        wspace=0.15, hspace=0.0)

    # Hide the middle spacer column (axes[2])
    axes[2].axis("off")

    # plot_channel_row2 only needs Active, Inactive, Radial, Legend
    ax_row = [axes[0], axes[1], axes[3], axes[4]]

    plot_channel_row2(
        ax_row,
        channel,  # channel selector
        on_imgs_dict,
        off_imgs_dict,
        rand_stack_4d,
        rand_avg_3d,
        title_for_ch="Channel",  # title is not used
        normalize_mode=normalize_mode,
        agg_method=agg_method,
        metric_kind=metric_kind,
        R_core_px=R_core_px,
        annulus_px=annulus_px,
        show_radial_diff=show_radial_diff,
        mark_radial_sig=mark_radial_sig,
        font_size=font_size
    )

    # Match panel heights
    box_img = axes[0].get_position()
    box_rad = axes[3].get_position()
    axes[3].set_position([box_rad.x0, box_img.y0, box_rad.width, box_img.height])

    box_leg = axes[4].get_position()
    axes[4].set_position([box_leg.x0, box_img.y0, box_leg.width, box_img.height])

    if save_pdf:
        fig.savefig(save_pdf, dpi=300, bbox_inches="tight")
        print("Saved →", save_pdf)
    if save_png:
        fig.savefig(save_png, dpi=300, bbox_inches="tight")
        print("Saved →", save_png)

    return fig


# ===== Batch export: mTetR and MCP =====
import os
import matplotlib.pyplot as plt

# 1) Prepare output directory
out_dir = os.path.join("..", "outputs", "Figs_thr1.30_1.15", "All_mean_SNAPonly_mTetR_MCP")
os.makedirs(out_dir, exist_ok=True)

# 2) Determine prefixes to process
prefixes = []
if 'created_vars' in globals() and created_vars:
    prefixes = sorted(
        set(v[:-len("_on_images")] for v in created_vars if v.endswith("_on_images"))
    )

# Fallback if needed
if not prefixes and 'df_first_sheet' in globals():
    _sanitize = globals().get('_sanitize_ident', lambda s: str(s))
    _extract  = globals().get('_extract_int',   lambda s, default=None: default or str(s))
    for _, row in df_first_sheet.dropna(subset=['pkl_file']).iterrows():
        target = _sanitize(row.get('STtag_KI_target', ''))
        factor = _sanitize(row.get('Factor_name', ''))
        rep    = _extract(row.get('replicate', ''), default=str(row.get('replicate', '')))
        prefixes.append(f"{target}_{factor}_rep{rep}")
    prefixes = sorted(set(prefixes))

# 3) Iterate and save PDFs for mTetR and MCP
skipped = []
saved   = []

for prefix in prefixes:
    on_name  = f"{prefix}_on_images"
    off_name = f"{prefix}_off_images"
    rnd_name = f"{prefix}_on_images_random"

    on_imgs  = globals().get(on_name,  None)
    off_imgs = globals().get(off_name, None)
    rnd      = globals().get(rnd_name, None)

    # Skip missing inputs
    missing = [n for n, obj in [(on_name,on_imgs), (off_name,off_imgs), (rnd_name,rnd)]
               if obj is None]
    if missing:
        skipped.append((prefix, f"missing: {', '.join(missing)}"))
        continue

    # mTetR (ch=2)
    out_pdf_mtetr = os.path.join(out_dir, f"{prefix}_mTetR.pdf")
    fig_mtetr = show_channel_horizontal(
        on_imgs_dict=on_imgs,
        off_imgs_dict=off_imgs,
        rand_stack_4d=rnd,
        channel=2,  # mTetR
        rand_avg_3d=None,
        figsize=(9, 3.2),
        font_size=16,
        normalize_mode="pixel_max",
        agg_method="mean",
        metric_kind="core_minus_annulus",
        R_core_px=2,
        annulus_px=(6,8),
        show_radial_diff=False,
        mark_radial_sig=True,
        save_pdf=out_pdf_mtetr
    )
    plt.close(fig_mtetr)
    saved.append(out_pdf_mtetr)

    # MCP (ch=1)
    out_pdf_mcp = os.path.join(out_dir, f"{prefix}_MCP.pdf")
    fig_mcp = show_channel_horizontal(
        on_imgs_dict=on_imgs,
        off_imgs_dict=off_imgs,
        rand_stack_4d=rnd,
        channel=1,  # MCP
        rand_avg_3d=None,
        figsize=(9, 3.2),
        font_size=16,
        normalize_mode="pixel_max",
        agg_method="mean",
        metric_kind="core_minus_annulus",
        R_core_px=2,
        annulus_px=(6,8),
        show_radial_diff=False,
        mark_radial_sig=True,
        save_pdf=out_pdf_mcp
    )
    plt.close(fig_mcp)
    saved.append(out_pdf_mcp)

# 4) Summary
print(f"\n=== Summary ===")
print(f"Total prefixes processed: {len(prefixes)}")
print(f"PDFs saved: {len(saved)}")
print(f"Skipped: {len(skipped)}")

if skipped:
    print("\nSkipped prefixes:")
    for p, reason in skipped:
        print(f" - {p}: {reason}")

# Variability analysis (center intensity / r_mcp at center)

Quantify cell‑to‑cell variability by focusing on the **3x3 neighborhood at the center** of each 19x19 crop.

- **SNAPtag (ch0):** mean and median of the 3x3 center region.
- **MCP (ch1):** `r_mcp` computed at the **center coordinate (9, 9)** using the STtag_analysis_v5 definition (trackpy `refine_com` mass).

Outputs are saved under `FIGS_ROOT/variability/`.

In [ ]:
# --- Setup (re-imports, config, output dirs) ---
import os
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Trackpy is required (no fallback)
try:
    import trackpy as tp
except Exception as e:
    raise RuntimeError(
        "trackpy is required to compute r_mcp (same definition as STtag_analysis_v5).\n"
        "Please install trackpy in the active conda environment and rerun this section."
    ) from e

# --- Channel mapping (from STtag_analysis_v5) ---
CH_SNAP = 0
CH_MCP  = 1

# --- Geometry (should match 19x19) ---
CENTER_YX = (9, 9)
CENTER_RADIUS_PX = 1  # 3x3 window => radius 1
MCP_PAD = 3           # STtag_analysis_v5 uses pad=3 (refine_com radius)

# --- Output root ---
if 'FIGS_ROOT' not in globals():
    # keep consistent with earlier sections (only used if notebook wasn't run from the top)
    FIGS_ROOT = '../outputs/Figs_thr1.30_1.15'
OUT_ROOT = os.path.join(FIGS_ROOT, 'variability')
OUT_VIOLIN = os.path.join(OUT_ROOT, 'violin')
os.makedirs(OUT_VIOLIN, exist_ok=True)

def _clean_text(s):
    return str(s).replace('\u00A0', ' ').replace('\u3000', ' ').strip()

def _extract_int(s, default=None):
    m = re.search(r'\d+', _clean_text(s))
    return int(m.group()) if m else default

def _sanitize_ident(s):
    s = _clean_text(s)
    s = re.sub(r'\W+', '_', s)
    s = s.strip('_')
    if not s:
        s = 'x'
    if s[0].isdigit():
        s = 'v_' + s
    return s

def _safe_filename(s):
    s = _clean_text(s)
    s = re.sub(r'[^0-9A-Za-z._-]+', '_', s)
    return s[:200] if len(s) > 200 else s

def _center_window(img2d, center_yx=CENTER_YX, radius=CENTER_RADIUS_PX):
    y0, x0 = center_yx
    y1 = max(0, y0 - radius)
    y2 = min(img2d.shape[0], y0 + radius + 1)
    x1 = max(0, x0 - radius)
    x2 = min(img2d.shape[1], x0 + radius + 1)
    return img2d[y1:y2, x1:x2]

def snap_center_stats(img2d):
    w = _center_window(np.asarray(img2d))
    return float(np.nanmean(w)), float(np.nanmedian(w))

def r_mcp_like_at_center(img2d, center_yx=CENTER_YX, pad=MCP_PAD):
    """Compute r_mcp at a specified coordinate (default: image center) using the STtag_analysis_v5 definition.
    - Use tp.refine_com mass divided by area = (pad^2 * pi)
    - Normalize by the whole-image mean (img.mean())
    """
    img = np.asarray(img2d, dtype=float)
    img_mean = float(np.nanmean(img))
    if not np.isfinite(img_mean) or img_mean == 0.0:
        return np.nan
    # pad with zeros, shift coords
    img_p = np.pad(img, pad_width=pad, mode='constant', constant_values=0.0)
    y0, x0 = center_yx
    coords = np.array([[float(y0 + pad), float(x0 + pad)]], dtype=float)
    df1 = tp.refine_com(img_p, img_p, radius=pad, coords=coords)
    if df1 is None or len(df1) == 0:
        return np.nan
    if 'raw_mass' in df1.columns:
        mass = float(df1.loc[df1.index[0], 'raw_mass'])
    elif 'mass' in df1.columns:
        mass = float(df1.loc[df1.index[0], 'mass'])
    else:
        # unexpected trackpy output schema
        return np.nan
    area = (pad * pad) * math.pi
    signal_int = mass / area
    return signal_int / img_mean

print('OUT_ROOT:', OUT_ROOT)
print('CH mapping: SNAP=ch0, MCP=ch1; center window=3x3; MCP pad=3')

## Step 1: Extract center metrics (SNAP 3x3 center / MCP r_mcp at the center coordinate)

For each `prefix` (gene x factor x replicate), this step extracts:
- **SNAP (ch0)** center 3x3 statistics (mean/median) from Active and Inactive images
- **MCP (ch1)** `r_mcp_center`, computed using the STtag_analysis_v5 definition but evaluated at the **center coordinate (9, 9)**

The result is a per-image table `df_center` with one row per image.

In [ ]:
# --- Build per-image center metrics table ---

# Prefer already-loaded df_first_sheet from earlier cells; otherwise reload from excel_path
if 'df_first_sheet' in globals():
    _df_src = df_first_sheet.copy()
else:
    if 'excel_path' not in globals():
        excel_path = '../../../data/01_Snapshot_analysis/pkl/pkl_file_position_original.xlsx'
    _df_src = pd.read_excel(excel_path, sheet_name=0)

need_cols = {'pkl_file','STtag_KI_target','Factor_name','replicate'}
missing_cols = need_cols - set(_df_src.columns)
if missing_cols:
    raise RuntimeError(f"df_first_sheet is missing required columns: {missing_cols}")

_df_src = _df_src.dropna(subset=['pkl_file']).copy()

# Build mapping: prefix -> (gene, factor, replicate) using the same naming rule used in the loader cell
prefix_meta = {}
for _, r in _df_src.iterrows():
    gene = _sanitize_ident(r.get('STtag_KI_target', ''))
    factor = _sanitize_ident(r.get('Factor_name', ''))
    rep = _extract_int(r.get('replicate', ''), default=_clean_text(r.get('replicate', '')))
    rep_str = str(rep)
    prefix = f"{gene}_{factor}_rep{rep_str}"
    # keep first occurrence
    if prefix not in prefix_meta:
        prefix_meta[prefix] = dict(gene=gene, factor=factor, replicate=rep_str)

# Prefix list: prefer existing 'prefixes' if it exists, else from mapping
if 'prefixes' in globals() and isinstance(prefixes, (list, tuple)) and len(prefixes):
    prefixes_use = list(prefixes)
else:
    prefixes_use = sorted(prefix_meta.keys())

def _get_image_dict(prefix, kind):
    """Return per-channel dict of 2D images for a prefix and kind in {'on','off'}."""
    name = f"{prefix}_{'on_images' if kind=='on' else 'off_images'}"
    if name in globals():
        return globals()[name]
    raise RuntimeError(
        f"Image data not found: {name}.\n"
        "Run the pkl loading cells (on/off/random generation) before executing this section."
    )

records = []
skipped_prefixes = []

for prefix in prefixes_use:
    if prefix not in prefix_meta:
        # If prefix list contains extra entries, try to parse minimally
        prefix_meta[prefix] = dict(gene=prefix.split('_')[0], factor='NA', replicate='NA')
    meta = prefix_meta[prefix]
    try:
        on_dict = _get_image_dict(prefix, 'on')
        off_dict = _get_image_dict(prefix, 'off')
    except Exception as e:
        skipped_prefixes.append((prefix, str(e)))
        continue

    # Active and Inactive
    for cond, img_dict in (("Active", on_dict), ("Inactive", off_dict)):
        snap_list = img_dict.get(CH_SNAP, [])
        mcp_list  = img_dict.get(CH_MCP,  [])
        if not isinstance(snap_list, (list, tuple)) or not isinstance(mcp_list, (list, tuple)):
            continue
        n = min(len(snap_list), len(mcp_list))
        for i_img in range(n):
            snap_img = snap_list[i_img]
            mcp_img  = mcp_list[i_img]
            snap_mean, snap_median = snap_center_stats(snap_img)
            r_center = r_mcp_like_at_center(mcp_img, center_yx=CENTER_YX, pad=MCP_PAD)
            records.append(dict(
                prefix=prefix,
                gene=meta.get('gene',''),
                factor=meta.get('factor',''),
                replicate=meta.get('replicate',''),
                cond=cond,
                index=int(i_img),
                snap_center_mean=snap_mean,
                snap_center_median=snap_median,
                r_mcp_center=r_center,
            ))

df_center = pd.DataFrame.from_records(records)
if df_center.empty:
    raise RuntimeError("df_center is empty. Confirm that image data (<prefix>_on/off_images) are loaded.")

# basic cleanup
df_center['gene'] = df_center['gene'].astype(str)
df_center['factor'] = df_center['factor'].astype(str)
df_center['cond'] = df_center['cond'].astype(str)
df_center = df_center.replace([np.inf, -np.inf], np.nan)

print('df_center:', df_center.shape)
print(df_center[['gene','factor','replicate','cond','snap_center_mean','snap_center_median','r_mcp_center']].head())
if skipped_prefixes:
    print(f"[info] skipped prefixes: {len(skipped_prefixes)}")
    print('  e.g.', skipped_prefixes[0][0])

# small-sample pairs (first N unique genexfactor)
N_SAMPLE_PAIRS = 6
pairs_all = df_center[['gene','factor']].drop_duplicates().values.tolist()
sample_pairs = pairs_all[:N_SAMPLE_PAIRS]
print('sample_pairs:', sample_pairs)

## Step 2: Active/Inactive differences (violin plots)

First, generate plots for `sample_pairs` (a small set of gene x factor combinations) to confirm the appearance. 
If the results look appropriate, generate the same plots for all gene x factor combinations and save as PDFs.

SNAP metrics default to `snap_center_mean` (mean of the 3x3 center window), but can be switched to `snap_center_median` if needed.

In [ ]:
# --- Violin plots: sample first, then all ---
from matplotlib.ticker import FormatStrFormatter

import logging

# Suppress fontTools INFO messages during PDF embedding
logging.getLogger("fontTools").setLevel(logging.WARNING)
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)

SNAP_METRIC = 'mean'  # 'mean' or 'median'
snap_col = 'snap_center_mean' if SNAP_METRIC == 'mean' else 'snap_center_median'

# ---- Style knobs (paper-ready) ----
FIGSIZE = (3, 3.6)
FONT_SIZE = 10
TITLE_SIZE = 11
LABEL_SIZE = 10
TICK_SIZE = 9
LEGEND_SIZE = 9
LINEWIDTH = 1.2

COLOR_ACTIVE = 'tab:red'
COLOR_INACTIVE = 'tab:blue'

def _rep_tag(rep_val):
    import re
    s = str(rep_val).strip()
    if s == '' or s.lower() in {'nan', 'none', 'na'}:
        return 'repNA'
    m = re.search(r"\d+", s)
    if m:
        return f"rep{int(m.group(0))}"
    return f"rep{_safe_filename(s)}"

def _rep_sort_key(rep_val):
    import re
    s = str(rep_val).strip()
    m = re.search(r"\d+", s)
    if m:
        return (0, int(m.group(0)))
    if s == '' or s.lower() in {'nan', 'none', 'na'}:
        return (2, 10**9)
    return (1, s)

def _format_p_value(p_value):
    if not np.isfinite(p_value):
        return "p = NA"
    # compact, paper-friendly formatting
    return f"p = {p_value:.2g}"

def _add_significance_bar(ax, x1, x2, data1, data2, text, lw=1.2):
    data1 = np.asarray(data1, float)
    data2 = np.asarray(data2, float)
    if data1.size == 0 or data2.size == 0:
        return

    y1 = np.nanmin(np.r_[data1, data2])
    y2 = np.nanmax(np.r_[data1, data2])
    if not np.isfinite(y1) or not np.isfinite(y2):
        return

    yr = (y2 - y1) if (y2 - y1) > 0 else 1.0
    y = y2 + 0.08 * yr
    h = 0.03 * yr

    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], color='0.2', lw=lw, clip_on=False)
    ax.text((x1 + x2) / 2, y + h + 0.01 * yr, text, ha='center', va='bottom', fontsize=LEGEND_SIZE)
    ax.set_ylim(top=y + h + 0.12 * yr)

def _violin(ax, data_active, data_inactive, title, ylabel):
    data = [np.asarray(data_active, float), np.asarray(data_inactive, float)]

    # Violin: hide median line, center line, and extrema lines
    parts = ax.violinplot(
        data,
        positions=[1, 2],
        widths=0.85,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )

    body_colors = [COLOR_ACTIVE, COLOR_INACTIVE]
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(body_colors[i] if i < len(body_colors) else '0.7')
        pc.set_edgecolor('0.25')
        pc.set_linewidth(LINEWIDTH)
        pc.set_alpha(0.55)

    # Ensure line elements are hidden if present
    for k in ('cmedians', 'cmins', 'cmaxes', 'cbars', 'cmeans'):
        if k in parts:
            parts[k].set_visible(False)

    # --- Small boxplot (solid black) + white median dot ---
    bp = ax.boxplot(
        data,
        positions=[1, 2],
        widths=0.18,
        patch_artist=True,
        showfliers=False,
        whis=(5, 95),
        boxprops=dict(facecolor='black', edgecolor='black', linewidth=LINEWIDTH),
        whiskerprops=dict(color='black', linewidth=LINEWIDTH),
        capprops=dict(color='black', linewidth=LINEWIDTH),
        medianprops=dict(color='none', linewidth=0),
    )

    # Hide horizontal caps
    for cap in bp.get('caps', []):
        cap.set_visible(False)

    # White median dots
    med = [np.nanmedian(data[0]), np.nanmedian(data[1])]
    ax.scatter(
        [1, 2],
        med,
        s=22,
        facecolors='white',
        edgecolors='black',
        linewidths=LINEWIDTH,
        zorder=5,
    )

    ax.set_xticks([1, 2])
    ax.set_xticklabels(['', ''])
    ax.tick_params(axis='x', length=0)

    #ax.set_title(title, loc='left')
    ax.set_ylabel(ylabel)
    ax.grid(True, axis='y', alpha=0.25)

def plot_violin_pair(df_pair, gene, factor, rep_label, out_pdf):
    import matplotlib as mpl
    from matplotlib.patches import Patch

    da = df_pair[df_pair['cond'] == 'Active']
    di = df_pair[df_pair['cond'] == 'Inactive']
    snap_a = da[snap_col].dropna().values
    snap_i = di[snap_col].dropna().values

    eps = 1e-12
    snap_a_log = np.log10(np.clip(np.asarray(snap_a, dtype=float), eps, None))
    snap_i_log = np.log10(np.clip(np.asarray(snap_i, dtype=float), eps, None))

    if len(snap_a_log) >= 1 and len(snap_i_log) >= 1:
        try:
            from scipy.stats import mannwhitneyu
        except Exception as e:
            raise RuntimeError(
                "scipy is required for Mann–Whitney U (Wilcoxon rank-sum) test. Please install scipy."
            ) from e
        p_value = mannwhitneyu(snap_a_log, snap_i_log, alternative='two-sided').pvalue
    else:
        p_value = np.nan

    rc = {
        # Arial first; fallback keeps execution robust if Arial unavailable
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'DejaVu Sans'],
        'font.size': FONT_SIZE,
        'axes.titlesize': TITLE_SIZE,
        'axes.labelsize': LABEL_SIZE,
        'xtick.labelsize': TICK_SIZE,
        'ytick.labelsize': TICK_SIZE,
        'legend.fontsize': LEGEND_SIZE,
        'axes.linewidth': LINEWIDTH,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
    }

    with mpl.rc_context(rc=rc):
        fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)
        # Fix axes box (adjust as needed)
        ax.set_position([0.18, 0.18, 0.45, 0.70])  # left, bottom, width, height
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

        _violin(
            ax,
            snap_a_log,
            snap_i_log,
            f"{gene} – {factor} ({rep_label})",
            'log10(SNAP intensity)',
        )

        p_txt = _format_p_value(p_value)
        _add_significance_bar(ax, 1, 2, snap_a_log, snap_i_log, p_txt, lw=LINEWIDTH)

        handles = [
            Patch(facecolor=COLOR_ACTIVE, edgecolor='0.25', alpha=0.55, label='Active'),
            Patch(facecolor=COLOR_INACTIVE, edgecolor='0.25', alpha=0.55, label='Inactive'),
        ]
        ax.legend(handles=handles, loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)

        os.makedirs(os.path.dirname(out_pdf), exist_ok=True)
        fig.savefig(out_pdf, dpi=300)
        plt.close(fig)
        print('Saved →', out_pdf)

# 1) sample pairs
for gene, factor in sample_pairs:
    df_gf = df_center[(df_center['gene'] == gene) & (df_center['factor'] == factor)].copy()
    if df_gf.empty:
        continue
    reps = sorted(df_gf['replicate'].drop_duplicates().tolist(), key=_rep_sort_key)
    for rep in reps:
        df_pair = df_gf[df_gf['replicate'] == rep].copy()
        if df_pair.empty:
            continue
        rep_label = _rep_tag(rep)
        out_pdf = os.path.join(
            OUT_VIOLIN,
            f"sample_violin_{_safe_filename(gene)}_{_safe_filename(factor)}_{rep_label}.pdf",
        )
        plot_violin_pair(df_pair, gene, factor, rep_label, out_pdf)

# 2) all pairs
pairs_all = df_center[['gene', 'factor']].drop_duplicates().values.tolist()
for gene, factor in pairs_all:
    df_gf = df_center[(df_center['gene'] == gene) & (df_center['factor'] == factor)].copy()
    if df_gf.empty:
        continue
    reps = sorted(df_gf['replicate'].drop_duplicates().tolist(), key=_rep_sort_key)
    for rep in reps:
        df_pair = df_gf[df_gf['replicate'] == rep].copy()
        if df_pair.empty:
            continue
        rep_label = _rep_tag(rep)
        out_pdf = os.path.join(
            OUT_VIOLIN,
            f"violin_{_safe_filename(gene)}_{_safe_filename(factor)}_{rep_label}.pdf",
        )
        plot_violin_pair(df_pair, gene, factor, rep_label, out_pdf)

## Correlation heatmap (SNAP vs MCP at the center)

Compute Spearman correlations between SNAP center intensity and MCP center r_mcp across selected genes and factors.

- **Input:** `df_center`.
- **Output:** heatmap PDF in `outputs/Figs_thr1.30_1.15/variability/heatmap/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "Arial",
})

# ----------------------------
# 1) Settings (edit here for other sets)
# ----------------------------
GENES = ["Nanog", "Sox2", "Usp5", "Dnmt3L"]

# Factor names must exactly match df_center["factor"] values
FACTOR_REP_LIST = [
    ("RNAPII_Ser5ph_cy", "2", "RNAPII-Ser5ph"),
    ("RNAPII_Ser2ph_cy", "1", "RNAPII-Ser2ph"),
    ("H3K9ac_cy", "1", "H3K9ac"),
    ("H3K27ac_cy", "2", "H3K27ac"),
    ("H3K4me3_cy", "1", "H3K4me3"),
    ("H3K27me3_cy", "1", "H3K27me3"),
    ("H2B_cy", "1", "H2B"),
    ("NLS_cy", "1", "NLS"),
]

# The H3K9ac dataset used in Supplementary Fig. 4c is normally rep1.
# Sox2 is the exception because its main Fig. 1c acquisition is public rep1;
# the unchanged 250909 correlation dataset is therefore public rep2.
REPLICATE_OVERRIDES = {("Sox2", "H3K9ac_cy"): "2"}

# Same as Cell 48: choose SNAP center metric (mean/median)
SNAP_METRIC = "mean"  # "mean" or "median"
SNAP_COL = "snap_center_mean" if SNAP_METRIC == "mean" else "snap_center_median"

# Heatmap display settings
FIGSIZE = (4, 6)          # approx width 4 : height 6
CMAP = "bwr"              # negative=blue, 0=white, positive=red
V_MIN, V_MAX = -1, 1
ANNOT_FMT = "{:.2f}"      # decimals inside cells


# ----------------------------
# 2) Spearman rho (fallback: rank-transform + Pearson if scipy unavailable)
# ----------------------------
def spearman_rho(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if x.size < 2:
        return np.nan

    try:
        from scipy.stats import spearmanr
        rho = spearmanr(x, y).statistic
        return float(rho)
    except Exception:
        # fallback: rank-transform + Pearson
        xr = pd.Series(x).rank(method="average").to_numpy()
        yr = pd.Series(y).rank(method="average").to_numpy()
        if xr.size < 2:
            return np.nan
        return float(np.corrcoef(xr, yr)[0, 1])


# ----------------------------
# 3) Build the matrix (8x4)
# ----------------------------
def build_rho_matrix(df_center, genes, factor_rep_list, snap_col, mcp_col="r_mcp_center"):
    rows = []
    row_labels = []
    for factor, rep, disp in factor_rep_list:
        row_vals = []
        for gene in genes:
            replicate_to_use = REPLICATE_OVERRIDES.get((gene, factor), rep)
            df_sub = df_center[
                (df_center["gene"] == gene) &
                (df_center["factor"] == factor) &
                (df_center["replicate"] == replicate_to_use)
            ]
            rho = spearman_rho(df_sub[snap_col].to_numpy(), df_sub[mcp_col].to_numpy())
            row_vals.append(rho)
        rows.append(row_vals)
        row_labels.append(disp)  # key for display names
    mat = np.array(rows, dtype=float)
    return mat, row_labels, list(genes)


mat, row_labels, col_labels = build_rho_matrix(
    df_center=df_center,
    genes=GENES,
    factor_rep_list=FACTOR_REP_LIST,
    snap_col=SNAP_COL,
    mcp_col="r_mcp_center",
)


# ----------------------------
# 4) Heatmap (annotate rho in each cell)
# ----------------------------
fig, ax = plt.subplots(figsize=FIGSIZE)

im = ax.imshow(mat, cmap=CMAP, vmin=V_MIN, vmax=V_MAX, aspect="auto")

# ticks
ax.set_xticks(np.arange(len(col_labels)))
ax.set_xticklabels(
    col_labels,
    rotation=90,
    ha="center",
    va="top",
    fontstyle="italic",
)
ax.tick_params(axis="x", pad=6)
ax.set_yticks(np.arange(len(row_labels)))
ax.set_yticklabels(row_labels)

# annotate (rho)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat[i, j]
        if np.isfinite(v):
            ax.text(j, i, ANNOT_FMT.format(v), ha="center", va="center", color="black", fontsize=9)
        else:
            ax.text(j, i, "NA", ha="center", va="center", color="black", fontsize=9)

# colorbar
cbar = fig.colorbar(im, ax=ax, fraction=0.05, pad=0.04)
cbar.set_label("Spearman ρ (SNAP vs relative MCP intensity at center)")

#ax.set_title("Spearman correlation (All points; Active/Inactive pooled)")

fig.tight_layout()
plt.show()

OUT_HEATMAP = os.path.join(FIGS_ROOT, "variability", "heatmap")
os.makedirs(OUT_HEATMAP, exist_ok=True)

out_pdf = os.path.join(OUT_HEATMAP, "heatmap_spearman_center_snap_vs_r_mcp_center.pdf")
fig.savefig(out_pdf)  # avoid bbox_inches="tight" to reduce size jitter
print("Saved →", out_pdf)

## Group-wise Active/ Inactive comparison: mean intensity from the recomputed CSV

Here we quantify state-dependent differences in **mean nuclear intensity** for each group (target gene x factor x replicate) using the recomputed table.

> **Assumptions (user-defined)**
> - In the same folder as `saved_onoff_data_for_paper_thr1.30_1.15.pkl`, there is exactly one file named `df_loaded_updated_with_state_thr1.30_1.15.csv`.

- **Step 0 (new):** recompute `state` (ON/OFF) directly from the raw spot TIFs and the updated CSV, so downstream analyses use a consistent thresholding rule.
- **Grouping:** target gene x target factor x `replicate`
- **State definition:** split by `state` (ON/OFF) into Active vs Inactive
- **Channels / metrics:** compare the distributions of mean-intensity columns  
  (`SNAPtag_mean_intensity`, `MCP_mean_intensity`, `mTetR_mean_intensity`)
- **Statistics:** Mann–Whitney U test (two-sided) for Active vs Inactive within each group and channel
- **Outputs:** tables/figures are written to the output paths specified in the following code cell


**Notes on the recomputation step**

- The script reads the dataset roots from the Excel file, loads `segmented/csv/df_loaded_updated.csv`, and recalculates `state` using the thresholds specified above.
- The output is written into each dataset's `pkl/` directory as `df_loaded_updated_with_state_thr1.30_1.15.csv` (same delimiter as the input CSV).
- If you need additional metrics (`r_mTetR`, `r_mcp`, distances), rerun with `--add-metrics`.


In [ ]:
import os
import re
from math import ceil
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import mannwhitneyu

# =========================
# Config
# =========================
INTENSITY_COLS = [
    ("SNAPtag_mean_intensity", "SNAPtag"),
    ("MCP_mean_intensity", "MCP"),
    ("mTetR_mean_intensity", "mTetR"),
]

STATE_LABELS = {"inactive": "Inactive", "active": "Active"}

# Output location (follows existing notebook convention)
FIGS_ROOT = os.path.join("..", "outputs", "Figs_thr1.30_1.15")
OUT_DIR = os.path.join(FIGS_ROOT, "variability", "violin_mean_intensity_from_csv")
os.makedirs(OUT_DIR, exist_ok=True)

SHOW = "q"  # q-values only
GRID_NROWS, GRID_NCOLS = 8, 8  # 8x8 panels per page
FIGSIZE = (24, 24)

# =========================
# Utilities
# =========================
def _bh_fdr(p_values: np.ndarray) -> np.ndarray:
    p = np.asarray(p_values, dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return q
    p_ok = p[ok]
    n = p_ok.size
    order = np.argsort(p_ok)
    p_sorted = p_ok[order]
    q_sorted = p_sorted * n / np.arange(1, n + 1)
    q_sorted = np.minimum.accumulate(q_sorted[::-1])[::-1]
    q_sorted = np.clip(q_sorted, 0.0, 1.0)
    q_ok = np.empty_like(p_ok)
    q_ok[order] = q_sorted
    q[ok] = q_ok
    return q


def _format_p_or_q(x: float) -> str:
    if x is None or (isinstance(x, float) and not np.isfinite(x)):
        return "n/a"
    if x < 1e-4:
        return "<1e-4"
    return f"{x:.3g}"


def _normalize_path_fallback(path: str) -> str:
    path = str(path)
    path = os.path.expanduser(path)
    if os.path.exists(path):
        return os.path.normpath(path)
    alt = os.path.join(os.getcwd(), path)
    if os.path.exists(alt):
        return os.path.normpath(alt)
    return os.path.normpath(path)


def _get_df_first_sheet() -> pd.DataFrame:
    if "df_first_sheet" in globals() and isinstance(globals()["df_first_sheet"], pd.DataFrame):
        return globals()["df_first_sheet"]
    excel_path = globals().get("excel_path", "../../../data/01_Snapshot_analysis/pkl/pkl_file_position_original.xlsx")
    return pd.read_excel(excel_path, sheet_name=0)


def _well_num_from_any(well_id) -> int:
    s = str(well_id)
    m = re.search(r"(\d+)", s)
    if m is None:
        raise ValueError(f"Could not parse numeric well id from: {well_id}")
    return int(m.group(1))


def _extract_well3_from_image_id(image_id: str) -> Optional[str]:
    if image_id is None:
        return None
    s = str(image_id)
    m = re.search(r"well\s*0*(\d+)", s, flags=re.IGNORECASE)
    if m is None:
        return None
    try:
        n = int(m.group(1))
    except Exception:
        return None
    return f"well{n:03d}"


def _normalize_state_to_active_inactive(x) -> Optional[str]:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().lower()
    if s == "":
        return None
    if "inactive" in s:
        return "inactive"
    if "active" in s:
        return "active"
    # allow ON/OFF, on/off, 1/0, True/False etc (string-based)
    if re.search(r"\bon\b", s):
        return "active"
    if re.search(r"\boff\b", s):
        return "inactive"
    if s in {"1", "true", "t", "yes"}:
        return "active"
    if s in {"0", "false", "f", "no"}:
        return "inactive"
    return None


def _draw_bracket(ax, x1: float, x2: float, y: float, text: str, fontsize: int = 7):
    trans = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
    ax.plot([x1, x1, x2, x2], [y, y + 0.03, y + 0.03, y], transform=trans, color="black", lw=0.8, clip_on=False)
    ax.text((x1 + x2) / 2, y + 0.035, text, transform=trans, ha="center", va="bottom", fontsize=fontsize)


def _pval_annotation_value(row: pd.Series) -> float:
    if SHOW == "q":
        return float(row.get("q_value", np.nan))
    return float(row.get("p_value", np.nan))


# =========================
# 1) Load per-group CSV (from xlsx["csv_file"]) and build long table
# =========================
rows = _get_df_first_sheet().copy()

# Replicate numbers describe public figure provenance. Some main-figure-only
# datasets were not part of the later aggregate state-dependence summary.
if "include_in_state_dependence_summary" in rows.columns:
    include_summary = rows["include_in_state_dependence_summary"].fillna(True).astype(bool)
    rows = rows.loc[include_summary].copy()

# csv_file is the source of truth for CSV paths
required_cols = ["csv_file", "STtag_KI_target", "Factor_name", "replicate"]
missing = [c for c in required_cols if c not in rows.columns]
if missing:
    raise KeyError(f"xlsx/df_first_sheet is missing required columns: {missing}")

rows = rows.dropna(subset=["csv_file"]).copy()
rows["STtag_KI_target"] = rows["STtag_KI_target"].astype(str)
rows["Factor_name"] = rows["Factor_name"].astype(str)
rows["replicate"] = rows["replicate"].astype(str)

# well_id grouping is not needed (each referenced CSV already contains only the intended well)
rows["well_id"] = "all"
has_well = False

# Unique group rows (avoid double counting if xlsx has duplicates)
_group_cols = ["csv_file", "STtag_KI_target", "Factor_name", "replicate"]
if "pkl_file" in rows.columns:
    _group_cols = ["pkl_file"] + _group_cols

group_rows = (
    rows[_group_cols]
    .drop_duplicates()
    .sort_values(["STtag_KI_target", "Factor_name", "replicate", "csv_file"])
)

_normalize_path = globals().get("_normalize_path", _normalize_path_fallback)
csv_cache: dict[str, pd.DataFrame] = {}
records = []
skipped = []

for idx, r in group_rows.iterrows():
    target = str(r["STtag_KI_target"]).strip()
    factor = str(r["Factor_name"]).strip()
    rep = str(r["replicate"]).strip()
    well3 = "all"

    pkl_file_for_log = None
    if "pkl_file" in group_rows.columns:
        pkl_file_for_log = str(r["pkl_file"]) if pd.notna(r["pkl_file"]) else None

    csv_file_raw = r.get("csv_file", None)
    if csv_file_raw is None or (isinstance(csv_file_raw, float) and np.isnan(csv_file_raw)) or str(csv_file_raw).strip() == "":
        skipped.append({
            "target": target,
            "factor": factor,
            "replicate": rep,
            "well": well3,
            "pkl_file": pkl_file_for_log,
            "csv_file": None,
            "csv": None,
            "reason": "Missing csv_file in xlsx",
        })
        continue

    # xlsx 'csv_file' is a relative full path; use as-is (resolve relative to CWD if needed)
    csv_path = Path(_normalize_path(str(csv_file_raw)))
    csv_path_str = str(csv_path)

    if not csv_path.exists():
        skipped.append({
            "target": target,
            "factor": factor,
            "replicate": rep,
            "well": well3,
            "pkl_file": pkl_file_for_log,
            "csv_file": str(csv_file_raw),
            "csv": csv_path_str,
            "reason": "CSV not found (from xlsx csv_file)",
        })
        continue

    if csv_path_str not in csv_cache:
        try:
            df_csv = pd.read_csv(csv_path_str)
        except Exception as e:
            skipped.append({
                "target": target,
                "factor": factor,
                "replicate": rep,
                "well": well3,
                "pkl_file": pkl_file_for_log,
                "csv_file": str(csv_file_raw),
                "csv": csv_path_str,
                "reason": f"Failed to read CSV: {type(e).__name__}: {e}",
            })
            continue

        # state normalization (required)
        if "state" not in df_csv.columns:
            skipped.append({
                "target": target,
                "factor": factor,
                "replicate": rep,
                "well": well3,
                "pkl_file": pkl_file_for_log,
                "csv_file": str(csv_file_raw),
                "csv": csv_path_str,
                "reason": "Missing required column in CSV: state",
            })
            continue

        df_csv["state_norm"] = df_csv["state"].map(_normalize_state_to_active_inactive)
        csv_cache[csv_path_str] = df_csv
    else:
        df_csv = csv_cache[csv_path_str]

    sub = df_csv.copy()

    # Keep only rows with valid state
    sub = sub.loc[sub["state_norm"].isin(["active", "inactive"])].copy()
    if sub.empty:
        skipped.append({
            "target": target,
            "factor": factor,
            "replicate": rep,
            "well": well3,
            "pkl_file": pkl_file_for_log,
            "csv_file": str(csv_file_raw),
            "csv": csv_path_str,
            "reason": "No rows with parsable state (state_norm empty)",
        })
        continue

    # Append intensity values as long format
    for col, ch_name in INTENSITY_COLS:
        if col not in sub.columns:
            skipped.append({
                "target": target,
                "factor": factor,
                "replicate": rep,
                "well": well3,
                "pkl_file": pkl_file_for_log,
                "csv_file": str(csv_file_raw),
                "csv": csv_path_str,
                "reason": f"Missing intensity column: {col}",
            })
            continue

        vals = pd.to_numeric(sub[col], errors="coerce")
        for state_norm, v in zip(sub["state_norm"].values, vals.values):
            if not np.isfinite(v):
                continue
            records.append({
                "target": target,
                "factor": factor,
                "replicate": rep,
                "well": well3,
                "state": STATE_LABELS[state_norm],
                "channel": ch_name,
                "value": float(v),
                "csv": csv_path_str,
            })

df_long_csv = pd.DataFrame.from_records(records)
if df_long_csv.empty:
    raise RuntimeError("No data extracted from CSVs (df_long_csv is empty). Check xlsx csv_file paths / CSV columns (state, intensity cols).")

long_csv = os.path.join(OUT_DIR, "mean_intensity_long_from_csv.csv")
df_long_csv.to_csv(long_csv, index=False)
print(f"Saved long table: {long_csv} (n={len(df_long_csv):,})")

if skipped:
    skipped_df = pd.DataFrame.from_records(skipped)
    skipped_csv = os.path.join(OUT_DIR, "mean_intensity_skipped_groups.csv")
    skipped_df.to_csv(skipped_csv, index=False)
    print(f"Saved skipped groups log: {skipped_csv} (n={len(skipped_df):,})")

# =========================
# 2) MW test per groupxchannel (Active vs Inactive) + BH/FDR
# =========================
stat_rows = []
group_cols = ["target", "factor", "replicate", "well", "channel"]
for (target, factor, rep, well, channel), sub in df_long_csv.groupby(group_cols):
    on_vals = sub.loc[sub["state"] == STATE_LABELS["active"], "value"].dropna().values
    off_vals = sub.loc[sub["state"] == STATE_LABELS["inactive"], "value"].dropna().values
    if on_vals.size == 0 or off_vals.size == 0:
        p_val = np.nan
        u_stat = np.nan
    else:
        u_stat, p_val = mannwhitneyu(on_vals, off_vals, alternative="two-sided")
    stat_rows.append({
        "target": target,
        "factor": factor,
        "replicate": rep,
        "well": well,
        "channel": channel,
        "n_active": int(on_vals.size),
        "n_inactive": int(off_vals.size),
        "u_stat": float(u_stat) if np.isfinite(u_stat) else np.nan,
        "p_value": float(p_val) if np.isfinite(p_val) else np.nan,
    })

df_stats_csv = pd.DataFrame(stat_rows)
df_stats_csv["q_value"] = _bh_fdr(df_stats_csv["p_value"].values)
stats_csv = os.path.join(OUT_DIR, "mean_intensity_mw_stats_from_csv.csv")
df_stats_csv.to_csv(stats_csv, index=False)
print(f"Saved stats table: {stats_csv} (n={len(df_stats_csv):,})")

stat_key = lambda r: (r["target"], r["factor"], r["replicate"], r["well"], r["channel"])
stat_map = {stat_key(r): r for _, r in df_stats_csv.iterrows()}

# =========================
# 3) Plot: 1 panel = 1 (target,factor,rep,well) group; 3-channel ON/OFF violins + q-values
# =========================
def plot_group_panel(ax, group_df: pd.DataFrame, target: str, factor: str, rep: str, well: str):
    datasets = []
    positions = []
    pos = 1
    for _, ch_name in INTENSITY_COLS:
        off_vals = group_df.loc[(group_df["channel"] == ch_name) & (group_df["state"] == STATE_LABELS["inactive"]), "value"].dropna().values
        on_vals = group_df.loc[(group_df["channel"] == ch_name) & (group_df["state"] == STATE_LABELS["active"]), "value"].dropna().values
        datasets.extend([off_vals, on_vals])
        positions.extend([pos, pos + 1])
        pos += 3

    if all(len(d) == 0 for d in datasets):
        ax.axis("off")
        return

    parts = ax.violinplot(
        datasets,
        positions=positions,
        showmeans=False,
        showmedians=True,
        showextrema=False,
        widths=0.8,
    )

    inactive_color = "tab:gray"
    active_color = "tab:blue"
    for i, body in enumerate(parts["bodies"]):
        body.set_edgecolor("black")
        body.set_linewidth(0.6)
        if i % 2 == 0:
            body.set_facecolor(inactive_color)
            body.set_alpha(0.35)
        else:
            body.set_facecolor(active_color)
            body.set_alpha(0.35)

    if "cmedians" in parts:
        parts["cmedians"].set_color("black")
        parts["cmedians"].set_linewidth(0.8)

    ax.set_title(f"{target}\n{factor}\nrep{rep} | {well}", fontsize=8)
    ax.tick_params(axis="y", labelsize=7)
    ax.tick_params(axis="x", labelsize=7)
    ax.grid(axis="y", alpha=0.15)

    xticks = [1.5, 4.5, 7.5]
    ax.set_xticks(xticks)
    ax.set_xticklabels([name for _, name in INTENSITY_COLS], rotation=0)

    y = 0.86
    for j, (_, ch_name) in enumerate(INTENSITY_COLS):
        x1 = 1 + j * 3
        x2 = 2 + j * 3
        key = (target, factor, rep, well, ch_name)
        r = stat_map.get(key, None)
        if r is None:
            text = "q=n/a"
        else:
            val = _pval_annotation_value(r)
            text = f"q={_format_p_or_q(val)}"
        _draw_bracket(ax, x1, x2, y=y, text=text, fontsize=7)

    ax.set_xlim(0, 9)


groups = (
    df_long_csv[["target", "factor", "replicate", "well"]]
    .drop_duplicates()
    .sort_values(["target", "factor", "replicate", "well"])
    .itertuples(index=False, name=None)
)
groups = list(groups)
n_groups = len(groups)
per_page = GRID_NROWS * GRID_NCOLS
n_pages = ceil(n_groups / per_page)

out_pdf = os.path.join(OUT_DIR, "mean_intensity_violin_on_off_by_group_qvalue.pdf")
print(f"Writing multipage PDF: {out_pdf}")

with PdfPages(out_pdf) as pdf:
    for page in range(n_pages):
        fig, axes = plt.subplots(GRID_NROWS, GRID_NCOLS, figsize=FIGSIZE, constrained_layout=True)
        axes = np.asarray(axes).reshape(-1)
        start = page * per_page
        end = min((page + 1) * per_page, n_groups)
        page_groups = groups[start:end]

        for ax in axes[len(page_groups):]:
            ax.axis("off")

        for i, (target, factor, rep, well) in enumerate(page_groups):
            ax = axes[i]
            gdf = df_long_csv[
                (df_long_csv["target"] == target)
                & (df_long_csv["factor"] == factor)
                & (df_long_csv["replicate"] == rep)
                & (df_long_csv["well"] == well)
            ]
            plot_group_panel(ax, gdf, target, factor, rep, well)

        fig.suptitle("Mean intensity per detection (CSV): Active vs Inactive, BH-FDR q-values", fontsize=12)
        pdf.savefig(fig)
        plt.close(fig)

print("Done.")
print(f"- PDF: {out_pdf}")
print(f"- Long data: {long_csv}")
print(f"- Stats: {stats_csv}")

# Check that reporter expression does not drive Active/Inactive calling

Evaluate whether nuclear mean intensities (SNAPtag / MCP / mTetR) can explain state assignment, focusing on **effect size** rather than nominal significance.

- **A:** ECDFs of log2‑transformed, group‑centered nuclear intensities (equal sampling per group).
- **B:** Group‑wise median differences (Active − Inactive; log2 scale) by channel.
- **C‑1:** ROC‑AUC for state prediction using intensity alone (AUC derived from Mann–Whitney U).

## Split state‑dependence plots and summaries by (cy) presence

Separate state‑dependence outputs by fluorescence pair to avoid mixing intensity distributions.

- **Background:** `(cy)` in `Factor_name` indicates `mTetR‑YFP / MCP‑CFP`; non‑`(cy)` indicates `mTetR‑GFP / MCP‑RFP`.
- **Scope:** only Cell 56 outputs (state‑dependence PDF and summary CSV) are split; violin plots are unchanged.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# User settings
# =========================
FONT_FAMILY = "Arial"
FIG_W, FIG_H = 15, 9
LABEL_FONTSIZE = 12
TICK_FONTSIZE = 10
SEED = 0

# Input tables
# long table must contain: target, factor, replicate, well, state, channel, value
LONG_CSV = Path("../outputs/Figs_thr1.30_1.15/variability/violin_mean_intensity_from_csv/mean_intensity_long_from_csv.csv")

# stats table is optional but recommended: target, factor, replicate, well, channel, n_active, n_inactive, u_stat, q_value
STATS_CSV = Path("../outputs/Figs_thr1.30_1.15/variability/violin_mean_intensity_from_csv/mean_intensity_mw_stats_from_csv.csv")

# Channels to show (order)
CHANNELS = ["SNAPtag", "MCP", "mTetR"]

# State labels (must match LONG_CSV)
STATE_ACTIVE = "Active"
STATE_INACTIVE = "Inactive"

# Robust transform for comparability across groups
EPS_LOG = 1e-6  # pseudocount for log transform

# ECDF: equal-weight sampling per groupxstate to avoid huge-N groups dominating
N_PER_GROUP_PER_STATE = 400  # adjust (200-1000 is usually fine)

# Minimum counts per state required for group-level summaries
MIN_N_PER_STATE = 20

# Output
OUT_DIR = Path("../outputs/Figs_thr1.30_1.15/mean_intensity_state_dependence_summary")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PDF = OUT_DIR / "reporter_expression_state_dependence_ECDF_effect_AUC.pdf"
OUT_SUMMARY_CSV = OUT_DIR / "group_level_effect_and_auc_summary.csv"

# =========================
# Load
# =========================
df = pd.read_csv(LONG_CSV)

required = {"target","factor","replicate","well","state","channel","value"}
missing = required.difference(df.columns)
if missing:
    raise KeyError(f"LONG_CSV is missing columns: {missing}")

# Normalize dtypes
df["replicate"] = df["replicate"].astype(str)
df["well"] = df["well"].astype(str)
df["target"] = df["target"].astype(str)
df["factor"] = df["factor"].astype(str)
df["channel"] = df["channel"].astype(str)
df["state"] = df["state"].astype(str)

df = df[df["channel"].isin(CHANNELS)].copy()
df = df[df["state"].isin([STATE_ACTIVE, STATE_INACTIVE])].copy()

df["value"] = pd.to_numeric(df["value"], errors="coerce")
df = df[np.isfinite(df["value"])].copy()

# Group key
group_cols = ["target","factor","replicate","well","channel"]

# log2 transform
df["log2_value"] = np.log2(df["value"].to_numpy(dtype=float) + EPS_LOG)

# Within-group median centering (for ECDF only)
group_median = df.groupby(group_cols)["log2_value"].transform("median")
df["log2_centered"] = df["log2_value"] - group_median

# =========================
# Group-level effect size table (Panel B) + ROC-AUC (Panel C-1)
# =========================
# Effect: delta_median_log2 = median(log2|Active) - median(log2|Inactive)
g_list = []
for gkey, gdf in df.groupby(group_cols):
    a = gdf.loc[gdf["state"] == STATE_ACTIVE, "log2_value"].to_numpy(dtype=float)
    b = gdf.loc[gdf["state"] == STATE_INACTIVE, "log2_value"].to_numpy(dtype=float)
    if (a.size < MIN_N_PER_STATE) or (b.size < MIN_N_PER_STATE):
        continue
    delta_med = float(np.median(a) - np.median(b))
    g_list.append({
        "target": gkey[0],
        "factor": gkey[1],
        "replicate": gkey[2],
        "well": gkey[3],
        "channel": gkey[4],
        "n_active": int(a.size),
        "n_inactive": int(b.size),
        "delta_median_log2": delta_med,
        "fold_change_median": float(np.exp2(delta_med)),  # exp2 avoids using power operator
    })

eff_df = pd.DataFrame(g_list)
if eff_df.empty:
    raise RuntimeError("No groups passed MIN_N_PER_STATE filter. Lower MIN_N_PER_STATE or check input table.")

# ROC-AUC from Mann–Whitney U (preferred: use your stats table so we match previous MW computation)
if STATS_CSV.exists():
    st = pd.read_csv(STATS_CSV)
    for c in ["target","factor","replicate","well","channel"]:
        st[c] = st[c].astype(str)
    st = st[st["channel"].isin(CHANNELS)].copy()

    st["auc_signed"] = st["u_stat"] / (st["n_active"] * st["n_inactive"])
    st["auc_best"] = np.maximum(st["auc_signed"], 1.0 - st["auc_signed"])

    # merge to one summary
    summary = eff_df.merge(
        st[group_cols + ["auc_signed","auc_best","q_value"]],
        on=group_cols,
        how="left"
    )
else:
    # Fallback: compute AUC per group from ranks (equivalent to MW-U).
    # This is slower for huge groups but works.
    def auc_from_values(pos, neg):
        x = np.concatenate([pos, neg])
        y = np.concatenate([np.ones(pos.size, dtype=int), np.zeros(neg.size, dtype=int)])
        order = np.argsort(x)
        ranks = np.empty_like(order, dtype=float)
        ranks[order] = np.arange(1, x.size + 1, dtype=float)
        # average ranks for ties
        # simple tie handling
        xs = x[order]
        i = 0
        while i < xs.size:
            j = i
            while j + 1 < xs.size and xs[j + 1] == xs[i]:
                j += 1
            if j > i:
                r = ranks[order[i:j+1]].mean()
                ranks[order[i:j+1]] = r
            i = j + 1
        r_pos = ranks[y == 1].sum()
        n_pos = pos.size
        n_neg = neg.size
        u = r_pos - n_pos * (n_pos + 1) / 2.0
        auc = u / (n_pos * n_neg)
        return float(auc)

    auc_rows = []
    for gkey, gdf in df.groupby(group_cols):
        a = gdf.loc[gdf["state"] == STATE_ACTIVE, "log2_value"].to_numpy(dtype=float)
        b = gdf.loc[gdf["state"] == STATE_INACTIVE, "log2_value"].to_numpy(dtype=float)
        if (a.size < MIN_N_PER_STATE) or (b.size < MIN_N_PER_STATE):
            continue
        auc = auc_from_values(a, b)
        auc_rows.append({
            "target": gkey[0],
            "factor": gkey[1],
            "replicate": gkey[2],
            "well": gkey[3],
            "channel": gkey[4],
            "auc_signed": auc,
            "auc_best": max(auc, 1.0 - auc),
        })
    auc_df = pd.DataFrame(auc_rows)
    summary = eff_df.merge(auc_df, on=group_cols, how="left")

summary.to_csv(OUT_SUMMARY_CSV, index=False)
print("Saved summary table:", OUT_SUMMARY_CSV)

# =========================
# Panel A: ECDF (equal-weight sampling per groupxstate)
# =========================
rng = np.random.default_rng(SEED)

def ecdf(vals):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.array([]), np.array([])
    x = np.sort(vals)
    y = np.arange(1, x.size + 1, dtype=float) / x.size
    return x, y

def pooled_equal_weight_samples(df_in, channel, state, n_per_group):
    take = []
    sub = df_in[(df_in["channel"] == channel) & (df_in["state"] == state)].copy()
    for _, gdf in sub.groupby(group_cols):
        vals = gdf["log2_centered"].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            continue
        k = min(n_per_group, vals.size)
        idx = rng.choice(vals.size, size=k, replace=False)
        take.append(vals[idx])
    if not take:
        return np.array([], dtype=float)
    return np.concatenate(take)

# =========================
# Plot (A: ECDF, B: effect size, C-1: AUC)
# =========================
plt.rcParams.update({
    "font.family": FONT_FAMILY,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig = plt.figure(figsize=(FIG_W, FIG_H))
gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 1.0], wspace=0.35, hspace=0.45)

# ---- Panel A: three ECDF subplots (one per channel)
for i, ch in enumerate(CHANNELS):
    ax = fig.add_subplot(gs[0, i])

    vals_inact = pooled_equal_weight_samples(df, ch, STATE_INACTIVE, N_PER_GROUP_PER_STATE)
    vals_act = pooled_equal_weight_samples(df, ch, STATE_ACTIVE, N_PER_GROUP_PER_STATE)

    x0, y0 = ecdf(vals_inact)
    x1, y1 = ecdf(vals_act)

    ax.plot(x0, y0, label="Inactive", linewidth=1.5)
    ax.plot(x1, y1, label="Active", linewidth=1.5)

    ax.set_title(f"A  ECDF: {ch}", fontsize=LABEL_FONTSIZE)
    ax.set_xlabel("log2 nuclear mean intensity (centered within group)", fontsize=LABEL_FONTSIZE)
    if i == 0:
        ax.set_ylabel("ECDF", fontsize=LABEL_FONTSIZE)
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    ax.grid(alpha=0.2)
    if i == 2:
        ax.legend(frameon=False, fontsize=TICK_FONTSIZE)

# ---- Panel B: distribution of group-level median shifts (log2)
axB = fig.add_subplot(gs[1, 0])
for j, ch in enumerate(CHANNELS):
    v = summary.loc[summary["channel"] == ch, "delta_median_log2"].to_numpy(dtype=float)
    v = v[np.isfinite(v)]
    x = np.full(v.size, j + 1, dtype=float) + rng.normal(0.0, 0.06, size=v.size)
    axB.scatter(x, v, s=12, alpha=0.35, edgecolors="none")
# boxplot
dataB = [summary.loc[summary["channel"] == ch, "delta_median_log2"].dropna().to_numpy(dtype=float) for ch in CHANNELS]
axB.boxplot(dataB, positions=np.arange(1, len(CHANNELS) + 1), showfliers=False)
axB.axhline(0.0, linestyle="--", linewidth=1)
axB.set_xticks(np.arange(1, len(CHANNELS) + 1))
axB.set_xticklabels(CHANNELS, fontsize=TICK_FONTSIZE, rotation=20)
axB.set_title("B  Group-wise median shift (Active − Inactive)", fontsize=LABEL_FONTSIZE)
axB.set_ylabel("Δ median log2 intensity", fontsize=LABEL_FONTSIZE)
axB.tick_params(axis="y", labelsize=TICK_FONTSIZE)
axB.grid(alpha=0.2)

# ---- Panel C-1: ROC-AUC summary per group (best-direction AUC)
axC = fig.add_subplot(gs[1, 1])
if "auc_best" not in summary.columns:
    raise RuntimeError("AUC not found in summary table. Provide STATS_CSV or use the fallback AUC computation.")
dataC = []
for ch in CHANNELS:
    v = summary.loc[summary["channel"] == ch, "auc_best"].to_numpy(dtype=float)
    v = v[np.isfinite(v)]
    dataC.append(v)

axC.boxplot(dataC, positions=np.arange(1, len(CHANNELS) + 1), showfliers=False)
for j, ch in enumerate(CHANNELS):
    v = dataC[j]
    x = np.full(v.size, j + 1, dtype=float) + rng.normal(0.0, 0.06, size=v.size)
    axC.scatter(x, v, s=12, alpha=0.35, edgecolors="none")

axC.axhline(0.5, linestyle="--", linewidth=1)
axC.set_xticks(np.arange(1, len(CHANNELS) + 1))
axC.set_xticklabels(CHANNELS, fontsize=TICK_FONTSIZE, rotation=20)
axC.set_ylim(0.45, 0.80)
axC.set_title("C-1  Predictability by nuclear intensity (ROC-AUC)", fontsize=LABEL_FONTSIZE)
axC.set_ylabel("best-direction AUC", fontsize=LABEL_FONTSIZE)
axC.tick_params(axis="y", labelsize=TICK_FONTSIZE)
axC.grid(alpha=0.2)

# ---- Panel C-2 (optional): leave blank or use for quantile-based Active fraction
axC2 = fig.add_subplot(gs[1, 2])
# axC2.axis("off")
# axC2.set_title("C-2  (optional) quantile-based Active fraction", fontsize=LABEL_FONTSIZE)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N_QUANTILES = 4
MIN_TOTAL_N = 80  # per group threshold to make quantiles meaningful

# Build per-group quantile Active enrichment
rows = []
for gkey, gdf in df.groupby(group_cols):
    # gdf contains one channel already because channel is in group_cols
    ch = gkey[4]
    if ch not in CHANNELS:
        continue

    x = gdf["log2_value"].to_numpy(dtype=float)
    y = (gdf["state"].to_numpy() == STATE_ACTIVE).astype(int)
    ok = np.isfinite(x)
    x = x[ok]
    y = y[ok]
    if x.size < MIN_TOTAL_N:
        continue
    if y.min() == y.max():
        continue  # need both states

    # quantile bins within group
    try:
        bins = pd.qcut(x, q=N_QUANTILES, labels=False, duplicates="drop")
    except Exception:
        continue
    if bins is None:
        continue
    bins = np.asarray(bins)
    if np.unique(bins).size < N_QUANTILES:
        continue

    p0 = y.mean()
    for q in range(N_QUANTILES):
        m = (bins == q)
        if m.sum() == 0:
            continue
        pq = y[m].mean()
        rows.append({
            "channel": ch,
            "quantile": q + 1,
            "active_frac": float(pq),
            "delta_from_baseline": float(pq - p0),
        })

qdf = pd.DataFrame(rows)
# mean ± SEM across groups
summ = qdf.groupby(["channel","quantile"]).agg(
    mean_delta=("delta_from_baseline","mean"),
    sem_delta=("delta_from_baseline", lambda v: np.std(v, ddof=1) / np.sqrt(len(v)) if len(v) > 1 else np.nan),
    n_groups=("delta_from_baseline","size")
).reset_index()

# Plot
#plt.rcParams.update({"font.family": FONT_FAMILY, "pdf.fonttype": 42, "ps.fonttype": 42})
#fig, ax = plt.subplots(figsize=(5.2, 3.6))

for ch in CHANNELS:
    sub = summ[summ["channel"] == ch].sort_values("quantile")
    axC2.errorbar(
        sub["quantile"].to_numpy(),
        sub["mean_delta"].to_numpy(),
        yerr=sub["sem_delta"].to_numpy(),
        marker="o",
        linewidth=1.5,
        label=ch
    )

axC2.axhline(0.0, linestyle="--", linewidth=1)
axC2.set_xticks(np.arange(1, N_QUANTILES + 1))
axC2.set_xlabel("Within-group intensity quantile", fontsize=LABEL_FONTSIZE)
axC2.set_ylabel("Δ Active fraction (vs group baseline)", fontsize=LABEL_FONTSIZE)
axC2.set_title("Quantile-based Active enrichment (equal-weight across groups)", fontsize=LABEL_FONTSIZE)
axC2.tick_params(axis="both", labelsize=TICK_FONTSIZE)
axC2.grid(alpha=0.2)
axC2.legend(frameon=False, fontsize=TICK_FONTSIZE)




fig.tight_layout()
fig.savefig(OUT_PDF, bbox_inches="tight")
print("Saved figure:", OUT_PDF)
plt.show()

# Quick numeric summary for manuscript
for ch in CHANNELS:
    v_fc = summary.loc[summary["channel"] == ch, "delta_median_log2"].dropna().to_numpy(dtype=float)
    v_auc = summary.loc[summary["channel"] == ch, "auc_best"].dropna().to_numpy(dtype=float)
    if v_fc.size > 0 and v_auc.size > 0:
        print(ch, "median Δlog2 =", float(np.median(v_fc)), "median AUC(best) =", float(np.median(v_auc)))




def _load_design_table() -> pd.DataFrame:
    if "df_first_sheet" in globals() and isinstance(globals()["df_first_sheet"], pd.DataFrame):
        return globals()["df_first_sheet"]
    excel_path = globals().get("excel_path", '../../../data/01_Snapshot_analysis/pkl/pkl_file_position_original.xlsx')
    return pd.read_excel(excel_path, sheet_name=0)


def _split_factors_by_cy(design: pd.DataFrame):
    if "Factor_name" not in design.columns:
        raise KeyError("Design xlsx is missing required column: Factor_name")
    s = design["Factor_name"].dropna().astype(str).str.strip()
    is_cy = s.str.contains(r"\(cy\)", case=False, regex=True, na=False)
    cy = set(s[is_cy].tolist())
    noncy = set(s[~is_cy].tolist())
    return cy, noncy


def _prepare_long_df_from_csv(long_csv: Path) -> pd.DataFrame:
    df = pd.read_csv(long_csv)
    required = {"target","factor","replicate","well","state","channel","value"}
    missing = required.difference(df.columns)
    if missing:
        raise KeyError(f"LONG_CSV is missing columns: {missing}")
    for c in ["replicate","well","target","factor","channel","state"]:
        df[c] = df[c].astype(str)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df[np.isfinite(df["value"])].copy()
    # log2 transform
    eps_log = globals().get("EPS_LOG", 1e-6)
    df["log2_value"] = np.log2(df["value"].to_numpy(dtype=float) + float(eps_log))
    # group key
    group_cols = ["target","factor","replicate","well","channel"]
    group_median = df.groupby(group_cols)["log2_value"].transform("median")
    df["log2_centered"] = df["log2_value"] - group_median
    return df


def _run_state_dependence(df_in: pd.DataFrame, out_pdf: Path, out_summary_csv: Path):
    CHANNELS = globals().get("CHANNELS", ["SNAPtag", "MCP", "mTetR"])
    STATE_ACTIVE = globals().get("STATE_ACTIVE", "Active")
    STATE_INACTIVE = globals().get("STATE_INACTIVE", "Inactive")
    MIN_N_PER_STATE = int(globals().get("MIN_N_PER_STATE", 20))
    N_PER_GROUP_PER_STATE = int(globals().get("N_PER_GROUP_PER_STATE", 400))
    SEED = int(globals().get("SEED", 0))

    FONT_FAMILY = globals().get("FONT_FAMILY", "Arial")
    FIG_W = int(globals().get("FIG_W", 15))
    FIG_H = int(globals().get("FIG_H", 9))
    LABEL_FONTSIZE = int(globals().get("LABEL_FONTSIZE", 12))
    TICK_FONTSIZE = int(globals().get("TICK_FONTSIZE", 10))

    STATS_CSV = globals().get("STATS_CSV", None)
    if STATS_CSV is not None:
        STATS_CSV = Path(STATS_CSV)

    df = df_in.copy()
    df = df[df["channel"].isin(CHANNELS)].copy()
    df = df[df["state"].isin([STATE_ACTIVE, STATE_INACTIVE])].copy()

    if df.empty:
        raise RuntimeError("No rows after filtering by channels/states.")

    group_cols = ["target","factor","replicate","well","channel"]

    # Group-level effect size table (Panel B) + ROC-AUC (Panel C-1)
    g_list = []
    for gkey, gdf in df.groupby(group_cols):
        a = gdf.loc[gdf["state"] == STATE_ACTIVE, "log2_value"].to_numpy(dtype=float)
        b = gdf.loc[gdf["state"] == STATE_INACTIVE, "log2_value"].to_numpy(dtype=float)
        if (a.size < MIN_N_PER_STATE) or (b.size < MIN_N_PER_STATE):
            continue
        delta_med = float(np.median(a) - np.median(b))
        g_list.append({
            "target": gkey[0],
            "factor": gkey[1],
            "replicate": gkey[2],
            "well": gkey[3],
            "channel": gkey[4],
            "n_active": int(a.size),
            "n_inactive": int(b.size),
            "delta_median_log2": delta_med,
            "fold_change_median": float(np.exp2(delta_med)),
        })

    eff_df = pd.DataFrame(g_list)
    if eff_df.empty:
        raise RuntimeError("No groups passed MIN_N_PER_STATE filter after splitting.")

    if STATS_CSV is not None and STATS_CSV.exists():
        st = pd.read_csv(STATS_CSV)
        for c in ["target","factor","replicate","well","channel"]:
            st[c] = st[c].astype(str)
        st = st[st["channel"].isin(CHANNELS)].copy()
        st["auc_signed"] = st["u_stat"] / (st["n_active"] * st["n_inactive"])
        st["auc_best"] = np.maximum(st["auc_signed"], 1.0 - st["auc_signed"])
        summary = eff_df.merge(
            st[group_cols + ["auc_signed","auc_best","q_value"]],
            on=group_cols,
            how="left",
        )
    else:
        def auc_from_values(pos, neg):
            x = np.concatenate([pos, neg])
            y = np.concatenate([np.ones(pos.size, dtype=int), np.zeros(neg.size, dtype=int)])
            order = np.argsort(x)
            ranks = np.empty_like(order, dtype=float)
            ranks[order] = np.arange(1, x.size + 1, dtype=float)
            xs = x[order]
            i = 0
            while i < xs.size:
                j = i
                while j + 1 < xs.size and xs[j + 1] == xs[i]:
                    j += 1
                if j > i:
                    r = ranks[order[i : j + 1]].mean()
                    ranks[order[i : j + 1]] = r
                i = j + 1
            r_pos = ranks[y == 1].sum()
            n_pos = pos.size
            n_neg = neg.size
            u = r_pos - n_pos * (n_pos + 1) / 2.0
            auc = u / (n_pos * n_neg)
            return float(auc)

        auc_rows = []
        for gkey, gdf in df.groupby(group_cols):
            a = gdf.loc[gdf["state"] == STATE_ACTIVE, "log2_value"].to_numpy(dtype=float)
            b = gdf.loc[gdf["state"] == STATE_INACTIVE, "log2_value"].to_numpy(dtype=float)
            if (a.size < MIN_N_PER_STATE) or (b.size < MIN_N_PER_STATE):
                continue
            auc = auc_from_values(a, b)
            auc_rows.append({
                "target": gkey[0],
                "factor": gkey[1],
                "replicate": gkey[2],
                "well": gkey[3],
                "channel": gkey[4],
                "auc_signed": auc,
                "auc_best": max(auc, 1.0 - auc),
            })
        auc_df = pd.DataFrame(auc_rows)
        summary = eff_df.merge(auc_df, on=group_cols, how="left")

    out_summary_csv.parent.mkdir(parents=True, exist_ok=True)
    summary.to_csv(out_summary_csv, index=False)
    print("Saved split summary table:", out_summary_csv)

    rng = np.random.default_rng(SEED)

    def ecdf(vals):
        vals = np.asarray(vals, dtype=float)
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            return np.array([]), np.array([])
        x = np.sort(vals)
        y = np.arange(1, x.size + 1, dtype=float) / x.size
        return x, y

    def pooled_equal_weight_samples(df_in, channel, state, n_per_group):
        take = []
        sub = df_in[(df_in["channel"] == channel) & (df_in["state"] == state)].copy()
        for _, gdf in sub.groupby(group_cols):
            vals = gdf["log2_centered"].to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            if vals.size == 0:
                continue
            k = min(n_per_group, vals.size)
            idx = rng.choice(vals.size, size=k, replace=False)
            take.append(vals[idx])
        if not take:
            return np.array([], dtype=float)
        return np.concatenate(take)

    plt.rcParams.update({"font.family": FONT_FAMILY, "pdf.fonttype": 42, "ps.fonttype": 42})
    fig = plt.figure(figsize=(FIG_W, FIG_H))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 1.0], wspace=0.35, hspace=0.45)

    # Panel A
    for i, ch in enumerate(CHANNELS):
        ax = fig.add_subplot(gs[0, i])
        vals_inact = pooled_equal_weight_samples(df, ch, STATE_INACTIVE, N_PER_GROUP_PER_STATE)
        vals_act = pooled_equal_weight_samples(df, ch, STATE_ACTIVE, N_PER_GROUP_PER_STATE)
        x0, y0 = ecdf(vals_inact)
        x1, y1 = ecdf(vals_act)
        ax.plot(x0, y0, label="Inactive", linewidth=1.5)
        ax.plot(x1, y1, label="Active", linewidth=1.5)
        ax.set_title(f"A  ECDF: {ch}", fontsize=LABEL_FONTSIZE)
        ax.set_xlabel("log2 nuclear mean intensity (centered within group)", fontsize=LABEL_FONTSIZE)
        if i == 0:
            ax.set_ylabel("ECDF", fontsize=LABEL_FONTSIZE)
        ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
        ax.grid(alpha=0.2)
        if i == 2:
            ax.legend(frameon=False, fontsize=TICK_FONTSIZE)

    # Panel B
    axB = fig.add_subplot(gs[1, 0])
    for j, ch in enumerate(CHANNELS):
        v = summary.loc[summary["channel"] == ch, "delta_median_log2"].to_numpy(dtype=float)
        v = v[np.isfinite(v)]
        x = np.full(v.size, j + 1, dtype=float) + rng.normal(0.0, 0.06, size=v.size)
        axB.scatter(x, v, s=12, alpha=0.35, edgecolors="none")
    dataB = [summary.loc[summary["channel"] == ch, "delta_median_log2"].dropna().to_numpy(dtype=float) for ch in CHANNELS]
    axB.boxplot(dataB, positions=np.arange(1, len(CHANNELS) + 1), showfliers=False)
    axB.axhline(0.0, linestyle="--", linewidth=1)
    axB.set_xticks(np.arange(1, len(CHANNELS) + 1))
    axB.set_xticklabels(CHANNELS, fontsize=TICK_FONTSIZE, rotation=20)
    axB.set_title("B  Group-wise median shift (Active − Inactive)", fontsize=LABEL_FONTSIZE)
    axB.set_ylabel("Δ median log2 intensity", fontsize=LABEL_FONTSIZE)
    axB.tick_params(axis="y", labelsize=TICK_FONTSIZE)
    axB.grid(alpha=0.2)

    # Panel C-1
    axC = fig.add_subplot(gs[1, 1])
    if "auc_best" not in summary.columns:
        raise RuntimeError("AUC not found in split summary table.")
    dataC = []
    for ch in CHANNELS:
        v = summary.loc[summary["channel"] == ch, "auc_best"].to_numpy(dtype=float)
        v = v[np.isfinite(v)]
        dataC.append(v)
    axC.boxplot(dataC, positions=np.arange(1, len(CHANNELS) + 1), showfliers=False)
    for j, ch in enumerate(CHANNELS):
        v = dataC[j]
        x = np.full(v.size, j + 1, dtype=float) + rng.normal(0.0, 0.06, size=v.size)
        axC.scatter(x, v, s=12, alpha=0.35, edgecolors="none")
    axC.axhline(0.5, linestyle="--", linewidth=1)
    axC.set_xticks(np.arange(1, len(CHANNELS) + 1))
    axC.set_xticklabels(CHANNELS, fontsize=TICK_FONTSIZE, rotation=20)
    axC.set_ylim(0.45, 0.80)
    axC.set_title("C-1  Predictability by nuclear intensity (ROC-AUC)", fontsize=LABEL_FONTSIZE)
    axC.set_ylabel("best-direction AUC", fontsize=LABEL_FONTSIZE)
    axC.tick_params(axis="y", labelsize=TICK_FONTSIZE)
    axC.grid(alpha=0.2)

    # Panel C-2
    axC2 = fig.add_subplot(gs[1, 2])
    N_QUANTILES = int(globals().get("N_QUANTILES", 4))
    MIN_TOTAL_N = int(globals().get("MIN_TOTAL_N", 80))
    rows_q = []
    for gkey, gdf in df.groupby(group_cols):
        ch = gkey[4]
        if ch not in CHANNELS:
            continue
        x = gdf["log2_value"].to_numpy(dtype=float)
        y = (gdf["state"].to_numpy() == STATE_ACTIVE).astype(int)
        ok = np.isfinite(x)
        x = x[ok]
        y = y[ok]
        if x.size < MIN_TOTAL_N:
            continue
        if y.min() == y.max():
            continue
        try:
            bins = pd.qcut(x, q=N_QUANTILES, labels=False, duplicates="drop")
        except Exception:
            continue
        if bins is None:
            continue
        bins = np.asarray(bins)
        if np.unique(bins).size < N_QUANTILES:
            continue
        p0 = y.mean()
        for q in range(N_QUANTILES):
            m = bins == q
            if m.sum() == 0:
                continue
            pq = y[m].mean()
            rows_q.append({"channel": ch, "quantile": q + 1, "delta_from_baseline": float(pq - p0)})
    qdf = pd.DataFrame(rows_q)
    if not qdf.empty:
        summ = (
            qdf.groupby(["channel", "quantile"])
            .agg(
                mean_delta=("delta_from_baseline", "mean"),
                sem_delta=("delta_from_baseline", lambda v: np.std(v, ddof=1) / np.sqrt(len(v)) if len(v) > 1 else np.nan),
                n_groups=("delta_from_baseline", "size"),
            )
            .reset_index()
        )
        for ch in CHANNELS:
            sub = summ[summ["channel"] == ch].sort_values("quantile")
            axC2.errorbar(
                sub["quantile"].to_numpy(),
                sub["mean_delta"].to_numpy(),
                yerr=sub["sem_delta"].to_numpy(),
                marker="o",
                linewidth=1.5,
                label=ch,
            )
    axC2.axhline(0.0, linestyle="--", linewidth=1)
    axC2.set_xticks(np.arange(1, int(globals().get("N_QUANTILES", 4)) + 1))
    axC2.set_xlabel("Within-group intensity quantile", fontsize=LABEL_FONTSIZE)
    axC2.set_ylabel("Δ Active fraction (vs group baseline)", fontsize=LABEL_FONTSIZE)
    axC2.set_title("Quantile-based Active enrichment (equal-weight across groups)", fontsize=LABEL_FONTSIZE)
    axC2.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    axC2.grid(alpha=0.2)
    axC2.legend(frameon=False, fontsize=TICK_FONTSIZE)

    fig.tight_layout()
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_pdf, bbox_inches="tight")
    print("Saved split figure:", out_pdf)
    plt.show()


design = _load_design_table()
cy_factors, noncy_factors = _split_factors_by_cy(design)
print(f"Design factors: cy={len(cy_factors)}, noncy={len(noncy_factors)}")

# Use already-prepared df from Cell 56 if available; else load from LONG_CSV
if "df" in globals() and isinstance(globals()["df"], pd.DataFrame):
    df_full = globals()["df"].copy()
    # Ensure derived columns exist
    if "log2_value" not in df_full.columns or "log2_centered" not in df_full.columns:
        df_full = _prepare_long_df_from_csv(Path(globals().get("LONG_CSV")))
else:
    df_full = _prepare_long_df_from_csv(Path(globals().get("LONG_CSV")))

df_cy = df_full[df_full["factor"].isin(cy_factors)].copy()
df_noncy = df_full[df_full["factor"].isin(noncy_factors)].copy()
print("Rows in split long table:", {"cy": len(df_cy), "noncy": len(df_noncy)})

OUT_DIR = Path(globals().get("OUT_DIR", "Figs_thr1.30_1.15/mean_intensity_state_dependence_summary"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_pdf_cy = OUT_DIR / "reporter_expression_state_dependence_ECDF_effect_AUC__cy.pdf"
out_pdf_noncy = OUT_DIR / "reporter_expression_state_dependence_ECDF_effect_AUC__noncy.pdf"
out_csv_cy = OUT_DIR / "group_level_effect_and_auc_summary__cy.csv"
out_csv_noncy = OUT_DIR / "group_level_effect_and_auc_summary__noncy.csv"

if len(df_cy) > 0:
    _run_state_dependence(df_cy, out_pdf_cy, out_csv_cy)
else:
    print("[WARN] No rows for (cy) subset; skipping.")

if len(df_noncy) > 0:
    _run_state_dependence(df_noncy, out_pdf_noncy, out_csv_noncy)
else:
    print("[WARN] No rows for non-(cy) subset; skipping.")